## V3.8 (October 2026 NOWCAST: observed inputs, SEAS5 rain only)

- Vegetation indices for 2026-10 are the **observed** monthly composites (`cfg.vi_fc_months = []`), not Notebook C.
- Sentinel-2 (including coastal B1 and water vapour B9 from the C_WV stacks) and Landsat are read for **2026_10 itself** (no October-2025 stand-in).
- Temperature, RH and wind are the **observed** daily files. Only rain comes from the SEAS5 files (`cfg.fc_only_vars = ["precip"]`), because CHIRPS does not cover October.
- **Band labels:** in the Sentinel_AllBands files B03 = red, B06 = red edge 3, B07 = NIR (labels are band positions); in the C_WV files B1 = coastal and B2 = water vapour. `sentinel_band_names` reads them this way (Section 4).
- Also in this version: 5 risk classes (Very low, Low, Medium, High, Very high), hybrid = 80 % statistical + 20 % mechanistic, no RH donor fill for Nyeri.
- **Retrain first** (Section 18, step 1): weights and class breaks are saved inside the models.

## Changes in V3 (monthly Landsat, Sentinel-2 bands, run-order fix)

- **Run-order fix:** the coastal + water vapour cell no longer prints `PRED_SETS` (it does not exist yet at that point). That `NameError` stopped Run All / Run All Above before the weather cell, which caused `name 'load_weather' is not defined`.
- **Landsat:** now the monthly stacks in `F:/VI_prediction/LANDSAT_STACK_MONTHLY/tiff/<County>/` (2020-2024; names end in `_01`, keep them). A record or map date takes the stack of its calendar month; months without a stack (2025 on, October 2026) take the same calendar month of the nearest year that has one. Own cache prefix `landsatm_`. Nyeri now has Landsat.
- **Sentinel-2:** red (`s.red`), NIR (`s.nir`), red-edge 3 (`s.red.edge` now means B7), coastal (`s.coastal`) and water vapour (`s.vapour`). Coastal and water vapour are read from the C_WV stacks first. MLND no longer uses `s.blue`.
- **Retrain before predicting:** the predictor lists changed, so the models saved by the earlier run no longer match. Back up `<County>/models` first.
- Do not run the rename script from the Landsat notebook; the `_01` day part is how the dates are read.


# LocateIT | Pest and Disease Modelling | Unified Pest Risk Pipeline (Python)
**Fall armyworm (FAW), Aphids, Thrips, Stem borer, African armyworm (AAW) and Maize Lethal Necrosis Disease (MLND)** in one notebook.

Every pest gets three components, saved as separate GeoTIFFs:
1. **Statistical**: MaxEnt, GAM, Random Forest, GBM, XGBoost, SVM, combined into an ensemble weighted by spatial cross-validation.
2. **Mechanistic**: temperature and degree-day rules with sourced thresholds (see `THR`, every value has a status and a source).
3. **Hybrid**: fixed weighted blend of 1 and 2. The statistical part always has the higher weight (default 80 % statistical, 20 % mechanistic,
   `cfg.hybrid_w_mech_fixed`) because it is built from real ground presence records.

**Presence only.** Infestation level or severity columns are never used. Every cleaned record is a presence. Background points are NOT absences: they describe the available environment, and the models measure how the conditions at the records differ from it.

Same logic as the R pipeline (`PestRisk_Unified_Pipeline.R`). Works at any grid size. Set `cfg.res_m = 30` for the project.
All raster work runs window by window from files on disk, so 16 GB of RAM is enough.

**How to run**
1. Edit the CONFIG cell (folders, counties, resolution).
2. Run all cells down to the last one.
3. Run cells: Kakamega is active. The Nyeri and Kilifi cells are commented out, remove the `#` when their datasets are complete.
   `predict_dates_only("Kakamega", ["2024-08-01"])` reuses saved models for any date, and `predict_points(...)` scores a table of points.
4. Every run also writes: a 70/30 hold-out accuracy workbook, all rasters as EPSG:4326 GeoTIFFs, and one PNG map per pest.

**Forecast version (October 2026).** TCARI, ARI, CRI and MSAVI are dropped everywhere (registry and predictor lists) because Notebook C does not forecast them.
For a month listed in `cfg.vi_fc_months` the index layers are read from Notebook C's GeoTIFFs. Section 18 at the end retrains with the reduced predictor list and maps October 2026.

**Hardware notes (Ryzen 5 PRO 4650G, 16 GB, RTX 3050)**
- XGBoost can use the RTX 3050 (`device="cuda"`). It is tested once, and the notebook falls back to CPU if CUDA is not available.
- Everything else (scikit-learn, pyGAM, elapid, rasterio) runs on the CPU, using `cfg.cores` threads.
- SVM is dropped automatically on grids finer than 100 m because predicting that many cells would take hours.

Plain ASCII only, so it runs the same on Windows and Linux.

In [ ]:
# Install anything missing (run once). On Windows use the same command in a Jupyter cell.
import importlib, importlib.util, subprocess, sys
_need = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "rasterio": "rasterio", "geopandas": "geopandas",
         "shapely": "shapely", "pyproj": "pyproj", "sklearn": "scikit-learn", "xgboost": "xgboost",
         "pygam": "pygam", "elapid": "elapid", "joblib": "joblib"}
_missing = [pip for mod, pip in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing:", _missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])

**Next code cell.** Run it after the cell above.

In [ ]:
import os, re, json, math, time, glob, logging, warnings, functools, operator
from types import SimpleNamespace
from dataclasses import dataclass
from datetime import date as _date, datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.transform import Affine, from_origin
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
from pyproj import Transformer, CRS
from scipy import ndimage
from joblib import Parallel, delayed, dump, load
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

## 0. CONFIG (edit this cell only)

**What this cell does:** holds every setting in one place (folders, counties, resolution, split, weights, outputs). **Run it first.** **You edit:** only this cell. Key switches: `hybrid_w_mech_fixed` (mechanistic share of the hybrid, statistical gets the rest), `model_prior` and `overfit_scale` (how much each ML model counts), `risk_scaling` and `class_presence_q` (how scores become classes), `min_month_share` and `uncovered_mode` (what to map in months the records do not cover), `test_frac` (0.30 hold-out), `export_products` and `plot_products` (which rasters and maps to write).

**Folders:** `v_root` (`F:/VI_prediction`) holds the weather (`HISTORICAL WEATHER/ERA5_LAND_TEMP_fixed`, `DAILY_RH/RH_<COUNTY>/daily_tifs`, CHIRPS, wind), the vegetation indices, the DEM (`SRTM`) and Landsat: the same files the LSTM notebooks use. `h_root` is now used only for soil, crop type/mask and Sentinel-2. After changing folders, use a fresh `out_root` or delete the `aligned` and `wx` caches so old layers are not reused.

In [ ]:
cfg = SimpleNamespace(
    # ---- folders (copied from the project folder list) ---------------------------------------
    root="D:/Projects/Pest & Disease Modelling",                     # project folder (no longer where outputs go, see out_root)
    h_root="F:/VI_prediction",                                       # soil (SoilGrids), crop type/mask (JRC_CropType_CropMask): moved from H: into the same folder as v_root
    v_root="F:/VI_prediction",                                       # weather (HISTORICAL WEATHER), vegetation indices, DEM (SRTM) and Landsat: same folders as the LSTM notebooks
    inc_root="F:/VI_prediction/Incidence_data",                      # one folder per pest, see INC_SPEC (moved from H:)
    boundary_dir="D:/Projects/Pest & Disease Modelling/APHIDS/Final_Pipeline/County_boundaries",
    county_shp=r"C:\Users\locateIT1\Downloads\ken_admin1.shp",   # national admin-1 shapefile; each county is picked from it by name. None = use boundary_dir/<County>.shp
    county_name_col="adm1_name",             # column of county_shp that holds the county name (Kakamega, Nyeri, Kilifi)
    drop_predictors=[],   # Sentinel-2 B1 and B9 are not available for all months yet (the 2025+ files have B2 to B12 only). Dropped from every predictor list so training and the 2026-10 map use the same inputs. Set [] once B1 and B9 are downloaded for all months
    out_root="F:/pest_unified_model",            # all outputs, caches and models are written here (the susceptibility run goes to F:/pest_unified_model_susceptibility)
    # ---- study design ----------------------------------------------------------------------
    counties=["Kakamega", "Nyeri", "Kilifi"],   # all three. The run cells at the end of the notebook run one county at a time (Nyeri and Kilifi are commented out until their data is complete)
    pests=["FAW", "APHIDS", "THRIPS", "STEMBORER", "MLND", "AAW"],   # products: FAW, APHIDS, MLND. THRIPS and STEMBORER are trained and mapped only because MLND uses them as vectors (mlnd_vectors). MLND must stay last
                                             # AAW = African armyworm
    res_m=30,                                # model grid in metres (use 1000 only for quick tests)
    crs="EPSG:6933",                         # equal-area, as in the original script. EPSG:32636 (Kakamega) or
                                             # EPSG:32737 (Nyeri, Kilifi) also work at 30 m.
    # ---- time handling ---------------------------------------------------------------------
    windows=(7, 14, 30),                     # rolling windows in days, ending on the observation date
    main_window=7,                           # window used for the un-suffixed weather names
    min_window_coverage=0.7,                 # a window needs this share of days, else NaN
    crop_season="Long_Rain2023",             # JRC crop mask / crop type season tag in the file names
    monthly_years=(2020, 2026),              # years searched for monthly index composites
    month_fallback="same_month",             # no composite for the record month: use the SAME calendar month of the
                                             # nearest year with data (seasonal match). "nearest" = closest month in time.
    ym_fallback="2024_07",                   # used only if no composite exists at all
    # ---- forecast vegetation indices from Notebook C (LSTM forecast, GeoTIFFs) ---------------------
    vi_fc_dir="F:/CORRECTION/geotiff",   # <scenario>/<County>/<County>_<INDEX>_<YYYY-MM>.tif, written by Notebook C
    vi_fc_scenario="mean",                   # Notebook C scenario folder: "mean" (the only one it wrote by default), or "p10", "p90", "normal" if you ran them
    vi_fc_months=["2026-10", "2026-11", "2026-12", "2027-01", "2027-02", "2027-03", "2027-04"], # NOWCAST: empty = vegetation indices are read from the OBSERVED monthly composites (F:/VI_prediction/<INDEX>/...), not from Notebook C
    fc_s2_same_month=True,                   # Sentinel-2 band layers do not exist for a forecast month: use the same calendar month of the nearest earlier year (seasonal stand-in)
    fc_prior_year=2025,                      # year whose Sentinel-2 bands and Landsat scene stand in for the forecast month (2025 = October 2025 for 2026-10). No search of other years
    fc_landsat_pick="first",                 # forecast month: "first" = the earliest Landsat scene of that month in fc_prior_year, "nearest" = nearest to the same day (within landsat_max_days)
    fc_landsat_prev_year=True,               # NOT USED with the monthly Landsat stacks: a month without a stack takes the same calendar month of the nearest year that has one
    predict_dates=None,                      # None = last date in the weather record
    # ---- SEAS5 forecast weather (coarse files made by the forecast notebooks; the lapse-rate correction to 30 m is done here, so do NOT point this at the 30 m files) ----
    use_forecast=False,                      # True = daily forecast files are appended after the last historical day (use it through predict_dates_only(..., forecast=True))
    fc_only_vars=["precip"],                 # which variables the SEAS5 files may fill when use_forecast=True. ["precip"] = rain only (CHIRPS has no October), temperature, RH and wind stay OBSERVED. None = all variables
    fc_root="F:/VI_prediction/FORECASTED_WEATHER_BC_V2",         # BIAS-CORRECTED coarse SEAS5 (output of the bias-correction notebook; rain and wind_dir copied unchanged). <group>/<var>/<scenario>/<County>_<var>_<scenario>_<YYYY-MM-DD>.tif : temperature/tmax, temperature/tmin, rainfall/rain, humidity/rh, wind/wind_speed, wind/wind_dir
    fc_scenario="mean",                      # "mean", "p10" or "p90"
    fc_bias_csv=None,                        # no bias table: the SEAS5 bias-correction notebook writes the corrected files straight to fc_root (no csv). Only used if fc_bias_vars is not empty, and then needs a table with columns county, var, start_month, lead, kind, value
    fc_bias_vars=[],                         # [] = no bias correction here, because fc_root is already bias-corrected (do not add one, it would be applied twice). Any of "temp" (applied to tmax AND tmin, so the daily range is kept), "rh", "rain", "wind"
    weather_start=None, weather_end=None,    # optional, e.g. "2022-01-01": load only this part of the daily weather
    # ---- weather handling ------------------------------------------------------------------
    lapse_correct=True,                      # correct coarse temperature to 30 m terrain
    lapse_rate=-0.0065,                      # deg C per metre
    wet_day_mm=1.0,
    wind_bands=dict(speed=1, u=2, v=3, direction=4),   # CONFIRMED by the inspection report: band 4 is 0-360 degrees. Used only if bands have no names
    # ---- incidence data --------------------------------------------------------------------
    incidence_cols=dict(lon="longitude", lat="latitude", date=None),   # date=None: use incidence_date_cols below
    incidence_date_cols=["initial_detection_date", "detection_date", "survey_date", "obs_date", "date", "created_at"],
                                             # per record, the first column with a usable date is the occurrence date.
                                             # planting_date is never used as the occurrence date.
    incidence_crop_regex={"FAW": "maize|corn", "THRIPS": "maize|corn", "STEMBORER": "maize|corn", "AAW": "maize|corn",
                          "MLND": "maize|corn", "APHIDS": None},   # APHIDS files hold vegetables, beans and citrus, no maize (user decision: all crops)
    incidence_exclude_report="weed",         # drop rows whose report_type matches this (WEED rows are not pests)
    incidence_pest_regex={"FAW": "fall.?armyworm|faw", "APHIDS": "aphid", "MLND": "necrosis|mlnd|mln", "AAW": "african", "STEMBORER": "stem.?bor", "THRIPS": "thrip"},   # on pest_name (None = off)
    kenya_box=(33.0, 42.5, -5.5, 5.5),       # lon min, lon max, lat min, lat max. Points outside are dropped
    min_presence=15,
    n_background=10000,
    thin_per_cell_month=True,
    # ---- host / crop mask ------------------------------------------------------------------
    host_mode="all_land",                    # "all_land": every cell inside the county is host, and the crop map enters only as the predictor crop.type (the model then learns
                                             #   from the records how much more often they sit on mapped maize). DEFAULT, because on the Kakamega crop map 84 % of FAW records,
                                             #   88 % of MLND records and 95 % of Aphids records fall on cells mapped as "other landcover" (code 10): the map misses most smallholder fields.
                                             # "crop_map": host = mapped cropland only (crop.mask == crop_mask_crop_value, or crop.type in maize_codes). Would drop most records here.
    maize_codes=None,                        # only for host_mode = "crop_map". crop.type codes that mean the host crop, e.g. [1] = maize. None = use the crop mask
    crop_mask_crop_value=1,                  # only for host_mode = "crop_map". In the JRC crop mask, 1 = cropland and 2 = other landcover (2 matches crop type 10 cell for cell)
    crop_mask_min=None,                      # no longer used (it treated value 2 as cropland, which was backwards)
    host_window=3,                           # host = share of cropland in this many cells across
    # ---- statistical models ----------------------------------------------------------------
    models=["maxent", "gam", "rf", "gbm", "xgboost", "svm"],
    cv_folds=5,
    cv_block_m=10000,                        # spatial block size for cross-validation
    min_auc=0.60,                            # models below this CV AUC get no ensemble weight
    min_auc_by_model=dict(gam=0.65),         # stricter floor for a model type (the GAM was at chance on the Kakamega hold-out even when its CV AUC was 0.61 to 0.69)
    # ---- hold-out test (70 / 30) and model weighting ---------------------------------------
    test_frac=0.30,                          # 30 % of the points are kept aside and never used for fitting or weighting
    split_method="block",                    # "block": whole spatial blocks go to the test set (honest, no neighbour leakage); "random"
    refit_on_all=True,                       # after testing, refit every model on 100 % of the points for the saved/deployed model
    # Prior weight of each model type. Basis: benchmark and ensemble studies of presence-background models (Valavi et al. 2022,
    # Ecological Monographs; Marmion et al. 2009, Diversity and Distributions; Araujo and New 2007, TREE) rank tuned MaxEnt, boosted
    # trees (GBM, XGBoost) and Random Forest at the top and GAM and SVM lower. The NUMBERS below are my own editable choice, not values
    # published in those papers. Final weight = prior x (CV AUC - 0.5), so a model must also perform well on THIS data.
    model_prior=dict(maxent=1.5, gbm=0.7, rf=0.7, xgboost=0.9, gam=0.6, svm=0.7),
                                             # MaxEnt is raised and RF / GBM / GAM are lowered because on the Kakamega hold-out MaxEnt and XGBoost kept their AUC
                                             # while RF, GBM and GAM lost 0.05 to 0.16. These numbers are MY editable choice, not published values.
    overfit_scale=0.50,                      # a model's weight is multiplied by exp(-gap / overfit_scale), gap = AUC on its own training rows minus its spatial CV AUC.
                                             # gap 0.1 = x0.82, gap 0.3 = x0.55. Mild on purpose: all tree models over-fit their own rows, so this only separates the extreme cases. Smaller number = harsher. None = off
    mech_min_auc=0.55,                       # None = always use hybrid_w_mech_fixed, even if the mechanistic part is poor. Otherwise: the mechanistic part gets weight ONLY if its AUC on the training part is above this
    risk_scaling="percentile",               # "percentile" (default): the statistical and the mechanistic score are each turned into a percentile of the available-
                                             # environment (background) scores, blended, and NOT clamped (no ties). The statistical reference is the HELD-OUT (30 %) part,
                                             # because a model fitted on all data scores new places like the hold-out does, not like cross-validation.
                                             # "stretch": the old 2nd to 98th percentile stretch with a clamp. "none": raw blended score.
    stretch_q=(0.02, 0.98),                  # only for risk_scaling = "stretch"
    class_mode="presence_quantiles",         # "presence_quantiles": class breaks are quantiles of the held-out PRESENCE scores (class_presence_q).
                                             # "fixed": use risk_breaks below.
    class_presence_q=(0.10, 0.30, 0.60, 0.85),   # FIVE classes. Breaks are quantiles of the held-out PRESENCE scores: Very low = below the score that 90 % of records reach;
                                             # Low up to the 30th percentile of the records; Medium up to the 60th; High up to the 85th; Very high above. MY choice, edit freely.
                                             # More Very high area = lower the last number. Four numbers = five classes.
    min_month_share=0.05,                    # a calendar month counts as COVERED by the training records if it holds at least this share of the pest's records
    uncovered_mode="mechanistic",            # map date in a month the records do not cover. EVERY pest still gets its hybrid and class maps.
                                             # "mechanistic": pests with a valid mechanistic part use it alone for that month; the others keep the statistical map (an extrapolation).
                                             # "model": always the statistical + mechanistic map. Such maps are marked in the map title and in the area-summary csv (season_covered = False).
    mech_empirical=["AAW", "STEMBORER"],     # pests whose mechanistic temperature response is taken from their OWN occurrence records (5th, 25th, 75th, 95th percentile of
                                             # the temperature at the records of the 70 % training part). NOT literature thresholds. Replace with sourced values when you have them.
    mech_empirical_q=(0.05, 0.25, 0.75, 0.95),
    collinear_cut=0.80,
    svm_at_fine_res=False,                   # SVM dropped on grids finer than 100 m
    svm_max_n=4000,
    gam_max_terms=10,
    seed=123,
    # ---- hybrid ----------------------------------------------------------------------------
    # The statistical output is built from real ground presence records, so it ALWAYS gets the larger weight.
    # Hybrid = (1 - w_mech) x statistical + w_mech x mechanistic, with w_mech fixed here (0.30 -> 70 % statistical, 30 % mechanistic).
    # w_mech must stay below 0.5 (the code stops otherwise). It is set to 0 for a pest only when the mechanistic part has no sourced
    # thresholds, or when its AUC on the training records is below mech_min_auc (it then does not match the observed presences).
    hybrid_w_mech_fixed=0.20,                # 80 % statistical, 20 % mechanistic
    hybrid_w_mech_max=0.45,                  # hard cap, keeps statistical above mechanistic
    risk_breaks=(0.20, 0.40, 0.60, 0.80),    # Very low / Low / Medium / High / Very high (used only if class_mode = "fixed")
    mech_use_placeholders=False,             # True adds the NOT-yet-verified RH and rainfall terms
    # ---- coupling between pests -----------------------------------------------------------
    aphids_use_faw_risk=True,
    mlnd_vectors=["APHIDS", "FAW", "THRIPS", "STEMBORER"],   # pests whose risk maps feed MLND. [] = MLND from its own records only
    mlnd_vector_weights=None,                # None = equal weights. PLACEHOLDER until transmission weights from literature are approved
                                             # e.g. {"APHIDS": 0.4, "THRIPS": 0.3, "FAW": 0.1, "STEMBORER": 0.2}
    # ---- data gaps handled in code ---------------------------------------------------------
    rh_fill_from={},                         # Nyeri RH is now complete, so no donor county is used. {"Nyeri": "Kakamega"} would fill missing days again
    landsat_max_days=16,                     # NOT USED with the monthly Landsat stacks (matched by calendar month, see nearest_scene)
    clip={"cvi": (0.0, 50.0)},               # ASSUMPTION: CVI max was 1945 (outliers). Review the cap.
    # ---- computing -------------------------------------------------------------------------
    cores=None,                              # None = logical cores minus 2
    # ---- exports and maps ---------------------------------------------------------------------
    export_products=["stat_ensemble", "mechanistic", "hybrid", "risk_class", "hybrid_raw", "stat_sd"],   # saved also as EPSG:4326 GeoTIFFs
    plot_products=["stat_ensemble", "mechanistic", "hybrid", "risk_class"],  # one figure per pest, county, date and product
    cmap="Spectral_r", plot_dpi=300,
    block_cells=1_000_000,                   # cells per processing window (lower if RAM is tight)
    use_gpu=True,                            # try XGBoost on CUDA, fall back to CPU
    reuse_cache=True,
    # ---- static susceptibility mode (run_static) -----------------------------------------------
    static_out_root=None,                    # None = <out_root>_susceptibility (a separate folder, nothing in out_root is touched)
    static_ref_dates=None,                   # None = the 15th of each month of the last 12 months of the weather record
    static_mode=False,                       # set by run_static, do not edit
    report_random_split=False,               # also score a random 70/30 split (comparison only, optimistic). run_static sets it
    cache_root=None,                         # aligned-layer and weather caches are read from here (run_static points it at out_root)
)

# Optional: override from an environment variable without editing the cell (used for testing)
if os.environ.get("PEST_CFG_OVERRIDE"):
    for _k, _v in json.loads(os.environ["PEST_CFG_OVERRIDE"]).items():
        setattr(cfg, _k, tuple(_v) if isinstance(_v, list) and _k in ("windows", "risk_breaks", "monthly_years", "kenya_box", "mlnd_vectors", "stretch_q", "class_presence_q", "mech_empirical", "mech_empirical_q") else _v)

if cfg.out_root is None:    cfg.out_root = os.path.join(cfg.root, "PestRisk_outputs_py")
if cfg.cores is None:       cfg.cores = max(1, (os.cpu_count() or 2) - 2)
os.makedirs(cfg.out_root, exist_ok=True)
rng = np.random.default_rng(cfg.seed)

## 0.1 Helpers

**What this cell does:** small logging, warning and file-name helpers used by every later cell. Nothing to edit. Output: messages printed with a time stamp.

In [ ]:
LOGGER = logging.getLogger("pestrisk")
LOGGER.setLevel(logging.INFO)
if not LOGGER.handlers:
    _h = logging.StreamHandler(sys.stdout)
    _h.setFormatter(logging.Formatter("[%(asctime)s] %(message)s", "%H:%M:%S"))
    LOGGER.addHandler(_h)

def set_logfile(path):
    for h in list(LOGGER.handlers):
        if isinstance(h, logging.FileHandler):
            LOGGER.removeHandler(h)
    fh = logging.FileHandler(path)
    fh.setFormatter(logging.Formatter("[%(asctime)s] %(message)s", "%H:%M:%S"))
    LOGGER.addHandler(fh)

def log(*a):  LOGGER.info("".join(str(x) for x in a))
def warn(*a): LOGGER.info("WARNING: " + "".join(str(x) for x in a))

def out_dir(county, *sub):
    root = cfg.cache_root if (cfg.cache_root and sub and sub[0] in ("aligned", "wx")) else cfg.out_root
    d = os.path.join(root, county, *sub)
    os.makedirs(d, exist_ok=True)
    return d

def fill_tokens(p, county, ym=None):
    p = (p.replace("{County}", county).replace("{COUNTY}", county.upper()).replace("{county}", county.lower())
          .replace("{cropseason}", cfg.crop_season).replace("{H}", cfg.h_root.rstrip("/\\")).replace("{V}", cfg.v_root.rstrip("/\\"))
          .replace("{INC}", cfg.inc_root.rstrip("/\\")).replace("{BOUND}", cfg.boundary_dir.rstrip("/\\")))
    if ym:
        yy, mm = ym[:4], ym[5:7]
        import calendar
        p = p.replace("{yyyy}", yy).replace("{mm}", mm).replace("{Month}", calendar.month_name[int(mm)])
    return p.replace("\\", "/")

def _ci_resolve(path):
    """exact path, else each folder/file name matched ignoring upper/lower case (needed on Linux, harmless on Windows)"""
    path = path.replace("\\", "/")
    if os.path.exists(path):
        return path
    parts = path.split("/")
    if re.match(r"^[A-Za-z]:$", parts[0]):
        cur, rest = parts[0] + "/", parts[1:]
    elif path.startswith("/"):
        cur, rest = "/", parts[1:]
    else:
        cur, rest = "", parts
    for part in rest:
        if part == "":
            continue
        try:
            names = os.listdir(cur if cur else ".")
        except OSError:
            return None
        hit = [n for n in names if n.lower() == part.lower()]
        if not hit:
            return None
        cur = os.path.join(cur, hit[0]) if cur else hit[0]
    return cur

def find_file(path, ym=None):
    """exact path, else case-insensitive match, else (monthly layers) any .tif in that folder carrying the same
    yyyy_mm, e.g. 'Kakamega_MTVI2_2024_10 - Copy.tif'. Returns None if nothing is found."""
    r = _ci_resolve(path)
    if r:
        return r
    if ym:
        d = _ci_resolve(os.path.dirname(path))
        if d and os.path.isdir(d):
            yy, mm = ym[:4], ym[5:7]
            pat = re.compile(r"(?<!\d)%s[_-]%s(?!\d)" % (yy, mm))
            base = os.path.basename(path).lower()
            stem = re.sub(r"\d{4}[_-]\d{2}.*$", "", base)               # name part before the date, e.g. 'ndvi_kakamega_'
            hits = sorted([f for f in os.listdir(d) if f.lower().endswith(".tif") and pat.search(f)], key=lambda f: (not f.lower().startswith(stem), len(f)))
            if hits:
                return os.path.join(d, hits[0])
    return None

def find_dir(path):
    r = _ci_resolve(path.rstrip("/"))
    return r if (r and os.path.isdir(r)) else None

def parse_dates(x):
    x = pd.Series(x).astype(str).str.slice(0, 10)
    best, bestn = None, -1
    for f in ["%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%Y/%m/%d", "%d-%m-%Y", "%d.%m.%Y", "%Y_%m_%d"]:
        d = pd.to_datetime(x, format=f, errors="coerce")
        n = int(d.notna().sum())
        if n > bestn:
            best, bestn = d, n
    return best

def dates_from_files(files):
    """date from names such as Kakamega_tmax_2024_07_15.tif, Kilifi_RH_2024-07-15.tif or Kakamega_tmax_20240715.tif"""
    out = []
    for f in files:
        b_ = os.path.basename(f)
        m = re.search(r"(?<!\d)((?:19|20)\d{2})[_-](\d{2})[_-](\d{2})(?!\d)", b_) or re.search(r"(?<!\d)((?:19|20)\d{2})(\d{2})(\d{2})(?!\d)", b_)   # 2024_07_15, 2024-07-15 or 20240715
        try:
            out.append(pd.Timestamp("%s-%s-%s" % m.groups()) if m else pd.NaT)
        except ValueError:
            out.append(pd.NaT)
    return pd.DatetimeIndex(out)

def ym_of(d):
    return pd.Timestamp(d).strftime("%Y_%m")

def clamp01(x):
    return np.clip(x, 0.0, 1.0)

def src_res_m(src):
    r = abs(src.res[0])
    return r * 111320.0 if src.crs and src.crs.is_geographic else r

## 1. Data registry
One row per layer. Tokens: `{County} {COUNTY} {county} {yyyy} {mm} {Month} {year} {cropseason}`.
`kind`: static, monthly, monthly_mb (multi-band), cat (categorical static). Missing files are reported and skipped,
so the notebook can run while downloads finish.

**What this cell does:** lists every input layer and where it sits on the H drive, then checks which files are found. **Output:** a table of found and missing layers. **You edit:** only if a folder or file name differs from the registry.

In [ ]:
def _build_registry():
    """One row per layer. `always` layers are loaded for every run; the others only if a predictor list uses them."""
    r = []
    def add(name, rel, kind, method="bilinear", bands=None, always=False):
        r.append(dict(name=name, rel=rel, kind=kind, method=method, bands=bands, always=always))
    # monthly vegetation indices on drive H. The same folder pattern is used for every index, and the exceptions are
    # handled by find_file (folder case, the file named County_INDEX_yyyy_mm, ' - Copy' suffixes).
    # Folder layout (same as the LSTM notebooks): F:/VI_prediction/<INDEX>/<County>/<INDEX>_<County>_<yyyy>_<mm>.tif ; SAVI and MTVI2 have an extra "tiff" folder.
    # TCARI, ARI, CRI and MSAVI are dropped: Notebook C does not forecast them (it forecasts the 10 indices below, SAVI and MTVI2 are added on the next two lines)
    for ix in ["CVI", "MCARI", "NDMI", "NDRE", "NDVI", "PRI", "PSRI", "RDVI"]:
        add(ix.lower(), "{V}/%s/{County}/%s_{County}_{yyyy}_{mm}.tif" % (ix, ix), "monthly")
    add("savi", "{V}/SAVI/tiff/{County}/SAVI_{County}_{yyyy}_{mm}.tif", "monthly")
    add("mtvi2", "{V}/MTVI2/tiff/{County}/MTVI2_{County}_{yyyy}_{mm}.tif", "monthly")   # same pattern as the other indices. "- Copy" files are ignored when this exists
    # soils, terrain (static)
    add("clay", "{H}/SoilGrids/{County}/clay_0-30cm_{County}.tif", "static")
    add("soil.ph", "{H}/SoilGrids/{County}/phh2o_0-30cm_{County}.tif", "static")
    add("sand", "{H}/SoilGrids/{County}/sand_0-30cm_{County}.tif", "static")
    add("soc", "{H}/SoilGrids/{County}/soc_0-30cm_{County}.tif", "static")
    add("dem", "{V}/SRTM/{County}/{county}_dem_clipped.tif", "static", always=True)
    # crop products (categorical: never interpolate)
    add("crop.mask", "{H}/JRC_CropType_CropMask/{County}/CropMask/{cropseason}_CropMask_{County}.tif", "cat", "near", always=True)
    add("crop.type", "{H}/JRC_CropType_CropMask/{County}/CropType/{cropseason}_CropType_{County}.tif", "cat", "near", always=True)
    # Sentinel-2 and Landsat composites: NO PATH WAS SUPPLIED YET. Edit these two patterns when the data is downloaded.
    # Sentinel-2: F:/VI_prediction/Sentinel_AllBands_<folder year>/<County>/S2_<County>_<yyyy>_<mm>.tif (the folder year is not always the file year, e.g. S2_Kakamega_2025_10.tif sits in Sentinel_AllBands_2024).
    # find_sentinel() looks in every Sentinel_AllBands_* folder, then in the old H-drive layout given here.
    add("sentinel", "{H}/Sentinel_AllBands_{yyyy}/{County}/sentinel2_bands/{County}_{yyyy}_{mm}_bands.tif", "monthly_mb",
        bands=["s.coastal", "s.blue", "s.green", "s.red", "s.red.edge1", "s.red.edge2", "s.red.edge3",
               "s.nir", "s.red.edge4", "s.vapour", "s.swir1", "s.swir2"])
    return pd.DataFrame(r)

REG = _build_registry()

# original predictor names that map onto the registry
ALIASES = {"s.red.edge": "s.red.edge3",   # red edge 3 (Sentinel-2 B7), as chosen for this model
           "s.water.vapour": "s.vapour"}

# Daily weather folders (GeoTIFFs; the date is read from the file name, underscores or hyphens)
WX_SPEC = {
    "tmax":   "{V}/HISTORICAL WEATHER/ERA5_LAND_TEMP_fixed/{County}/tmax/",
    "tmin":   "{V}/HISTORICAL WEATHER/ERA5_LAND_TEMP_fixed/{County}/tmin/",
    "precip": "{V}/HISTORICAL WEATHER/CHIRPS_DATA_PY_daily/filled_daily/{County}/",
    "rh":     "{V}/HISTORICAL WEATHER/DAILY_RH/RH_{COUNTY}/daily_tifs/",
    "wind":   "{V}/HISTORICAL WEATHER/WIND_DATA_PY_daily/county_daily_direct/{County}/",   # 4 bands, see cfg.wind_bands
}
INC_SPEC = {
    "FAW":       "{INC}/Fall_Armyworms/{county}_fall_armyworm_incidence.csv",
    "APHIDS":    "{INC}/Aphids/{county}_aphids_incidence.csv",
    "MLND":      "{INC}/MLND/{COUNTY}.csv",
    "AAW":       "{INC}/African_armyworm/{County}_africanarmyworm.csv",
    "STEMBORER": "{INC}/Stemborer/{County}_Stemborer.csv",
    "THRIPS":    "{INC}/Thrips/{County}_thrips.csv",
}
# Landsat MONTHLY stacks 2020-2024 (monthly means; 3 bands: COASTAL, BLUE, LST_K), Kakamega, Kilifi and Nyeri. NIR/SWIR are NOT in these files.
# File names are LANDSAT_STACK_<County>_<YYYY>_<MM>_01.tif: the day part is always 01 and must stay (the date reader needs YYYY_MM_DD).
# LANDSAT_STACK (the complete one, up to Sep 2026). The first folder that exists is used; the log line "Landsat <County>: n stacks ..." shows which.
LANDSAT_DIRS = ["{V}/LANDSAT_STACK/{County}/", "{V}/LANDSAT_STACK/tiff/{County}/"]   # first = the layout Notebook A reads (VI_ROOT/LANDSAT_STACK/<County>/LANDSAT_STACK_<County>_<yyyy>_<mm>.tif)
LANDSAT_DIR = LANDSAT_DIRS[0]
LANDSAT_BANDS = {"coastal": "l.coastal", "blue": "l.blue", "green": "l.green", "red": "l.red", "nir": "l.nir",
                 "swir1": "l.swir1", "swir2": "l.swir2", "lst_k": "l.lst", "lst": "l.lst",
                 "l_coastal": "l.coastal", "l_blue": "l.blue", "l_lst_k": "l.lst", "l_lst": "l.lst"}   # band descriptions as COASTAL, L_COASTAL, LST_K, L_LST_K ...
SHP_SPEC = "{BOUND}/{County}.shp"

## 2. Common model grid and raster helpers

**What this cell does:** builds the common 30 m model grid for a county (extent, CRS, county mask) and the functions to read and write rasters on it. **Output:** a grid object used by all later steps. Nothing to edit.

In [ ]:
from rasterio.crs import CRS as RCRS

@dataclass
class Grid:
    transform: Affine
    width: int
    height: int
    crs: object
    inside: np.ndarray = None      # bool (height, width): True inside the county polygon

    @property
    def res(self):
        return abs(self.transform.a)

    def pad(self, n):
        return Grid(self.transform * Affine.translation(-n, -n), self.width + 2 * n, self.height + 2 * n, self.crs)

    def xy_to_rc(self, x, y):
        col = np.floor((np.asarray(x) - self.transform.c) / self.transform.a).astype(int)
        row = np.floor((self.transform.f - np.asarray(y)) / (-self.transform.e)).astype(int)
        return row, col

    def rc_to_xy(self, row, col):
        x = self.transform.c + (np.asarray(col) + 0.5) * self.transform.a
        y = self.transform.f + (np.asarray(row) + 0.5) * self.transform.e
        return x, y

    def same_geometry(self, path):
        try:
            with rasterio.open(path) as r:
                return (r.width == self.width and r.height == self.height and
                        all(abs(a - b) < 1e-6 for a, b in zip(tuple(r.transform)[:6], tuple(self.transform)[:6])))
        except Exception:
            return False

def write_tif(path, arr, grid, names=None, dtype="float32", nodata=np.nan):
    arr = np.asarray(arr)
    if arr.ndim == 2:
        arr = arr[None]
    prof = dict(driver="GTiff", height=arr.shape[1], width=arr.shape[2], count=arr.shape[0], dtype=dtype,
                crs=grid.crs, transform=grid.transform, nodata=nodata, compress="deflate", BIGTIFF="IF_SAFER")
    if arr.shape[1] >= 256 and arr.shape[2] >= 256:
        prof.update(tiled=True, blockxsize=256, blockysize=256)
    with rasterio.open(path, "w", **prof) as dst:
        dst.write(arr.astype(dtype))
        if names:
            for i, n in enumerate(names, 1):
                dst.set_band_description(i, n)
    return path

def read_county_boundary(county):
    """Polygon(s) of one county in the shapefile's own CRS. From the national file cfg.county_shp (rows picked by cfg.county_name_col), else boundary_dir/<County>.shp."""
    if cfg.county_shp:
        f = _ci_resolve(cfg.county_shp)
        if f is None:
            raise FileNotFoundError("County shapefile not found: %s" % cfg.county_shp)
        g = gpd.read_file(f)
        if g.crs is None:
            g = g.set_crs("EPSG:4326")
        if cfg.county_name_col not in g.columns:
            raise KeyError("column '%s' not in %s. Columns: %s" % (cfg.county_name_col, f, ", ".join(map(str, g.columns))))
        key = lambda s_: re.sub(r"[^a-z0-9]", "", str(s_).lower())
        sel = g[g[cfg.county_name_col].map(key) == key(county)]
        if sel.empty:
            raise ValueError("county '%s' not found in column '%s'. Names there: %s" % (county, cfg.county_name_col, ", ".join(sorted(map(str, g[cfg.county_name_col].unique()))[:80])))
        return sel
    f = find_file(fill_tokens(SHP_SPEC, county))
    if f is None:
        raise FileNotFoundError("Study area shapefile not found for %s: %s" % (county, fill_tokens(SHP_SPEC, county)))
    g = gpd.read_file(f)
    if g.crs is None:
        g = g.set_crs("EPSG:4326")
    return g

def county_geom(county):
    return read_county_boundary(county).to_crs(cfg.crs)

def make_grid(county):
    f = os.path.join(out_dir(county, "aligned"), "%s_%sm_grid.tif" % (county, cfg.res_m))
    if cfg.reuse_cache and os.path.exists(f):
        with rasterio.open(f) as r:
            return Grid(r.transform, r.width, r.height, r.crs, r.read(1) == 1)
    g = county_geom(county)
    xmin, ymin, xmax, ymax = g.total_bounds
    rs = float(cfg.res_m)
    xmin, xmax = math.floor(xmin / rs) * rs, math.ceil(xmax / rs) * rs
    ymin, ymax = math.floor(ymin / rs) * rs, math.ceil(ymax / rs) * rs
    w, h = int(round((xmax - xmin) / rs)), int(round((ymax - ymin) / rs))
    tr = Affine(rs, 0, xmin, 0, -rs, ymax)
    inside = features.rasterize([(geom, 1) for geom in g.geometry if geom is not None and not geom.is_empty],
                                out_shape=(h, w), transform=tr, fill=0, dtype="uint8").astype(bool)
    grid = Grid(tr, w, h, RCRS.from_user_input(cfg.crs), inside)
    write_tif(f, inside.astype("uint8"), grid, names=[".inside"], dtype="uint8", nodata=0)
    log("Grid ", county, ": ", w, " x ", h, " cells at ", cfg.res_m, " m (", format(w * h, ","), " cells)")
    return grid

def grid_path(county):
    return os.path.join(out_dir(county, "aligned"), "%s_%sm_grid.tif" % (county, cfg.res_m))

## 2.1 Layer stacks read from disk (window by window, or at points)

**What this cell does:** reads predictor layers window by window (or at points) so the full 30 m stack is never in memory. Nothing to edit.

In [ ]:
class Stack:
    """A list of (name, file, band) entries. Never holds the rasters in memory."""
    def __init__(self, items=None, meta=None):
        self.items = list(items or [])
        self.meta = dict(meta or {})

    @classmethod
    def from_file(cls, path, names):
        return cls([(n, path, i + 1) for i, n in enumerate(names)])

    @property
    def names(self):
        return [n for n, _, _ in self.items]

    def merged(self, *others):
        out, seen = [], set()
        for st in (self,) + others:
            if st is None:
                continue
            for it in st.items:
                if it[0] not in seen:
                    seen.add(it[0]); out.append(it)
        meta = dict(self.meta)
        for st in others:
            if st is not None:
                meta.update(st.meta)
        return Stack(out, meta)

    def _by_file(self):
        d = {}
        for n, p, b in self.items:
            d.setdefault(p, []).append((n, b))
        return d

    def read_window(self, window):
        cols = {}
        for p, nb in self._by_file().items():
            with rasterio.open(p) as src:
                a = src.read([b for _, b in nb], window=window).astype("float32")
                nd = src.nodata
                if nd is not None and np.isfinite(nd):
                    a[a == nd] = np.nan
                a[~np.isfinite(a)] = np.nan                       # -inf / inf are used as no-data in some index files
            for (n, _), arr in zip(nb, a):
                cols[n] = arr.ravel()
        return pd.DataFrame(cols)

    def read_array(self, name):
        for n, p, b in self.items:
            if n == name:
                with rasterio.open(p) as src:
                    a = src.read(b).astype("float32")
                    if src.nodata is not None and not np.isnan(src.nodata):
                        a[a == src.nodata] = np.nan
                return a
        raise KeyError(name)

    def read_points(self, rows, cols):
        rows, cols = np.asarray(rows), np.asarray(cols)
        out = {}
        for p, nb in self._by_file().items():
            with rasterio.open(p) as src:
                h, w = src.height, src.width
                ok = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
                a = src.read([b for _, b in nb]).astype("float32")
                nd = src.nodata
                if nd is not None and np.isfinite(nd):
                    a[a == nd] = np.nan
                a[~np.isfinite(a)] = np.nan
            for (n, _), arr in zip(nb, a):
                v = np.full(len(rows), np.nan, dtype="float32")
                v[ok] = arr[rows[ok], cols[ok]]
                out[n] = v
        return pd.DataFrame(out)

## 2.2 Align any raster to the grid (cached on disk)
Finer source than the grid: average (continuous) or mode (categorical), never subsample.
Coarser source: bilinear (continuous) or nearest (categorical).

**What this cell does:** resamples any raster onto the model grid and caches the result on disk, so the second run is much faster. Delete the cache folder if you replace an input file.

In [ ]:
def align_to_grid(src_path, grid, method, out_file):
    if cfg.reuse_cache and os.path.exists(out_file) and grid.same_geometry(out_file):
        return out_file
    with rasterio.open(src_path) as src:
        if src.crs is None:
            raise ValueError("source has no CRS: %s" % src_path)
        categorical = (method == "near")
        fine = src_res_m(src) < cfg.res_m * 0.75
        if fine:
            rs = Resampling.mode if categorical else Resampling.average
        else:
            rs = Resampling.nearest if categorical else Resampling.bilinear
        dst = np.full((src.count, grid.height, grid.width), np.nan, dtype="float32")
        infnd = src.nodata is not None and not np.isfinite(src.nodata) and not np.isnan(src.nodata)
        for b in range(1, src.count + 1):
            if infnd or (src.dtypes[b - 1].startswith("float") and src.nodata is None):
                a = src.read(b).astype("float32")                 # clean -inf/inf before averaging, one band at a time
                a[~np.isfinite(a)] = np.nan
                reproject(source=a, destination=dst[b - 1], src_transform=src.transform, src_crs=src.crs,
                          dst_transform=grid.transform, dst_crs=grid.crs, resampling=rs, src_nodata=np.nan, dst_nodata=np.nan)
            else:
                reproject(source=rasterio.band(src, b), destination=dst[b - 1],
                          src_transform=src.transform, src_crs=src.crs,
                          dst_transform=grid.transform, dst_crs=grid.crs,
                          resampling=rs, src_nodata=src.nodata, dst_nodata=np.nan)
    write_tif(out_file, dst, grid)
    return out_file

def find_sentinel(row, county, ym):
    """Sentinel-2 file of a month: F:/VI_prediction/Sentinel_AllBands_*/<County>/S2_<County>_<yyyy>_<mm>.tif (any folder year), else the old H-drive layout."""
    vr = cfg.v_root.rstrip("/\\")
    base = _ci_resolve(vr)
    pats = []
    if base and os.path.isdir(base):
        for d in sorted(os.listdir(base), reverse=True):
            if re.match(r"^Sentinel_AllBands_\d{4}$", d, re.I):
                pats.append("%s/%s/%s/S2_%s_%s_%s.tif" % (vr, d, county, county, ym[:4], ym[5:7]))
    pats.append(fill_tokens(row["rel"], county, ym))
    for p in pats:
        r = find_file(p, ym)
        if r:
            return r
    return None

def row_file(row, county, ym):
    """file of one registry row for a month (or static)"""
    if row["name"] == "sentinel":
        return find_sentinel(row, county, ym)
    return find_file(fill_tokens(row["rel"], county, ym), ym)

VI_FC_NAMES = {"cvi", "mcari", "ndmi", "ndre", "ndvi", "pri", "psri", "rdvi", "savi", "mtvi2"}   # the indices Notebook C forecasts

def _is_fc_ym(ym):
    return bool(ym) and ym in set(cfg.vi_fc_months or [])

def fc_vi_file(county, name, ym):
    """Notebook C forecast GeoTIFF of one index and month, or None."""
    f = os.path.join(cfg.vi_fc_dir, cfg.vi_fc_scenario, county, "%s_%s_%s-%s.tif" % (county, name.upper(), ym[:4], ym[5:7]))
    return _ci_resolve(f)

S2_BAND_NAMES = {"B1": "s.coastal", "B2": "s.blue", "B3": "s.green", "B4": "s.red", "B5": "s.red.edge1", "B6": "s.red.edge2", "B7": "s.red.edge3",
                 "B8": "s.nir", "B8A": "s.red.edge4", "B9": "s.vapour", "B11": "s.swir1", "B12": "s.swir2"}
S2_ORDER_11 = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12", "fill_flag"]      # the F-drive S2_<County>_<yyyy>_<mm>.tif files (no B1 coastal, no B9 vapour)
_S2_WARNED = set()
# Band LABELS as they are written in the files (NOT the Sentinel-2 band codes):
#  - the Sentinel_AllBands files are labelled B01, B02, ... B11 by POSITION: B03 = red, B06 = red edge 3, B07 = NIR (B02 = green and B01 = blue by the same order;
#    B04 = red edge 1, B05 = red edge 2, B08 = red edge 4, B09 = SWIR1, B10 = SWIR2, B11 = fill flag)
#  - the C_WV (coastal and water vapour) files are labelled B1 = coastal and B2 = water vapour
S2_POS_NAMES = {1: "s.blue", 2: "s.green", 3: "s.red", 4: "s.red.edge1", 5: "s.red.edge2", 6: "s.red.edge3", 7: "s.nir", 8: "s.red.edge4", 9: "s.swir1", 10: "s.swir2", 11: "fill_flag"}
CWV_POS_NAMES = {1: "s.coastal", 2: "s.vapour"}

def _band_label(d):
    """'Band 03: B03' / 'B03' / 'B3' -> ('B03' text, number) ; 'fill_flag' -> ('fill_flag', None) ; else (None, None)"""
    d = re.sub(r"^\s*band\s*\d+\s*:\s*", "", (d or "").strip(), flags=re.I)
    if d.lower() == "fill_flag":
        return "fill_flag", None
    m = re.fullmatch(r"B0*(\d{1,2})(A?)", d, re.I)
    return (d, int(m.group(1))) if m else (None, None)

def sentinel_band_names(path, nb, default12, layer="sentinel"):
    """Predictor name of every band of a Sentinel-2 file.
    layer 'sentinel_cwv': labels B1 = s.coastal, B2 = s.vapour (position 1 and 2 if a band has no label).
    layer 'sentinel'    : labels B01..B11 are POSITIONS (B03 = s.red, B06 = s.red.edge3, B07 = s.nir, see S2_POS_NAMES). Older files that carry real
                          Sentinel-2 codes (B2, B4, B8, B8A, B12 ...) or no labels still work as before (11 bands = S2_ORDER_11)."""
    with rasterio.open(path) as s:
        desc = [(d or "") for d in s.descriptions]
    lab = [_band_label(d) for d in desc]
    if layer == "sentinel_cwv":
        names = []
        for k, (t, num) in enumerate(lab, 1):
            n_ = num if num is not None else k
            names.append(CWV_POS_NAMES.get(n_, "s.cwv_b%d" % n_))
    else:
        padded = [bool(re.fullmatch(r"(?:band\s*\d+\s*:\s*)?B\d{2}", (d or "").strip(), re.I)) for d in desc]
        nums = [n_ for _, n_ in lab if n_ is not None]
        positional = (len(desc) == nb and nb <= 11 and any(padded) and nums and max(nums) <= 11 and
                      all(p_ or t == "fill_flag" for p_, (t, _) in zip(padded, lab)))
        if positional:
            names = [("fill_flag" if t == "fill_flag" else S2_POS_NAMES.get(n_, "s.b%02d" % n_)) for t, n_ in lab]
        else:
            codes = []
            for d in desc:
                m = re.search(r"\b(B\d{1,2}A?|fill_flag)\b", d, re.I)
                codes.append(m.group(1).upper().replace("FILL_FLAG", "fill_flag") if m else None)
            if len(codes) == nb and all(c is not None for c in codes):
                pass
            elif nb == 11:
                codes = list(S2_ORDER_11)
            elif nb >= len(default12):
                return list(default12) + ["s.extra%d" % (k + 1) for k in range(nb - len(default12))]
            else:
                return None
            names = [S2_BAND_NAMES.get(c, "s." + c.lower()) for c in codes]
    if (path, layer) not in _S2_WARNED:
        _S2_WARNED.add((path, layer)); log("Sentinel-2 bands in ", os.path.basename(path), " (", layer, "): ", ", ".join("%s=%s" % (d or "?", n) for d, n in zip(desc, names)))
    return names

def _fill_fc_with_mean(f, grid, name, county, ym):
    """Notebook C leaves the pixels it could not forecast as no data. Inside the county they are filled with the county MEAN of
    that index and month (instead of a training median later on). Safe to call again: a filled file has nothing left to fill."""
    with rasterio.open(f) as r:
        a = r.read().astype("float32"); desc = list(r.descriptions)
    ins = grid.inside if grid.inside is not None else np.ones(a.shape[1:], bool)
    ok = np.isfinite(a[0])
    miss = ins & ~ok
    if not miss.any():
        return
    if not (ins & ok).any():
        warn(county, " ", name, " ", ym, ": Notebook C layer has no valid pixel inside the county, nothing to average.")
        return
    mean = float(np.nanmean(a[0][ins & ok]))
    a[0][miss] = mean
    write_tif(f, a, grid, names=desc if any(desc) else None)
    log(county, " ", name, " ", ym, ": ", format(int(miss.sum()), ","), " no-data pixels (", "%.0f%%" % (100.0 * miss.sum() / max(int(ins.sum()), 1)),
        " of the county) filled with the county mean ", "%.4f" % mean)

def get_layer(row, county, grid, ym=None):
    """Aligned file for one registry row. Returns (names, path) or None.
    For a forecast month (cfg.vi_fc_months) the vegetation index layers are read from Notebook C's GeoTIFFs."""
    is_fc = row["kind"] == "monthly" and row["name"] in VI_FC_NAMES and _is_fc_ym(ym)
    if is_fc:
        p = fc_vi_file(county, row["name"], ym)
        if p is None:
            warn("no Notebook C forecast file for ", row["name"], " ", ym, " in ", os.path.join(cfg.vi_fc_dir, cfg.vi_fc_scenario, county))
    else:
        p = row_file(row, county, ym)
    if p is None:
        return None
    tag = "%s_%s" % (row["name"], ym) if row["kind"] in ("monthly", "monthly_mb") else row["name"]
    if is_fc:
        tag += "_fc_" + cfg.vi_fc_scenario          # separate cache file, never mixed with an observed composite
    f = os.path.join(out_dir(county, "aligned"), tag + ".tif")
    try:
        align_to_grid(p, grid, row["method"], f)
    except Exception as e:
        warn("could not align ", row["name"], " (", e, ")")
        return None
    if is_fc:
        _fill_fc_with_mean(f, grid, row["name"], county, ym)
    with rasterio.open(f) as r:
        nb = r.count
    if row["kind"] == "monthly_mb":
        names = sentinel_band_names(p, nb, row["bands"], row["name"])
        if names is None:
            warn(row["name"], " has ", nb, " bands and no usable band descriptions. Skipped.")
            return None
        return names, f
    return [row["name"]], f

## 3. Static layers, terrain and the host layer

**What this cell does:** derives slope, aspect and elevation from the DEM and builds the host layer. By default (`host_mode = all_land`) every cell in the county is host and the crop map enters only as the predictor `crop.type`, because the crop map misses most of the field records. `host_mode = crop_map` uses mapped cropland only (non-host cells then get risk 0).

In [ ]:
def terrain_from_dem(dem_pad, res):
    """slope (deg), northness, eastness, roughness (TRI) from a DEM padded by 2 cells. Horn method, as in terra."""
    z = dem_pad.astype("float64")
    a, b, c = z[:-2, :-2], z[:-2, 1:-1], z[:-2, 2:]
    d, e, f = z[1:-1, :-2], z[1:-1, 1:-1], z[1:-1, 2:]
    g, h, i = z[2:, :-2], z[2:, 1:-1], z[2:, 2:]
    dzdx = ((c + 2 * f + i) - (a + 2 * d + g)) / (8 * res)
    dzdy = ((a + 2 * b + c) - (g + 2 * h + i)) / (8 * res)          # north minus south
    slope = np.degrees(np.arctan(np.hypot(dzdx, dzdy)))
    asp = np.arctan2(-dzdx, -dzdy)                                  # downslope direction, clockwise from north
    flat = (np.hypot(dzdx, dzdy) == 0) & np.isfinite(e)
    nor = np.where(flat, 0.0, np.cos(asp)); eas = np.where(flat, 0.0, np.sin(asp))
    tri = (np.abs(a - e) + np.abs(b - e) + np.abs(c - e) + np.abs(d - e) +
           np.abs(f - e) + np.abs(g - e) + np.abs(h - e) + np.abs(i - e)) / 8.0
    def emb(x):
        o = np.full(z.shape, np.nan); o[1:-1, 1:-1] = x; return o
    return emb(e), emb(slope), emb(nor), emb(eas), emb(tri)

def get_static(county, grid):
    stk, missing = Stack(), []
    for _, row in REG[REG.kind.isin(["static", "cat"]) & (REG.name != "dem")].iterrows():
        if not row["always"] and row["name"] not in needed_names():
            continue
        got = get_layer(row, county, grid)
        if got is None:
            missing.append(row["name"])
        else:
            stk = stk.merged(Stack.from_file(got[1], got[0]))
    if missing:
        log("Static layers not found (skipped): ", ", ".join(missing))
    tf = os.path.join(out_dir(county, "aligned"), "terrain.tif")
    drow = REG[REG.name == "dem"].iloc[0]
    p = find_file(fill_tokens(drow["rel"], county))
    TN = ["elevation", "slope", "northness", "eastness", "roughness"]
    if p is not None:
        if not (cfg.reuse_cache and os.path.exists(tf) and grid.same_geometry(tf)):
            gp = grid.pad(2)
            dp = align_to_grid(p, gp, "bilinear", os.path.join(out_dir(county, "aligned"), "dem_pad2.tif"))
            with rasterio.open(dp) as r:
                dem = r.read(1).astype("float32")
            layers = terrain_from_dem(dem, grid.res)
            ter = np.stack([x[2:-2, 2:-2] for x in layers]).astype("float32")
            write_tif(tf, ter, grid, names=TN)
        stk = stk.merged(Stack.from_file(tf, TN))
    else:
        warn("No DEM for ", county, ". Terrain predictors and temperature lapse correction are unavailable.")
    return stk

def make_host(county, grid, static):
    tag = cfg.host_mode if cfg.host_mode != "crop_map" else "cropmap_%s" % ("_".join(str(int(c)) for c in cfg.maize_codes) if cfg.maize_codes else "mask%s" % cfg.crop_mask_crop_value)
    f = os.path.join(out_dir(county, "aligned"), "host_%s.tif" % tag)          # the mode is in the file name, so an old cached host layer is never reused
    if cfg.reuse_cache and os.path.exists(f) and grid.same_geometry(f):
        return Stack.from_file(f, ["host"])
    nm = static.names
    if cfg.host_mode == "all_land":
        h = np.ones((grid.height, grid.width), "float32")
    elif cfg.maize_codes is not None and "crop.type" in nm:
        h = np.isin(static.read_array("crop.type"), cfg.maize_codes).astype("float32")
    elif "crop.mask" in nm:
        h = (static.read_array("crop.mask") == cfg.crop_mask_crop_value).astype("float32")
    else:
        warn("No crop mask or crop type for ", county, ". Host set to 1 everywhere.")
        h = np.ones((grid.height, grid.width), "float32")
    h = np.where(np.isnan(h), 0, h).astype("float32")
    if cfg.host_mode != "all_land" and cfg.host_window > 1:
        # share of cropland in the neighbourhood (0 to 1). A GPS point a few metres off the mapped field edge
        # still sees host, and mixed cells carry proportionally less risk.
        k = int(cfg.host_window)
        num = ndimage.uniform_filter(h, size=k, mode="constant", cval=0.0)
        den = ndimage.uniform_filter(np.ones_like(h), size=k, mode="constant", cval=0.0)
        h = (num / den).astype("float32")
    write_tif(f, h, grid, names=["host"])
    log("host layer (", cfg.host_mode, "): ", "%.1f" % (100.0 * float((h > 0).mean())), "% of the grid is host")
    return Stack.from_file(f, ["host"])

## 4. Monthly layers (vegetation indices, Sentinel-2, Landsat, LST)

**What this cell does:** finds the vegetation index, Sentinel-2, Landsat and LST layers for each month and picks the right one for a date. Missing months are reported, not fatal.

In [ ]:
def available_months(county):
    ref = REG[(REG.kind == "monthly") & REG.name.isin(["ndvi", "savi", "ndre", "mcari", "rdvi"])]
    out = []
    for y in range(int(cfg.monthly_years[0]), int(cfg.monthly_years[1]) + 1):
        for m in range(1, 13):
            ym = "%d_%02d" % (y, m)
            if any(find_file(fill_tokens(r, county, ym), ym) is not None for r in ref.rel):
                out.append(ym)
    return out

def needed_names():
    """layers any selected pest uses, plus the sources of derived predictors"""
    names = set()
    for p in cfg.pests:
        names |= set(PRED_SETS.get(p, []))
    if "mtvi2" in names:
        names |= {"s.nir", "s.red", "s.green"}             # fallback: MTVI2 is derived from Sentinel-2 if its raster is missing
    names |= {ALIASES[a] for a in ALIASES if a in names}
    return names

def needs_landsat():
    return any(n.startswith("l.") for n in needed_names())

_WARNED_YM = set()
def pick_ym(date, avail, allow_fc=True):
    """Monthly composite for a record date. If that month has no composite, use the SAME calendar month of the nearest
    year that has one (seasonal match), because a composite from another season would describe a different crop condition."""
    ym = ym_of(date)
    if allow_fc and _is_fc_ym(ym):
        return ym                      # forecast month: the index layers come from Notebook C (see get_layer)
    if ym in avail:
        return ym
    if not avail:
        return cfg.ym_fallback
    nm = lambda s_: int(s_[:4]) * 12 + int(s_[5:7])
    same = [a for a in avail if a[5:7] == ym[5:7]]
    if cfg.month_fallback == "same_month" and same:
        best = min(same, key=lambda a: abs(nm(a) - nm(ym)))
        how = "same calendar month of the nearest year"
    else:
        best = min(avail, key=lambda a: abs(nm(a) - nm(ym)))
        how = "nearest month"
    if ym not in _WARNED_YM:
        warn("no monthly composite for ", ym, ". Using ", best, " (", how, ")."); _WARNED_YM.add(ym)
    return best

_MONTHLY_CACHE = {}
_AVAIL_CACHE = {}
def get_monthly(county, grid, ym):
    key = (county, ym, cfg.res_m, cfg.vi_fc_scenario, cfg.fc_s2_same_month)
    if key in _MONTHLY_CACHE:
        return _MONTHLY_CACHE[key]
    stk = Stack()
    for _, row in REG[REG.kind.isin(["monthly", "monthly_mb"])].iterrows():
        nn = needed_names()
        used = any(b in nn for b in row["bands"]) if row["kind"] == "monthly_mb" else row["name"] in nn
        if not used:
            continue
        ym_row = ym
        if _is_fc_ym(ym) and not (row["kind"] == "monthly" and row["name"] in VI_FC_NAMES) and cfg.fc_s2_same_month:
            # Sentinel-2 bands of a forecast month do not exist: same calendar month of the nearest earlier year (seasonal stand-in)
            # most recent earlier year whose file for THIS layer exists (checked on the Sentinel-2 files themselves, not on the index composites)
            cands = ["%d_%s" % (int(cfg.fc_prior_year), ym[5:7])]               # pinned: the same month of cfg.fc_prior_year only
            ym_row = next((c_ for c_ in cands if row_file(row, county, c_) is not None), None)
            if ym_row is None:
                warn(county, ": no ", cands[0], " Sentinel-2 composite found for ", ym, ". Sentinel-2 predictors get training medians.")
                ym_row = ym
            elif (county, ym, ym_row) not in _WARNED_YM:
                warn(county, ": forecast month ", ym, ": Sentinel-2 bands taken from ", ym_row, " (previous year, same month).")
                _WARNED_YM.add((county, ym, ym_row))
        got = get_layer(row, county, grid, ym_row)
        if got is not None:
            stk = stk.merged(Stack.from_file(got[1], got[0]))
    if not stk.items:
        _MONTHLY_CACHE[key] = None
        return None
    if "s.nir" in stk.names:
        stk.meta["nir_max"] = float(np.nanmax(stk.read_array("s.nir")))
    _MONTHLY_CACHE[key] = stk
    return stk

def derive_monthly_columns(df, meta):
    """MTVI2 (Haboudane et al. 2004) is in the aphid predictor list but no MTVI2 raster exists: derive it from Sentinel-2."""
    need = ["s.nir", "s.red", "s.green"]
    if "mtvi2" not in df.columns and all(n in df.columns for n in need):
        nir, red, grn = df["s.nir"], df["s.red"], df["s.green"]
        if meta.get("nir_max", 0) > 2:                    # reflectance stored as 0 to 10000
            nir, red, grn = nir / 10000.0, red / 10000.0, grn / 10000.0
        num = 1.5 * (1.2 * (nir - grn) - 2.5 * (red - grn))
        den = np.sqrt((2 * nir + 1) ** 2 - (6 * nir - 5 * np.sqrt(red)) - 0.5)
        df["mtvi2"] = (num / den).astype("float32")
    for a, b in ALIASES.items():
        if b in df.columns and a not in df.columns:
            df[a] = df[b]
    for c, (lo, hi) in (cfg.clip or {}).items():
        if c in df.columns:
            df[c] = df[c].clip(lo, hi)
    return df

def _landsat_dates(fl):
    """month of each Landsat file. Accepts ..._2026_09.tif (Notebook A's names) and ..._2026_09_01.tif"""
    out = []
    for f in fl:
        m = re.search(r"(?<!\d)((?:19|20)\d{2})[_-](\d{2})(?:[_-](\d{2}))?(?!\d)", os.path.basename(f))
        try:
            out.append(pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3) or 1)) if m else pd.NaT)
        except ValueError:
            out.append(pd.NaT)
    return pd.DatetimeIndex(out)

_LS_IDX = {}
def landsat_index(county):
    """sorted scene dates and files for a county, or (None, None) if the county has no Landsat folder"""
    if county in _LS_IDX:
        return _LS_IDX[county]
    d = next((x for x in (find_dir(fill_tokens(p_, county)) for p_ in LANDSAT_DIRS) if x), None)
    res = (None, None)
    if d:
        fl = [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(".tif")]
        dts = _landsat_dates(fl)
        ok = ~pd.isna(dts)
        fl = [f for f, k in zip(fl, ok) if k]; dts = dts[ok]
        if len(fl):
            o = np.argsort(dts.values)
            res = (pd.DatetimeIndex(dts[o]), [fl[i] for i in o])
            log("Landsat ", county, " (", d, "): ", len(fl), " stacks ", res[0].min().date(), " to ", res[0].max().date())
    if res[0] is None:
        warn("no Landsat scenes for ", county, ". l.* predictors will be missing there.")
    _LS_IDX[county] = res
    return res

def nearest_scene(county, date):
    """Landsat MONTHLY stack for the month of `date` (files are named ..._YYYY_MM_01.tif, one per month).
    A month without a stack (2025 onward, and the forecast month) takes the same calendar month of the nearest
    year that has one: a seasonal stand-in. Returns (file, timestamp) like the old per-scene function."""
    dts, fl = landsat_index(county)
    if dts is None:
        return None
    d = pd.Timestamp(date)
    key = np.asarray(dts.year * 12 + dts.month)
    want = d.year * 12 + d.month
    hit = np.flatnonzero(key == want)
    if len(hit):
        return fl[int(hit[0])], dts[int(hit[0])]
    same = np.flatnonzero(np.asarray(dts.month) == d.month)
    if not len(same):
        warn(county, ": no Landsat stack for calendar month ", d.month, ". l.* predictors get training medians for ", ym_of(d), ".")
        return None
    i = int(same[np.argmin(np.abs(key[same] - want))])
    if (county, "ls", ym_of(d)) not in _WARNED_YM:
        _WARNED_YM.add((county, "ls", ym_of(d)))
        warn(county, ": no Landsat stack for ", ym_of(d), ". Using ", ym_of(dts[i]), " (same calendar month of the nearest year).")
    return fl[i], dts[i]

_LS_CACHE = {}
def get_landsat_stack(county, grid, scene_file):
    """aligned Landsat scene as a Stack with names l.coastal, l.blue, l.lst (deg C)"""
    if scene_file in _LS_CACHE:
        return _LS_CACHE[scene_file]
    tag = re.sub(r"[^0-9]", "", os.path.basename(scene_file))
    f = os.path.join(out_dir(county, "aligned"), "landsatm_%s.tif" % tag)
    stk = None
    try:
        with rasterio.open(scene_file) as s:
            desc = [(x or "").lower() for x in s.descriptions]
        names = [LANDSAT_BANDS.get(x) for x in desc]
        if not any(names) and len(desc) == 3:
            names = ["l.coastal", "l.blue", "l.lst"]            # Notebook A: band 1 = COASTAL, 2 = BLUE, 3 = LST_K (files without band descriptions)
        if not (cfg.reuse_cache and os.path.exists(f) and grid.same_geometry(f)):
            align_to_grid(scene_file, grid, "bilinear", f)
            with rasterio.open(f) as r:
                a = r.read().astype("float32")
            for i, n in enumerate(names):
                if n == "l.lst" and np.isfinite(a[i]).any() and np.nanmedian(a[i]) > 150:
                    a[i] = a[i] - 273.15                         # Kelvin to deg C
            write_tif(f, a, grid, names=[n or "skip%d" % (i + 1) for i, n in enumerate(names)])
        keep = [(n, f, i + 1) for i, n in enumerate(names) if n]
        stk = Stack(keep)
    except Exception as e:
        warn("could not prepare Landsat scene ", os.path.basename(scene_file), " (", e, ")")
    _LS_CACHE[scene_file] = stk
    return stk

def data_report(county):
    rows = []
    for _, r in REG.iterrows():
        ym = cfg.ym_fallback if r["kind"] in ("monthly", "monthly_mb") else None
        rows.append((r["name"], r["kind"], row_file(r, county, ym) is not None))
    layers = pd.DataFrame(rows, columns=["layer", "kind", "found"])
    wx = {}
    for v, t in WX_SPEC.items():
        d = find_dir(fill_tokens(t, county))
        wx[v] = len([f for f in os.listdir(d) if f.lower().endswith(".tif")]) if d else 0
    inc = {p: find_file(fill_tokens(t, county)) is not None for p, t in INC_SPEC.items()}
    return dict(layers=layers, weather_files=wx, incidence=inc)

In [ ]:
# ---- add the coastal + water vapour stack (B1, B9) as a second Sentinel-2 source ----
CWV_FOLDER = {"Kakamega": "C_WV_KACH", "Kilifi": "C_WV_KILIFI", "Nyeri": "C_WV_NYERI"}

def find_cwv(county, ym):
    key = CWV_FOLDER.get(county, "C_WV_" + county.upper())
    p = "%s/coastal_water_vapour/%s/S2_CWV_%s_%s_%s.tif" % (cfg.v_root.rstrip("/\\"), key, county, ym[:4], ym[5:7])
    return find_file(p, ym)

if "sentinel_cwv" not in set(REG.name):
    REG = pd.concat([pd.DataFrame([dict(name="sentinel_cwv", rel="", kind="monthly_mb", method="bilinear",
                                             bands=["s.coastal", "s.vapour"], always=False)]), REG], ignore_index=True)   # first: its B1 / B9 win over any B1 / B9 in the main file

def row_file(row, county, ym):
    """file of one registry row for a month (or static)"""
    if row["name"] == "sentinel":
        return find_sentinel(row, county, ym)
    if row["name"] == "sentinel_cwv":
        return find_cwv(county, ym)
    return find_file(fill_tokens(row["rel"], county, ym), ym)

# clear caches so the new layers are picked up
_MONTHLY_CACHE.clear()

# ---- check: bands and files for all three counties ----
rows = []
for c in ["Kakamega", "Kilifi", "Nyeri"]:
    for ym in ["2020_01", "2024_10", "2025_10", "2026_09"]:
        f1 = find_sentinel(REG[REG.name == "sentinel"].iloc[0], c, ym)
        f2 = find_cwv(c, ym)
        nb1 = nb2 = None
        if f1:
            with rasterio.open(f1) as s: nb1 = s.count
        if f2:
            with rasterio.open(f2) as s: nb2 = s.count
        rows.append(dict(county=c, ym=ym, main_found=bool(f1), main_bands=nb1, cwv_found=bool(f2), cwv_bands=nb2))
print(pd.DataFrame(rows).to_string(index=False))
# (the check of s.coastal / s.vapour in the predictor lists is in the cell after PRED_SETS: PRED_SETS does not exist yet here)


## 5. Daily weather
Daily stacks stay at their native resolution (a few km) in memory as small arrays. For any date we compute
rolling-window features ending on that date, interpolate those few feature layers to the model grid and correct
temperature for terrain. This avoids resampling 1000+ daily layers to 30 m.

**What this cell does:** reads daily weather and turns it into rolling 7, 14 and 30 day features ending on each record or map date, corrected for terrain. **Output:** weather predictors per date.

In [ ]:
def read_wx_var(county, v):
    d = find_dir(fill_tokens(WX_SPEC[v], county))
    if d is None:
        return None
    fl = [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(".tif")]
    if not fl:
        return None
    dts = dates_from_files(fl)
    ok = ~pd.isna(dts)
    if (~ok).any():
        warn(v, ": ", int((~ok).sum()), " files without a yyyy_mm_dd date in the name were ignored")
    fl = [f for f, k in zip(fl, ok) if k]; dts = dts[ok]
    o = np.argsort(dts.values); fl = [fl[i] for i in o]; dts = dts[o]
    dup = dts.duplicated()
    if dup.any():
        warn(v, ": ", int(dup.sum()), " duplicate dates dropped")
        fl = [f for f, k in zip(fl, ~dup) if k]; dts = dts[~dup]
    keep = np.ones(len(dts), bool)
    if cfg.weather_start:
        keep &= np.asarray(dts >= pd.Timestamp(cfg.weather_start))
    if cfg.weather_end:
        keep &= np.asarray(dts <= pd.Timestamp(cfg.weather_end))
    fl = [f for f, k in zip(fl, keep) if k]; dts = dts[keep]
    if not fl:
        return None
    with rasterio.open(fl[0]) as s:
        tm = dict(transform=s.transform, crs=s.crs, height=s.height, width=s.width, nb=s.count,
                  names=[x or "" for x in s.descriptions])
    return dict(files=fl, dates=pd.DatetimeIndex(dts), tmpl=tm)

def band_stacks(x, ks):
    """read the requested bands of every daily file in one pass: {band: array(days, rows, cols)}.
    Files whose size or position differs from the first file (the template) are resampled onto the template grid (bilinear), so one odd file no longer stops the run."""
    out = {k: [] for k in ks}
    tm = x["tmpl"]; n_re = 0; shapes = {}
    for f in x["files"]:
        with rasterio.open(f) as s:
            nd = s.nodata
            same = (s.height == tm["height"] and s.width == tm["width"] and
                    all(abs(p_ - q_) < 1e-6 * max(1.0, abs(q_)) for p_, q_ in zip(tuple(s.transform)[:6], tuple(tm["transform"])[:6])))
            if same:
                a = s.read(list(ks)).astype("float32")
                if nd is not None and not np.isnan(nd):
                    a[a == nd] = np.nan
            else:
                n_re += 1; shapes[(s.height, s.width)] = shapes.get((s.height, s.width), 0) + 1
                a = np.full((len(ks), tm["height"], tm["width"]), np.nan, dtype="float32")
                for j, k in enumerate(ks):
                    src = s.read(k).astype("float32")
                    if nd is not None and not np.isnan(nd):
                        src[src == nd] = np.nan
                    reproject(src, a[j], src_transform=s.transform, src_crs=s.crs, dst_transform=tm["transform"], dst_crs=tm["crs"],
                              resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
                    m_ = np.isnan(a[j])
                    if m_.any() and not m_.all():          # cells outside this (smaller) file: take the nearest cell of the file (the grids line up, so inside values are unchanged)
                        a[j] = a[j][tuple(ndimage.distance_transform_edt(m_, return_distances=False, return_indices=True))]
        for k, arr in zip(ks, a):
            out[k].append(arr)
    if n_re:
        warn(n_re, " of ", len(x["files"]), " daily files had a different size or position than the first file (", tm["height"], "x", tm["width"], "). Resampled onto it, cells outside the smaller file filled from its nearest edge cell (about 0.1 degree). Other sizes found: ", shapes)
    return {k: np.stack(v) for k, v in out.items()}

def wind_band_map(names, nb):
    """which band is speed, direction, u and v: from the band names if they exist, else cfg.wind_bands"""
    pats = dict(speed=r"speed|spd|^ws|wspd", direction=r"dir|deg", u=r"(^|[^a-z])u([^a-z]|$)|u_?mean|u_?comp|zonal|east",
                v=r"(^|[^a-z])v([^a-z]|$)|v_?mean|v_?comp|meridional|north")
    found = {}
    for key, p in pats.items():
        for i, n in enumerate(names, 1):
            if n and re.search(p, n.lower()) and i not in found.values():
                found[key] = i; break
    if len(found) == 4:
        log("wind bands from band names: ", found)
        return found
    log("wind bands taken from cfg.wind_bands: ", cfg.wind_bands, " (band names found: ", names, ")")
    return dict(cfg.wind_bands)

def fill_rh_from_donor(county, W):
    """Days without RH get the county-mean RH of a donor county (cfg.rh_fill_from). A coarse stand-in: the donor may be far away."""
    donor = (cfg.rh_fill_from or {}).get(county)
    if not donor or donor == county or "tmax" not in W:
        return
    want = pd.DatetimeIndex(W["tmax"]["dates"])
    have = W["rh"]["dates"] if "rh" in W else pd.DatetimeIndex([])
    miss = want.difference(have)
    if len(miss) == 0:
        return
    xd = read_wx_var(donor, "rh")
    if xd is None:
        warn("RH fill: no donor RH files in ", donor); return
    pos = {d: i for i, d in enumerate(xd["dates"])}
    use = [d for d in miss if d in pos]
    if not use:
        warn("RH fill: donor ", donor, " has none of the ", len(miss), " missing days"); return
    if "rh" in W:
        tm = W["rh"]["tmpl"]; base = W["rh"]["M"]; shp = base.shape[1:]
        mask = np.isfinite(base[0])
    else:
        warn("RH: ", county, " has no RH files at all. Donor county mean ", donor, " used on the tmax grid.")
        tm = W["tmax"]["tmpl"]; shp = (tm["height"], tm["width"]); base = np.zeros((0,) + shp, "float32"); mask = np.ones(shp, bool)
        have = pd.DatetimeIndex([])
    newM = []
    for d in use:
        with rasterio.open(xd["files"][pos[d]]) as s:
            a = s.read(1).astype("float32")
        v = np.nanmean(a)
        if np.nanmax(a) <= 1.5:
            v = v * 100
        newM.append(np.where(mask, v, np.nan).astype("float32"))
    allM = np.concatenate([base, np.stack(newM)]); alld = have.append(pd.DatetimeIndex(use))
    o = np.argsort(alld.values)
    W["rh"] = dict(M=allM[o], dates=pd.DatetimeIndex(alld[o]), tmpl=tm)
    warn("RH for ", county, ": ", len(use), " of ", len(want), " days filled with the ", donor, " county-mean RH (surrogate, not measured in ", county, ")")
    try:
        pd.DataFrame(dict(date=[str(d.date()) for d in use], source=donor)).to_csv(
            os.path.join(out_dir(county, "models"), "rh_filled_days.csv"), index=False)
    except Exception:
        pass

# ---------------- SEAS5 forecast days appended after the historical record ----------------
_FC_LUT = {}
def _fc_bias_lut():
    if "lut" not in _FC_LUT:
        lut = {}
        if cfg.fc_bias_vars:
            if cfg.fc_bias_csv and os.path.exists(cfg.fc_bias_csv):
                b = pd.read_csv(cfg.fc_bias_csv)
                for r in b.to_dict("records"):
                    if np.isfinite(r["value"]):
                        lut[(r["county"], r["var"], int(r["start_month"]), int(r["lead"]))] = (r["kind"], float(r["value"]))
            else:
                warn("forecast bias table not found: ", cfg.fc_bias_csv, ". No bias correction applied.")
        _FC_LUT["lut"] = lut
    return _FC_LUT["lut"]

def _fc_files(county, group, name):
    d = os.path.join(cfg.fc_root, group, name, cfg.fc_scenario)
    pat = re.compile(r"^%s_%s_%s_(\d{4}-\d{2}-\d{2})\.tif$" % (re.escape(county), re.escape(name), re.escape(cfg.fc_scenario)), re.I)
    out = {}
    if os.path.isdir(d):
        for f in os.listdir(d):
            m = pat.match(f)
            if m:
                out[pd.Timestamp(m.group(1))] = os.path.join(d, f)
    return out

def _fc_on_template(path, tmpl, pre=None):
    """one forecast day on the grid of the historical layer (bilinear; NaN rim filled from the nearest cell)"""
    with rasterio.open(path) as s:
        a = s.read(1).astype("float32")
        if s.nodata is not None and not np.isnan(s.nodata):
            a[a == s.nodata] = np.nan
        if pre is not None:
            a = pre(a)
        out = np.full((tmpl["height"], tmpl["width"]), np.nan, "float32")
        reproject(a, out, src_transform=s.transform, src_crs=s.crs, dst_transform=tmpl["transform"], dst_crs=tmpl["crs"],
                  resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    m = np.isnan(out)
    if m.any() and not m.all():
        out = out[tuple(ndimage.distance_transform_edt(m, return_distances=False, return_indices=True))]
    return out

def _fc_correct(arr, bias_var, county, date, start):
    """Notebook D's bias value for (county, variable, start month, lead). delta: subtract | ratio: multiply."""
    if bias_var not in (cfg.fc_bias_vars or []):
        return arr
    lead = (pd.Timestamp(date).to_period("M") - pd.Timestamp(start).to_period("M")).n
    e = _fc_bias_lut().get((county, bias_var, int(pd.Timestamp(start).month), int(lead)))
    if e is None:
        return arr
    out = arr - e[1] if e[0] == "delta" else arr * e[1]
    if bias_var == "rh": out = np.clip(out, 0, 100)
    elif bias_var in ("rain", "wind"): out = np.clip(out, 0, None)
    return out

def extend_with_forecast(county, W):
    """Append the forecast days that come AFTER the last historical day of each variable. Rows are resampled to the grid of the historical layer, so the
    lapse-rate correction to 30 m (elev_coarse) keeps working unchanged. Units: temperature degC (Kelvin converted), rain mm/day, RH %, wind m/s."""
    if not cfg.use_forecast:
        return
    only = getattr(cfg, "fc_only_vars", None)          # None = every variable, else the list of variables that SEAS5 may fill (keys: tmax, tmin, precip, rh, wind)
    spec = {"tmax": ("temperature", "tmax", "temp"), "tmin": ("temperature", "tmin", "temp"),
            "precip": ("rainfall", "rain", "rain"), "rh": ("humidity", "rh", "rh")}
    for v, (g, n, bv) in spec.items():
        if only is not None and v not in only:
            continue
        if v not in W:
            warn("forecast: no historical ", v, " for ", county, "; forecast not appended"); continue
        ff = _fc_files(county, g, n)
        if not ff:
            warn("forecast: no ", n, " files for ", county, " in ", os.path.join(cfg.fc_root, g, n, cfg.fc_scenario)); continue
        start = min(ff)
        new = sorted(d for d in ff if d > W[v]["dates"].max())
        if not new:
            continue
        tm = W[v]["tmpl"]; arrs = []
        for d in new:
            a = _fc_on_template(ff[d], tm)
            if v in ("tmax", "tmin") and np.isfinite(np.nanmedian(a)) and np.nanmedian(a) > 150:
                a = a - 273.15
            if v == "rh" and np.isfinite(np.nanmax(a)) and np.nanmax(a) <= 1.5:
                a = a * 100
            arrs.append(_fc_correct(a, bv, county, d, start))
        W[v] = dict(M=np.concatenate([W[v]["M"], np.stack(arrs).astype("float32")]), dates=W[v]["dates"].append(pd.DatetimeIndex(new)), tmpl=tm)
        log("forecast ", v, ": ", len(new), " days appended (", new[0].date(), " to ", new[-1].date(), ", scenario ", cfg.fc_scenario, ")")
    if "wspd" in W and (only is None or "wind" in only):
        fs, fd = _fc_files(county, "wind", "wind_speed"), _fc_files(county, "wind", "wind_dir")
        new = sorted(d for d in fs if d > W["wspd"]["dates"].max())
        if new:
            tm = W["wspd"]["tmpl"]; start = min(fs)
            sp = np.stack([_fc_correct(_fc_on_template(fs[d], tm), "wind", county, d, start) for d in new]).astype("float32")
            have_dir = all(d in fd for d in new)
            if "wu" in W and have_dir:
                sn = np.stack([_fc_on_template(fd[d], tm, lambda a: np.sin(np.deg2rad(a))) for d in new])     # direction is circular: resample sin and cos
                cs = np.stack([_fc_on_template(fd[d], tm, lambda a: np.cos(np.deg2rad(a))) for d in new])
                th = np.arctan2(sn, cs)                                                                          # direction the wind comes FROM, as in wx_native
                u, vv = (-sp * np.sin(th)).astype("float32"), (-sp * np.cos(th)).astype("float32")
            else:
                if "wu" in W:
                    warn("forecast: wind_dir files missing for ", county, ". Wind u, v, direction are NaN on the forecast days.")
                u = vv = np.full_like(sp, np.nan)
            dts = W["wspd"]["dates"].append(pd.DatetimeIndex(new))
            W["wspd"] = dict(M=np.concatenate([W["wspd"]["M"], sp]), dates=dts, tmpl=tm)
            if "wu" in W:
                W["wu"] = dict(M=np.concatenate([W["wu"]["M"], u]), dates=dts, tmpl=W["wu"]["tmpl"])
                W["wv"] = dict(M=np.concatenate([W["wv"]["M"], vv]), dates=dts, tmpl=W["wv"]["tmpl"])
            log("forecast wind: ", len(new), " days appended")

def load_weather(county):
    W = {}
    for v in ["tmax", "tmin", "precip", "rh"]:
        x = read_wx_var(county, v)
        if x is None:
            warn("no daily files for ", v); continue
        M = band_stacks(x, [1])[1]
        med = np.nanmedian(M)
        if v in ("tmax", "tmin") and np.isfinite(med) and med > 150:
            M = M - 273.15; log(v, ": converted Kelvin to deg C")
        if v == "rh" and np.isfinite(np.nanmax(M)) and np.nanmax(M) <= 1.5:
            M = M * 100; log("rh: converted fraction to %")
        W[v] = dict(M=M, dates=x["dates"], tmpl=x["tmpl"])
        log(v, ": ", len(x["files"]), " days ", x["dates"].min().date(), " to ", x["dates"].max().date())
    xw = read_wx_var(county, "wind")
    if xw is not None:
        nb = xw["tmpl"]["nb"]
        if nb >= 4:
            bm = wind_band_map(xw["tmpl"]["names"], nb)
            bs = band_stacks(xw, [bm["speed"], bm["u"], bm["v"]])
            W["wspd"] = dict(M=bs[bm["speed"]], dates=xw["dates"], tmpl=xw["tmpl"])
            W["wu"] = dict(M=bs[bm["u"]], dates=xw["dates"], tmpl=xw["tmpl"])
            W["wv"] = dict(M=bs[bm["v"]], dates=xw["dates"], tmpl=xw["tmpl"])
        else:
            warn("wind files have ", nb, " bands, expected 4. Only band 1 (speed) is used.")
            W["wspd"] = dict(M=band_stacks(xw, [1])[1], dates=xw["dates"], tmpl=xw["tmpl"])
    else:
        warn("no daily wind files")
    extend_with_forecast(county, W)
    fill_rh_from_donor(county, W)
    if "tmax" in W and "tmin" in W and not W["tmax"]["dates"].equals(W["tmin"]["dates"]):
        warn("tmax and tmin dates differ. Check the files.")
    return W

def wx_last_date(W):
    return min(x["dates"].max() for x in W.values())

def wx_first_date(W):
    return max(x["dates"].min() for x in W.values())

def fname(base, w):
    return base if w == cfg.main_window else "%s_%dd" % (base, w)

def win_idx(dates, d, w):
    d = pd.Timestamp(d)
    i = np.where((dates >= d - pd.Timedelta(days=w - 1)) & (dates <= d))[0]
    return i if len(i) >= math.ceil(cfg.min_window_coverage * w) else np.array([], dtype=int)

def _mean(M):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return np.nanmean(M, axis=0)

def _sum(M):
    s = np.nansum(M, axis=0)
    return np.where(np.isfinite(M).sum(axis=0) == 0, np.nan, s)

def _sd(M):
    n = np.isfinite(M).sum(axis=0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        v = np.nanstd(M, axis=0, ddof=1)
    return np.where(n < 2, np.nan, v)

def _dry_spell(P, thr):
    run = np.zeros(P.shape[1:]); best = np.zeros(P.shape[1:])
    for j in range(P.shape[0]):
        dry = np.isfinite(P[j]) & (P[j] < thr)
        run = (run + 1) * dry
        best = np.maximum(best, run)
    return best

def wx_native(W, d):
    """Features for one date at native resolution: {group: (dict name -> 2D array, tmpl)}"""
    d = pd.Timestamp(d); out = {}
    if "tmax" in W and "tmin" in W:
        L = {}; shp = W["tmax"]["M"].shape[1:]
        for w in cfg.windows:
            i = win_idx(W["tmax"]["dates"], d, w)
            if len(i):
                mx = W["tmax"]["M"][i]; mn = W["tmin"]["M"][i]; tm = (mx + mn) / 2
                L[fname("tmax_mean", w)] = _mean(mx); L[fname("tmin_mean", w)] = _mean(mn)
                L[fname("tmean_mean", w)] = _mean(tm); L[fname("tmean_sd", w)] = _sd(tm)
                L[fname("dtr_mean", w)] = _mean(mx - mn)
            else:
                for b in ["tmax_mean", "tmin_mean", "tmean_mean", "tmean_sd", "dtr_mean"]:
                    L[fname(b, w)] = np.full(shp, np.nan)
        out["temp"] = (L, W["tmax"]["tmpl"])
    if "precip" in W:
        L = {}; shp = W["precip"]["M"].shape[1:]
        for w in cfg.windows:
            i = win_idx(W["precip"]["dates"], d, w)
            if len(i):
                P = W["precip"]["M"][i]
                L[fname("precip_sum", w)] = _sum(P)
                L[fname("rain_days", w)] = (np.nan_to_num(P, nan=-1) >= cfg.wet_day_mm).sum(axis=0).astype(float)
                L[fname("dry_spell", w)] = _dry_spell(P, cfg.wet_day_mm)
            else:
                for b in ["precip_sum", "rain_days", "dry_spell"]:
                    L[fname(b, w)] = np.full(shp, np.nan)
        out["rain"] = (L, W["precip"]["tmpl"])
    if "rh" in W:
        L = {}; shp = W["rh"]["M"].shape[1:]
        for w in cfg.windows:
            i = win_idx(W["rh"]["dates"], d, w)
            L[fname("rh_mean", w)] = _mean(W["rh"]["M"][i]) if len(i) else np.full(shp, np.nan)
        out["rh"] = (L, W["rh"]["tmpl"])
    if "wspd" in W:
        L = {}; shp = W["wspd"]["M"].shape[1:]
        for w in cfg.windows:
            i = win_idx(W["wspd"]["dates"], d, w)
            if len(i):
                L[fname("wind.speed_mean", w)] = _mean(W["wspd"]["M"][i])
                if "wu" in W:
                    u = _mean(W["wu"]["M"][i]); v = _mean(W["wv"]["M"][i])
                    L[fname("wind.u_mean", w)] = u; L[fname("wind.v_mean", w)] = v
                    # direction is circular: rebuilt from mean u and v
                    L[fname("wind.direction_mean", w)] = np.mod(270 - np.degrees(np.arctan2(v, u)), 360)
            else:
                for b in ["wind.speed_mean", "wind.u_mean", "wind.v_mean", "wind.direction_mean"]:
                    if b == "wind.speed_mean" or "wu" in W:
                        L[fname(b, w)] = np.full(shp, np.nan)
        out["wind"] = (L, W["wspd"]["tmpl"])
    return out

def pad_native(a, tmpl, n=2):
    """Pad by n cells and fill the new rim by neighbour means so bilinear interpolation has no NaN edge."""
    p = np.pad(a.astype("float64"), n, constant_values=np.nan)
    k = np.ones((3, 3))
    for _ in range(2):
        valid = np.isfinite(p)
        s = ndimage.convolve(np.where(valid, p, 0.0), k, mode="constant")
        c = ndimage.convolve(valid.astype(float), k, mode="constant")
        m = ~valid & (c > 0)
        p[m] = s[m] / c[m]
    tr = tmpl["transform"] * Affine.translation(-n, -n)
    return p, tr

def elev_coarse_on_grid(county, grid, W, elev):
    """Elevation as the weather grid sees it, interpolated to the model grid (for the lapse-rate correction)."""
    f = os.path.join(out_dir(county, "aligned"), "elev_coarse.tif")
    if cfg.reuse_cache and os.path.exists(f) and grid.same_geometry(f):
        return Stack.from_file(f, ["elev_coarse"])
    tm = W["tmax"]["tmpl"]
    ec = np.full((tm["height"], tm["width"]), np.nan, dtype="float32")
    reproject(elev.astype("float32"), ec, src_transform=grid.transform, src_crs=grid.crs,
              dst_transform=tm["transform"], dst_crs=tm["crs"], resampling=Resampling.average,
              src_nodata=np.nan, dst_nodata=np.nan)
    p, tr = pad_native(ec, tm)
    out = np.full((grid.height, grid.width), np.nan, dtype="float32")
    reproject(p.astype("float32"), out, src_transform=tr, src_crs=tm["crs"], dst_transform=grid.transform,
              dst_crs=grid.crs, resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    write_tif(f, out, grid, names=["elev_coarse"])
    return Stack.from_file(f, ["elev_coarse"])

def temp_layers():
    return [fname(b, w) for w in cfg.windows for b in ("tmax_mean", "tmin_mean", "tmean_mean")]

def wx_to_grid(county, grid, W, d, elev, ecoarse):
    """Features for one date on the model grid, written to a cached GeoTIFF."""
    d = pd.Timestamp(d)
    f = os.path.join(out_dir(county, "wx"), "wx_%s%s.tif" % (d.strftime("%Y-%m-%d"), ("_fc_" + cfg.fc_scenario) if cfg.use_forecast else ""))
    if cfg.reuse_cache and os.path.exists(f) and grid.same_geometry(f):
        with rasterio.open(f) as r:
            return Stack.from_file(f, list(r.descriptions))
    nat = wx_native(W, d)
    bands, names = [], []
    off = None
    if cfg.lapse_correct and elev is not None and ecoarse is not None:
        off = cfg.lapse_rate * (elev - ecoarse)
    tl = set(temp_layers())
    for g, (L, tm) in nat.items():
        for nm, arr in L.items():
            p, tr = pad_native(arr, tm)
            out = np.full((grid.height, grid.width), np.nan, dtype="float32")
            reproject(p.astype("float32"), out, src_transform=tr, src_crs=tm["crs"], dst_transform=grid.transform,
                      dst_crs=grid.crs, resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
            if g == "temp" and off is not None and nm in tl:
                out = out + off
            bands.append(out); names.append(nm)
    write_tif(f, np.stack(bands), grid, names=names)
    return Stack.from_file(f, names)

_TRANSFORMERS = {}
def wx_extract(W, d, x, y, grid_crs, elev_pt=None, ec_pt=None):
    """Same features at points (grid-CRS coordinates). Same logic as the raster path."""
    nat = wx_native(W, d)
    res = {}
    tl = set(temp_layers())
    for g, (L, tm) in nat.items():
        key = (grid_crs.to_string(), tm["crs"].to_string())
        if key not in _TRANSFORMERS:
            _TRANSFORMERS[key] = Transformer.from_crs(grid_crs.to_wkt(), tm["crs"].to_wkt(), always_xy=True)
        px, py = _TRANSFORMERS[key].transform(np.asarray(x), np.asarray(y))
        for nm, arr in L.items():
            p, tr = pad_native(arr, tm)
            col = (px - tr.c) / tr.a - 0.5
            row = (tr.f - py) / (-tr.e) - 0.5
            v = ndimage.map_coordinates(p, [row, col], order=1, mode="nearest")
            if g == "temp" and cfg.lapse_correct and elev_pt is not None and ec_pt is not None and nm in tl:
                v = v + cfg.lapse_rate * (elev_pt - ec_pt)
            res[nm] = v
    return pd.DataFrame(res)

## 6. Mechanistic component
Every parameter lives in `THR` with its status and source, and is exported with each run.
- `sourced`: value taken from a published study named in `source`
- `design`: modelling choice, not a biological measurement
- `PLACEHOLDER`: NOT verified against literature. Off unless `cfg.mech_use_placeholders = True`

**What this cell does:** the rule-based part. FAW, Aphids and MLND use sourced thresholds in `THR`. Thrips uses the optimum you supplied (17 to 22 C, high mortality above 22 C) with a proxy lower limit and an assumed upper limit, both marked `design`. AAW and Stem borer take their temperature limits from their own occurrence records (`cfg.mech_empirical`), marked empirical. **You edit:** `THR` when you have a verified value.

In [ ]:
def _build_thr():
    r = []
    def a(pest, comp, par, value, unit, status, source):
        r.append(dict(pest=pest, comp=comp, par=par, value=value, unit=unit, status=status, source=source))
    S_FAW = "Insects 2022, 13(12):1084 (S. frugiperda egg to adult, 7 temperature-development models; https://www.mdpi.com/2075-4450/13/12/1084)"
    S_APH = "Appl. Entomol. Zool. 41(1):171, Taiwan (R. maidis on corn, 6-35 C; https://www.jstage.jst.go.jp/article/aez/41/1/41_1_171/_article)"
    a("FAW", "temp", "tlow",    13.51, "C", "sourced", S_FAW)
    a("FAW", "temp", "topt_lo", 28,    "C", "sourced", S_FAW + " optimum range 28-30 C")
    a("FAW", "temp", "topt_hi", 30,    "C", "sourced", S_FAW + " optimum range 28-30 C")
    a("FAW", "temp", "tup",     34.13, "C", "sourced", S_FAW)
    a("FAW", "gen",  "dd_gen",  490,   "degree-days above tlow", "sourced", S_FAW + " egg to adult")
    a("FAW", "w",    "temp",    0.6,   "weight", "design", "relative weight of thermal response in the geometric mean")
    a("FAW", "w",    "gen",     0.4,   "weight", "design", "relative weight of generation potential (30 d degree-days / 490)")
    a("FAW", "rh",   "a", 50, "%",  "PLACEHOLDER", "not verified: lower RH limit")
    a("FAW", "rh",   "b", 65, "%",  "PLACEHOLDER", "not verified: lower RH optimum")
    a("FAW", "rh",   "c", 85, "%",  "PLACEHOLDER", "not verified: upper RH optimum")
    a("FAW", "rh",   "d", 100, "%", "PLACEHOLDER", "not verified: upper RH limit")
    a("FAW", "rain", "r0", 60, "mm per 7 d",  "PLACEHOLDER", "not verified: weekly rain where egg/larval washout starts")
    a("FAW", "rain", "r1", 200, "mm per 7 d", "PLACEHOLDER", "not verified: weekly rain with maximum washout")
    a("FAW", "w",    "rh",   0.15, "weight", "PLACEHOLDER", "only used if cfg.mech_use_placeholders")
    a("FAW", "w",    "rain", 0.15, "weight", "PLACEHOLDER", "only used if cfg.mech_use_placeholders")
    a("APHIDS", "temp", "tlow",    4.44, "C", "sourced", S_APH)
    a("APHIDS", "temp", "topt_lo", 25,   "C", "sourced", S_APH + " highest intrinsic rate of increase (rm = 0.329) at 25 C")
    a("APHIDS", "temp", "topt_hi", 25,   "C", "sourced", S_APH + " peak at 25 C; no plateau is claimed")
    a("APHIDS", "temp", "tup",     35,   "C", "sourced", S_APH + " at 35 C a few nymphs survived but none reproduced")
    a("APHIDS", "gen",  "dd_gen",  119.05, "degree-days above tlow", "sourced", S_APH)
    a("APHIDS", "w",    "temp",    1.0,  "weight", "design", "thermal response carries the sourced part of the aphid model")
    a("APHIDS", "rh",   "a", 40, "%",  "PLACEHOLDER", "not verified")
    a("APHIDS", "rh",   "b", 55, "%",  "PLACEHOLDER", "not verified")
    a("APHIDS", "rh",   "c", 80, "%",  "PLACEHOLDER", "not verified")
    a("APHIDS", "rh",   "d", 100, "%", "PLACEHOLDER", "not verified")
    a("APHIDS", "rain", "r0", 50, "mm per 7 d",  "PLACEHOLDER", "not verified: heavy rain washes aphids off plants (general ecology, no threshold sourced)")
    a("APHIDS", "rain", "r1", 150, "mm per 7 d", "PLACEHOLDER", "not verified")
    a("APHIDS", "w",    "rh",   0.15, "weight", "PLACEHOLDER", "only used if cfg.mech_use_placeholders")
    a("APHIDS", "w",    "rain", 0.15, "weight", "PLACEHOLDER", "only used if cfg.mech_use_placeholders")
    # MLND = MCMV + SCMV. SCMV is carried by R. maidis (Ethiopia vector study, Springer article s42690-024-01185-w).
    # MCMV is carried by thrips (Frankliniella sp.) and leaf beetles (Oulema sp.) in the same study.
    a("MLND", "vector", "source", np.nan, "text", "sourced",
      "SCMV by R. maidis; MCMV by thrips and leaf beetles (https://link.springer.com/article/10.1007/s42690-024-01185-w)")
    a("MLND", "w", "aphid_vector", 1.0, "weight", "design", "aphid thermal response used as the sourced vector-pressure term")
    S_THR = "USER-SUPPLIED (J. Isaboke, project knowledge, 2026-10-08), no citation given. Thrips optimum 17-22 C, high mortality above 22 C"
    a("THRIPS", "temp", "tlow",    10.1, "C", "design", "PROXY: Thrips palmi lower development threshold 10.1 C (eje.cz 1999), a different species. Edit when a threshold for the project thrips is known")
    a("THRIPS", "temp", "topt_lo", 17,   "C", "user_supplied", S_THR)
    a("THRIPS", "temp", "topt_hi", 22,   "C", "user_supplied", S_THR)
    a("THRIPS", "temp", "tup",     26,   "C", "design", "NOT GIVEN: the response is set to fall to 0 by 26 C (4 C above 22 C) to represent high mortality. Edit")
    a("THRIPS", "w",    "temp",    1.0,  "weight", "design", "thermal response only")
    a("MLND", "thrips", "tlow",    np.nan, "C", "PLACEHOLDER", "no thrips (Frankliniella williamsi) thermal thresholds verified. Add from literature.")
    a("MLND", "thrips", "topt_lo", np.nan, "C", "PLACEHOLDER", "to be sourced")
    a("MLND", "thrips", "topt_hi", np.nan, "C", "PLACEHOLDER", "to be sourced")
    a("MLND", "thrips", "tup",     np.nan, "C", "PLACEHOLDER", "to be sourced")
    a("MLND", "w", "thrips", 0.5, "weight", "PLACEHOLDER", "only used if thrips thresholds are filled in and cfg.mech_use_placeholders")
    return pd.DataFrame(r)

THR = _build_thr()

def thr_get(pest, comp, par):
    v = THR.loc[(THR.pest == pest) & (THR.comp == comp) & (THR.par == par), "value"]
    if not len(v):
        raise KeyError("threshold missing: %s/%s/%s" % (pest, comp, par))
    return float(v.iloc[0])

def trap(x, a, b, c, d):
    """trapezoid response: 0 at a, rises to 1 at b, stays 1 to c, falls to 0 at d"""
    x = np.asarray(x, dtype="float64")
    up = clamp01((x - a) / (b - a)) if b > a else (x >= a).astype(float)
    dn = clamp01((d - x) / (d - c)) if d > c else (x <= d).astype(float)
    return np.where(np.isnan(x), np.nan, up * dn)

# Expected value of f(T) when daily T within the window is Normal(mean, sd). 5-point Gauss-Hermite.
# This lets a 30 m, terrain-corrected window MEAN stand in for 7-30 daily values, near thresholds too.
GH_Z = np.array([-2.8569700138728056, -1.3556261799742659, 0.0, 1.3556261799742659, 2.8569700138728056])
GH_W = np.array([0.011257411327720691, 0.2220759220056126, 0.5333333333333333, 0.2220759220056126, 0.011257411327720691])

def exp_resp(mu, sdv, f):
    mu = np.asarray(mu, dtype="float64")
    sdv = np.where(np.isnan(sdv), 1.5, sdv); sdv = np.maximum(sdv, 0.05)
    out = 0.0
    for z, w in zip(GH_Z, GH_W):
        out = out + w * f(mu + sdv * z)
    return out

def mech_temp_resp(pest, mu, sdv):
    p = lambda k: thr_get(pest, "temp", k)
    return exp_resp(mu, sdv, lambda t: trap(t, p("tlow"), p("topt_lo"), p("topt_hi"), p("tup")))

def mech_gen_frac(pest, mu30, sd30):
    tlow, tup = thr_get(pest, "temp", "tlow"), thr_get(pest, "temp", "tup")
    dd1 = exp_resp(mu30, sd30, lambda t: np.maximum(0, np.minimum(t, tup) - tlow))
    dd30 = 30 * dd1
    return dd30, clamp01(dd30 / thr_get(pest, "gen", "dd_gen"))

def geo_mean(L, w, eps=0.01):
    """weighted geometric mean of limiting factors. eps stops one zero from erasing the others."""
    w = np.asarray(w, dtype=float); w = w / w.sum()
    s = 0.0
    for wi, li in zip(w, L):
        s = s + wi * np.log(np.maximum(li, eps))
    return np.exp(s)

MECH_PESTS = {"FAW", "APHIDS", "MLND", "THRIPS"}   # thresholds in THR. STEMBORER and AAW take empirical thresholds from their own records (cfg.mech_empirical)

def has_mech_part(pest):
    return pest in MECH_PESTS or pest in (cfg.mech_empirical or [])

def empirical_temp_thr(t_pres):
    """thermal response limits from the temperature at the presence records: (tlow, topt_lo, topt_hi, tup) = 5th, 25th, 75th, 95th percentile.
    Returns None when there are too few values or the limits collapse."""
    t = np.asarray(t_pres, dtype=float); t = t[np.isfinite(t)]
    if len(t) < max(cfg.min_presence, 15):
        return None
    a, b, c, d = [float(v) for v in np.quantile(t, list(cfg.mech_empirical_q))]
    if not (a < b <= c < d):
        return None
    return (a, b, c, d)

def mech_index(pest, df, emp=None):
    """Mechanistic index in [0,1] for the rows of df. Needs tmean_mean, tmean_sd, tmean_mean_30d, tmean_sd_30d (+ host).
    emp = (tlow, topt_lo, topt_hi, tup) taken from the pest's own records (AAW, Stem borer)."""
    if pest not in MECH_PESTS and emp is None:
        return np.full(len(df), np.nan)
    host = df["host"].to_numpy(dtype=float) if "host" in df.columns else 1.0
    need = ["tmean_mean", "tmean_sd", "tmean_mean_30d", "tmean_sd_30d"]
    miss = [n for n in need if n not in df.columns]
    if miss:
        raise KeyError("mechanistic model needs: " + ", ".join(miss))
    g = lambda c: df[c].to_numpy(dtype="float64")
    use_ph = bool(cfg.mech_use_placeholders)
    L, w = [], []
    if emp is not None:
        L.append(exp_resp(g("tmean_mean"), g("tmean_sd"), lambda t: trap(t, *emp))); w.append(1.0)
    elif pest == "THRIPS":
        L.append(mech_temp_resp("THRIPS", g("tmean_mean"), g("tmean_sd"))); w.append(thr_get("THRIPS", "w", "temp"))
    elif pest == "FAW":
        L.append(mech_temp_resp("FAW", g("tmean_mean"), g("tmean_sd"))); w.append(thr_get("FAW", "w", "temp"))
        _, frac = mech_gen_frac("FAW", g("tmean_mean_30d"), g("tmean_sd_30d"))
        L.append(frac); w.append(thr_get("FAW", "w", "gen"))
    elif pest == "APHIDS":
        L.append(mech_temp_resp("APHIDS", g("tmean_mean"), g("tmean_sd"))); w.append(thr_get("APHIDS", "w", "temp"))
    elif pest == "MLND":
        # sourced part: aphid (SCMV) vector pressure from the R. maidis thermal response
        L.append(mech_temp_resp("APHIDS", g("tmean_mean"), g("tmean_sd"))); w.append(thr_get("MLND", "w", "aphid_vector"))
        tv = [thr_get("MLND", "thrips", k) for k in ("tlow", "topt_lo", "topt_hi", "tup")]
        if use_ph and all(np.isfinite(tv)):
            L.append(exp_resp(g("tmean_mean"), g("tmean_sd"), lambda t: trap(t, *tv))); w.append(thr_get("MLND", "w", "thrips"))
    if use_ph and pest in ("FAW", "APHIDS"):
        if "rh_mean" in df.columns:
            L.append(trap(g("rh_mean"), *[thr_get(pest, "rh", k) for k in "abcd"])); w.append(thr_get(pest, "w", "rh"))
        if "precip_sum" in df.columns:
            r0, r1 = thr_get(pest, "rain", "r0"), thr_get(pest, "rain", "r1")
            L.append(1 - clamp01((g("precip_sum") - r0) / (r1 - r0))); w.append(thr_get(pest, "w", "rain"))
    return host * geo_mean(L, w)

## 7. County context

**What this cell does:** loads everything one county needs (grid, layers, weather, mask) into a single context object. Output: the context used for training and prediction.

In [ ]:
def build_context(county):
    log("---- Building context for ", county, " ----")
    grid = make_grid(county)
    static = get_static(county, grid)
    host = make_host(county, grid, static)
    W = load_weather(county)
    elev = static.read_array("elevation") if "elevation" in static.names else None
    ecs = elev_coarse_on_grid(county, grid, W, elev) if (elev is not None and "tmax" in W) else None
    ec = ecs.read_array("elev_coarse") if ecs is not None else None
    avail = available_months(county)
    log("Monthly composites found: ", (avail[0] + " to " + avail[-1]) if avail else "none",
        " | weather range: ", ("%s to %s" % (wx_first_date(W).date(), wx_last_date(W).date())) if W else "none")
    return SimpleNamespace(county=county, grid=grid, static=static, host=host, W=W, elev=elev,
                           ecoarse=ecs, ec=ec, avail=avail)

## 8. Incidence data and training table

**What this cell does:** reads your field records and keeps **presence only**. Infestation level or severity is ignored. It cleans dates and coordinates, adds background points, and attaches predictor values. **Output:** the training table per pest and the log lines about dropped rows.

In [ ]:
def read_incidence(county, pest, ctx):
    f = find_file(fill_tokens(INC_SPEC[pest], county))
    if f is None:
        warn("incidence file not found for ", pest, " in ", county, ": ", fill_tokens(INC_SPEC[pest], county))
        return None
    try:
        d = pd.read_csv(f)
    except UnicodeDecodeError:
        d = pd.read_csv(f, encoding="latin-1")
    nm = list(d.columns); low = [str(c).lower().strip() for c in nm]
    ic = cfg.incidence_cols
    def col_of(want):
        for n, l in zip(nm, low):
            if l == want.lower():
                return n
        return None
    c_lon = col_of(ic.get("lon") or "longitude"); c_lat = col_of(ic.get("lat") or "latitude")
    if c_lon is None or c_lat is None:
        raise ValueError("Cannot find longitude/latitude columns in %s. Columns: %s" % (f, ", ".join(map(str, nm))))
    n0 = len(d)
    d = d.copy()
    # ---- filters: crop and pest name (kept only if enough rows remain) ----
    rc = col_of("report_type")
    if rc is not None and cfg.incidence_exclude_report:
        bad = d[rc].astype(str).str.contains(cfg.incidence_exclude_report, case=False, regex=True, na=False)
        if bad.any():
            log(pest, " ", county, ": ", int(bad.sum()), " rows dropped (report_type matches '", cfg.incidence_exclude_report, "')")
            d = d[~bad].reset_index(drop=True)
    crx = cfg.incidence_crop_regex.get(pest) if isinstance(cfg.incidence_crop_regex, dict) else cfg.incidence_crop_regex
    for colname, rx, what in [("crop_name", crx, "crop"),
                              ("pest_name", cfg.incidence_pest_regex.get(pest) if isinstance(cfg.incidence_pest_regex, dict) else None, "pest")]:
        c = col_of(colname)
        if c is None or not rx:
            continue
        top = d[c].astype(str).str.strip().value_counts().head(8).to_dict()
        log(pest, " ", county, ": top ", colname, " values ", top)
        k = d[c].astype(str).str.contains(rx, case=False, regex=True, na=False)
        if int(k.sum()) >= cfg.min_presence:
            d = d[k].reset_index(drop=True)
        else:
            warn(pest, " ", county, ": ", what, " filter '", rx, "' keeps only ", int(k.sum()), " rows. Filter NOT applied. Check ", colname, " values above.")
    # PRESENCE ONLY: infestation level / severity columns are never read or used. Every kept record is a presence (1).
    for extra in ("report_type",):
        c = col_of(extra)
        if c is not None:
            log(pest, " ", county, ": ", extra, " values ", d[c].astype(str).str.strip().value_counts().head(6).to_dict())
    d["_lon"] = pd.to_numeric(d[c_lon], errors="coerce"); d["_lat"] = pd.to_numeric(d[c_lat], errors="coerce")
    x0, x1, y0, y1 = cfg.kenya_box
    d = d[d._lon.notna() & d._lat.notna() & d._lon.between(x0, x1) & d._lat.between(y0, y1)].reset_index(drop=True)
    # ---- occurrence date: first usable of the priority list (planting_date is never used) ----
    dd = pd.Series(pd.NaT, index=d.index, dtype="datetime64[ns]"); src = pd.Series("none", index=d.index)
    for cn in cfg.incidence_date_cols:
        c = col_of(cn)
        if c is None:
            continue
        pdte = parse_dates(d[c])
        fill = dd.isna() & pdte.notna().values
        dd[fill] = pdte.values[fill.values] if hasattr(pdte, "values") else pdte[fill]
        src[fill] = cn
    log(pest, " ", county, ": occurrence date source ", src.value_counts().to_dict())
    d["_date"] = dd.values
    tr = Transformer.from_crs("EPSG:4326", ctx.grid.crs.to_wkt(), always_xy=True)
    x, y = tr.transform(d._lon.to_numpy(), d._lat.to_numpy())
    row, col = ctx.grid.xy_to_rc(x, y)
    inb = (row >= 0) & (row < ctx.grid.height) & (col >= 0) & (col < ctx.grid.width)
    ins = np.zeros(len(d), bool)
    ins[inb] = ctx.grid.inside[row[inb], col[inb]]
    d = d[ins].reset_index(drop=True); x, y, row, col = x[ins], y[ins], row[ins], col[ins]
    n1 = len(d)
    W = ctx.W
    dd = pd.to_datetime(d["_date"])
    if W and dd.notna().any():
        lo = wx_first_date(W) + pd.Timedelta(days=20); hi = wx_last_date(W)
        okd = dd.isna() | ((dd >= lo) & (dd <= hi))
        if (~okd).any():
            warn(pest, ": ", int((~okd).sum()), " records dropped, dated outside ", lo.date(), " to ", hi.date(), " (weather record)")
        k = okd.to_numpy()
        d, x, y, row, col, dd = d[k].reset_index(drop=True), x[k], y[k], row[k], col[k], dd[k].reset_index(drop=True)
    if dd.isna().all():
        fb = wx_last_date(W) if W else pd.NaT
        warn(pest, ": no usable dates. All records use the weather window ending ", fb, ". The model will then be spatial only.")
        dd = pd.Series([fb] * len(d))
    elif dd.isna().any():
        md = dd.dropna().sort_values().iloc[len(dd.dropna()) // 2]
        warn(pest, ": ", int(dd.isna().sum()), " records without date set to the median date ", md.date())
        dd = dd.fillna(md)
    pts = pd.DataFrame(dict(x=x, y=y, date=dd.values, cell=row.astype("int64") * ctx.grid.width + col))
    pts["ym"] = pd.to_datetime(pts.date).dt.strftime("%Y_%m")
    key = (pts.cell.astype(str) + "_" + pts.ym) if cfg.thin_per_cell_month else (pts.cell.astype(str) + "_" + pts.date.astype(str))
    pts = pts[~key.duplicated()].reset_index(drop=True)
    log(pest, " ", county, ": ", n0, " rows -> ", n1, " inside county -> ", len(pts), " after cleaning and thinning")
    try:
        ad = out_dir(county, "models")
        pts.to_csv(os.path.join(ad, "incidence_clean_%s.csv" % pest), index=False)
        pts.groupby("ym").size().rename("records").to_csv(os.path.join(ad, "records_by_month_%s.csv" % pest))
    except Exception as e:
        warn("could not write incidence audit files: ", e)
    return pts

def sample_background(n, ctx, date_pool):
    """BACKGROUND (not absences): random cells inside the county AND the host area, each given a date drawn from the presence dates.
    They describe the environment that is available, so the models can learn how the conditions at the records differ from it."""
    h = ctx.host.read_array("host")
    ok = np.flatnonzero((h > 0).ravel() & ctx.grid.inside.ravel())
    if len(ok) < n * 0.5:
        warn("only ", len(ok), " background cells found in the host area")
    k = min(n, len(ok))
    cells = rng.choice(ok, size=k, replace=False)
    r, c = np.divmod(cells, ctx.grid.width)
    x, y = ctx.grid.rc_to_xy(r, c)
    dp = pd.to_datetime(pd.Series(date_pool)).to_numpy()
    return pd.DataFrame(dict(x=x, y=y, date=rng.choice(dp, size=k, replace=True)))

def build_training_matrix(ctx, pts):
    """static + monthly composite of the month + rolling weather of the date, at point locations"""
    pts = pts.reset_index(drop=True)
    rows, cols = ctx.grid.xy_to_rc(pts.x.to_numpy(), pts.y.to_numpy())
    base = pd.concat([ctx.static.read_points(rows, cols), ctx.host.read_points(rows, cols)], axis=1)
    elev_pt = base["elevation"].to_numpy() if "elevation" in base.columns else None
    ec_pt = ctx.ecoarse.read_points(rows, cols)["elev_coarse"].to_numpy() if ctx.ecoarse is not None else None
    dts = pd.to_datetime(pts.date)
    mon_ym = np.array([pick_ym(d, ctx.avail) for d in dts])
    mon_parts = []
    for g in np.unique(mon_ym):
        idx = np.flatnonzero(mon_ym == g)
        m = get_monthly(ctx.county, ctx.grid, g)
        if m is None:
            continue
        e = derive_monthly_columns(m.read_points(rows[idx], cols[idx]), m.meta)
        e.index = idx; mon_parts.append(e)
    mon = pd.concat(mon_parts).sort_index() if mon_parts else None
    lan = None
    if needs_landsat():
        sc = [nearest_scene(ctx.county, d) for d in dts]
        files = np.array([s[0] if s else "" for s in sc], dtype=object)
        lp = []
        for fpath in [x for x in np.unique(files) if x]:
            idx = np.flatnonzero(files == fpath)
            st = get_landsat_stack(ctx.county, ctx.grid, fpath)
            if st is None:
                continue
            e = st.read_points(rows[idx], cols[idx]); e.index = idx; lp.append(e)
        if lp:
            lan = pd.concat(lp).sort_index().reindex(range(len(pts)))
            log("Landsat monthly stack matched for ", int((files != "").sum()), " of ", len(pts), " records")
        else:
            log("Landsat: no monthly stack for any record in ", ctx.county)
    wx_parts = []
    dstr = dts.dt.strftime("%Y-%m-%d").to_numpy()
    for d in np.unique(dstr):
        idx = np.flatnonzero(dstr == d)
        e = wx_extract(ctx.W, d, pts.x.to_numpy()[idx], pts.y.to_numpy()[idx], ctx.grid.crs,
                       elev_pt[idx] if elev_pt is not None else None, ec_pt[idx] if ec_pt is not None else None)
        e.index = idx; wx_parts.append(e)
    wx = pd.concat(wx_parts).sort_index() if wx_parts else None
    parts = [pts[["x", "y", "date"]], base] + [p.reset_index(drop=True) for p in (mon, lan, wx) if p is not None]
    out = pd.concat([parts[0].reset_index(drop=True), parts[1].reset_index(drop=True)] + parts[2:], axis=1)
    return out.loc[:, ~out.columns.duplicated()]

# ---- static (susceptibility) mode: every date-dependent layer is replaced by its mean over reference dates of the year ----
STATIC_EXTRA = ["tmean_mean_season_sd", "precip_sum_30d_season_sd", "rh_mean_30d_season_sd"]    # seasonality, from the same layers
_SEASON_BASE = {"tmean_mean_season_sd": "tmean_mean", "precip_sum_30d_season_sd": "precip_sum_30d", "rh_mean_30d_season_sd": "rh_mean_30d"}

def static_ref_dates(ctx):
    if cfg.static_ref_dates:
        return [pd.Timestamp(d) for d in cfg.static_ref_dates]
    if not ctx.W:
        raise RuntimeError("static mode needs the daily weather record")
    d0 = pd.Timestamp(wx_first_date(ctx.W)) + pd.Timedelta(days=31); d1 = pd.Timestamp(wx_last_date(ctx.W))
    ds = [m + pd.Timedelta(days=14) for m in pd.date_range(d0.normalize().replace(day=1), d1, freq="MS")]
    ds = [d for d in ds if d0 <= d <= d1][-12:]
    if len(ds) < 3:
        warn("static mode: only ", len(ds), " reference dates fit in the weather record")
    return ds

def build_static_matrix(ctx, pts):
    """same columns as build_training_matrix, but every date-dependent column is the MEAN over the reference dates (annual summary),
    plus the seasonal standard deviation of a few weather columns. Terrain, soil and crop type are unchanged."""
    pts = pts.reset_index(drop=True); refs = static_ref_dates(ctx)
    log("static mode: annual means over ", len(refs), " reference dates ", refs[0].date(), " to ", refs[-1].date())
    mats = []
    for d in refs:
        q = pts[["x", "y"]].copy(); q["date"] = d
        mats.append(build_training_matrix(ctx, q).reset_index(drop=True))
    cols = sorted(set.intersection(*[set(m.columns) for m in mats]) - {"x", "y", "date"}, key=list(mats[0].columns).index)
    num = [c for c in cols if pd.api.types.is_numeric_dtype(mats[0][c])]
    arr = np.stack([m[num].to_numpy(dtype=float) for m in mats])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mean = np.nanmean(arr, axis=0); sd = np.nanstd(arr, axis=0)
    out = pd.concat([pts[["x", "y"]], pd.DataFrame(mean, columns=num)], axis=1)
    out["date"] = pts["date"].to_numpy() if "date" in pts.columns else pd.NaT
    for nm, base in _SEASON_BASE.items():
        if base in num:
            out[nm] = sd[:, num.index(base)]
    return out

def pred_set(pest):
    return PRED_SETS[pest] + (STATIC_EXTRA if cfg.static_mode else [])

# Predictors requested per pest (original lists; names use the corrected weather names)
WX_MAIN = ["tmax_mean", "tmin_mean", "precip_sum", "rh_mean", "wind.speed_mean", "wind.u_mean", "wind.v_mean", "wind.direction_mean"]
WX_EXTRA = [b + "_14d" for b in ["tmax_mean", "tmin_mean", "precip_sum", "rain_days", "dry_spell", "rh_mean"]] + \
           [b + "_30d" for b in ["tmax_mean", "tmin_mean", "precip_sum", "rain_days", "dry_spell", "rh_mean"]]
TERRAIN = ["elevation", "slope", "northness", "eastness", "roughness"]
PRED_SETS = {
    "FAW": TERRAIN + WX_MAIN + WX_EXTRA + ["crop.type", "sand", "soc", "savi", "rdvi", "ndmi", "ndvi", "mcari", "ndre",
                                           "psri", "pri"],
    "APHIDS": TERRAIN + WX_MAIN + WX_EXTRA + ["crop.type", "clay", "l.lst", "soc", "s.coastal", "s.red", "l.blue", "mtvi2",
                                              "l.swir1", "psri", "s.red.edge", "savi", "ndre", "mcari"],
    # African armyworm: from the importance ranking sent for this pest (landsat_Coastal, landsat_NIR, PRI, humidity, rainfall,
    # wind speed, CVI, sentinel NIR, MCARI, PSRI). l.nir and s.nir do not exist in the data on drive H, they are skipped with a warning.
    "AAW": TERRAIN + WX_MAIN + WX_EXTRA + ["crop.type", "sand", "soc", "l.coastal", "l.nir", "pri", "cvi", "s.nir", "mcari", "psri"],
    # PROVISIONAL: one importance list was sent without a pest label. Stem borer and thrips use this general set until it is confirmed.
    "STEMBORER": TERRAIN + WX_MAIN + WX_EXTRA + ["crop.type", "sand", "clay", "soc", "l.coastal", "l.lst", "pri", "cvi", "ndvi", "savi", "ndre", "psri"],
    "THRIPS": TERRAIN + WX_MAIN + WX_EXTRA + ["crop.type", "sand", "clay", "soc", "l.coastal", "l.lst", "pri", "cvi", "ndvi", "savi", "ndre", "psri"],
    # MLND: its own records plus the weighted pressure of the other pests (vector_pressure), see add_coupling
    "MLND": TERRAIN + WX_MAIN + WX_EXTRA + ["crop.type", "sand", "clay", "ndvi", "ndre", "soc", "l.blue", "savi", "pri",
                                            "s.coastal", "s.red.edge", "s.vapour"],
}
DROP_PRED = set(cfg.drop_predictors or [])
PRED_SETS = {k_: [v_ for v_ in lst_ if v_ not in DROP_PRED] for k_, lst_ in PRED_SETS.items()}      # see cfg.drop_predictors
if DROP_PRED:
    log("Predictors dropped for every pest (cfg.drop_predictors): ", ", ".join(sorted(DROP_PRED)))
RISK_COL = {"FAW": "faw_risk", "APHIDS": "aphids_risk", "THRIPS": "thrips_risk", "STEMBORER": "stemborer_risk", "AAW": "aaw_risk"}
COUPLING = {"FAW": [], "APHIDS": ["faw_risk"], "THRIPS": [], "STEMBORER": [], "AAW": [], "MLND": ["vector_pressure"]}

In [ ]:
print("s.coastal in predictor lists:", [k for k, v in PRED_SETS.items() if "s.coastal" in v])
print("s.vapour in predictor lists:", [k for k, v in PRED_SETS.items() if "s.vapour" in v])

In [ ]:
for c in ["Kakamega", "Kilifi", "Nyeri"]:
    f = find_cwv(c, "2025_10")
    with rasterio.open(f) as s:
        print(c, s.count, list(s.descriptions))

## 9. Statistical models

**What this cell does:** defines the six ML models (MaxEnt, GAM, Random Forest, GBM, XGBoost, SVM), their fitting and prediction. Nothing to edit here, tune through `cfg`.

In [ ]:
def balance_weights(y):
    """presences and background count equally"""
    y = np.asarray(y); n = len(y); n1 = (y == 1).sum(); n0 = (y == 0).sum()
    return np.where(y == 1, n / (2.0 * n1), n / (2.0 * n0))

def design_matrix(df, vars_, crop_levels=None):
    cont = [v for v in vars_ if v != "crop.type"]
    X = pd.DataFrame({v: (pd.to_numeric(df[v], errors="coerce").astype("float64") if v in df.columns
                          else pd.Series(np.nan, index=df.index)) for v in cont}, index=df.index)
    if "crop.type" in vars_ and crop_levels is not None and len(crop_levels):
        ct = df["crop.type"] if "crop.type" in df.columns else pd.Series(np.nan, index=df.index)
        for l in crop_levels:
            X["crop.type_%g" % l] = (ct.notna() & (ct == l)).astype("float64")
    return X

def prune_collinear(X, cut, keep=()):
    """drop one of every pair of highly correlated predictors"""
    if X.shape[1] < 2:
        return list(X.columns)
    cm = X.corr().abs().fillna(0.0)
    cols = list(cm.columns); A = cm.to_numpy().copy(); np.fill_diagonal(A, 0.0)
    drop = []
    while True:
        m = A.max()
        if m <= cut:
            break
        i, j = np.argwhere(A == m)[0]
        a, b = cols[i], cols[j]
        cand = [c for c in (a, b) if c not in keep]
        if not cand:
            A[i, j] = A[j, i] = 0.0; continue
        dr = (a if A[i].mean() >= A[j].mean() else b) if len(cand) == 2 else cand[0]
        k = cols.index(dr); drop.append(dr); A[k, :] = 0.0; A[:, k] = 0.0
    return [c for c in cols if c not in drop]

def auc_of(y, p):
    y = np.asarray(y); p = np.asarray(p, dtype=float); ok = np.isfinite(p)
    if len(np.unique(y[ok])) < 2:
        return np.nan
    return float(roc_auc_score(y[ok], p[ok]))

def presence_metrics(ytr, ptr, yte, pte):
    """PRESENCE-BACKGROUND measures only. Background cells are NOT absences, so nothing here counts a background cell as a wrong call.
    Threshold = a low quantile of the scores at the TRAINING presences (10th percentile, and the lowest one).
    sensitivity_*          share of the held-out presences that score at or above the threshold (about 0.90 for p10 if the model transfers)
    pct_landscape_above_*  share of the available-environment (background) cells at or above it: the smaller, the more selective the model
    sens_to_area_ratio_*   sensitivity / landscape share (above 1 = better than a random map of the same size)"""
    ytr = np.asarray(ytr).astype(int); yte = np.asarray(yte).astype(int)
    ptr = np.asarray(ptr, dtype=float); pte = np.asarray(pte, dtype=float)
    out = {}
    trp = ptr[(ytr == 1) & np.isfinite(ptr)]
    tep = pte[(yte == 1) & np.isfinite(pte)]; ter = pte[(yte == 0) & np.isfinite(pte)]
    for tag, thr in (("p10", np.quantile(trp, 0.10) if len(trp) else np.nan), ("min", trp.min() if len(trp) else np.nan)):
        if np.isfinite(thr) and len(tep) and len(ter):
            sens = float((tep >= thr).mean()); area = float((ter >= thr).mean())
            out.update({"threshold_" + tag: float(thr), "sensitivity_" + tag: sens, "pct_landscape_above_" + tag: 100.0 * area,
                        "sens_to_area_ratio_" + tag: sens / area if area > 0 else np.nan})
        else:
            out.update({"threshold_" + tag: np.nan, "sensitivity_" + tag: np.nan, "pct_landscape_above_" + tag: np.nan,
                        "sens_to_area_ratio_" + tag: np.nan})
    return out

def boyce_index(y, p, nbins=10):
    """continuous Boyce index (presence-only check): ratio of the share of presences to the share of background in each score bin,
    then Spearman correlation with the bin order. Needs no absences. +1 = presences sit in the high scores, 0 = no better than random, negative = wrong way."""
    from scipy.stats import spearmanr
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); ok = np.isfinite(p)
    y, p = y[ok], p[ok]
    if (y == 1).sum() < 5 or (y == 0).sum() < 20:
        return np.nan
    edges = np.unique(np.quantile(p[y == 0], np.linspace(0, 1, nbins + 1)))
    if len(edges) < 4:
        return np.nan
    edges[0] = -np.inf; edges[-1] = np.inf
    pi = np.histogram(p[y == 1], edges)[0] / max((y == 1).sum(), 1)
    ei = np.histogram(p[y == 0], edges)[0] / max((y == 0).sum(), 1)
    f = np.where(ei > 0, pi / np.where(ei > 0, ei, 1), np.nan)
    k = np.isfinite(f)
    if k.sum() < 3 or np.nanstd(f[k]) == 0:
        return np.nan
    return float(spearmanr(np.arange(len(f))[k], f[k])[0])

def split_train_test(xy, y, test_frac, block_m, method="block", seed=123):
    """70/30 split. 'block': whole spatial blocks go to the test set until about test_frac of the presences are in it
    (no near-neighbour leakage). Falls back to a stratified random split if the blocks are too few. Returns (train_idx, test_idx)."""
    y = np.asarray(y).astype(int); r = np.random.default_rng(seed); n = len(y)
    def stratified():
        te = np.zeros(n, bool)
        for cl in (0, 1):
            ix = np.flatnonzero(y == cl); r.shuffle(ix)
            te[ix[: int(round(test_frac * len(ix)))]] = True
        return te
    if method == "random":
        te = stratified()
    else:
        blk = pd.Series(list(zip(np.floor(xy[:, 0] / block_m).astype(int), np.floor(xy[:, 1] / block_m).astype(int))))
        codes, ub = pd.factorize(blk); nb = len(ub)
        npres = np.bincount(codes, weights=(y == 1).astype(float), minlength=nb)
        want = test_frac * (y == 1).sum()
        order = r.permutation(nb); tb = np.zeros(nb, bool); got = 0.0
        for b in order:
            if got >= want:
                break
            if npres[b] > 0 or got < want * 0.2:
                tb[b] = True; got += npres[b]
        te = tb[codes]
        if (not te.any()) or te.sum() > 0.5 * n or len(np.unique(y[te])) < 2 or len(np.unique(y[~te])) < 2 or (y[te] == 1).sum() < 3:
            warn("spatial blocks cannot give a clean 70/30 split here. Using a stratified random split.")
            te = stratified()
    return np.flatnonzero(~te), np.flatnonzero(te)

# ---- XGBoost on the GPU: tested once, falls back to CPU -----------------------------------
_XGB = {"gpu": None}
def xgb_gpu_ok():
    if _XGB["gpu"] is not None:
        return _XGB["gpu"]
    ok = False
    if cfg.use_gpu:
        try:
            import xgboost as xgb
            with warnings.catch_warnings(record=True) as ws:
                warnings.simplefilter("always")
                m = xgb.XGBClassifier(device="cuda", tree_method="hist", n_estimators=1)
                m.fit(np.random.default_rng(1).normal(size=(40, 2)), np.tile([0, 1], 20))
            msgs = " ".join(str(w.message) for w in ws)
            ok = not ("No visible GPU" in msgs or "Device is changed" in msgs or "not compiled with CUDA" in msgs)
        except Exception:
            ok = False
        log("XGBoost GPU (CUDA): ", "available, will be used" if ok else "not available, using CPU")
    _XGB["gpu"] = ok
    return ok

def fit_model(type_, X, y, w):
    """Returns dict(type, cols, fit, ...) or None on failure."""
    cols = list(X.columns)
    Xa = X.to_numpy(dtype="float64"); y = np.asarray(y).astype(int)
    try:
        if type_ == "maxent":
            import elapid
            m = elapid.MaxentModel(feature_types=["linear", "quadratic"], transform="cloglog", n_lambdas=20, max_iter=300,
                                   convergence_tolerance=1e-4, n_cpus=int(cfg.cores))   # fewer lambdas and looser tolerance: same ranking, far faster
            m.fit(Xa, y, sample_weight=w)
            return dict(type=type_, cols=cols, fit=m)
        if type_ == "gam":
            from pygam import LogisticGAM, s, l
            ua = np.array([abs((auc_of(y, Xa[:, j]) if np.isfinite(auc_of(y, Xa[:, j])) else 0.5) - 0.5) for j in range(len(cols))])
            nun = np.array([len(np.unique(Xa[:, j])) for j in range(len(cols))])
            smooth = [j for j in np.argsort(-ua) if nun[j] >= 8][: int(cfg.gam_max_terms)]
            ref = [j for j, c in enumerate(cols) if c.startswith("crop.type_")][-1:]       # reference level of the dummies
            linear = [j for j in range(len(cols)) if nun[j] < 8 and j not in ref]
            terms = None
            for j in smooth + linear:
                t = s(int(j), n_splines=5) if j in smooth else l(int(j))
                terms = t if terms is None else terms + t
            if terms is None:
                raise ValueError("no usable terms")
            mu, sd = Xa.mean(0), Xa.std(0); sd[sd == 0] = 1.0
            Z = (Xa - mu) / sd                       # standardised so the linear terms are well conditioned
            g, last = None, None
            for lam in (6.0, 60.0, 600.0):                 # more smoothing if the fit does not converge
                try:
                    g = LogisticGAM(terms, lam=lam, max_iter=60).fit(Z, y, weights=w); break
                except Exception as e:
                    last = e
            if g is None:
                # last resort: three splines per smooth term, strong smoothing, fewer iterations
                t2 = None
                for j in smooth + linear:
                    t = s(int(j), n_splines=4) if j in smooth else l(int(j))
                    t2 = t if t2 is None else t2 + t
                for lam in (2000.0, 20000.0):
                    try:
                        g = LogisticGAM(t2, lam=lam, max_iter=50).fit(Z, y, weights=w); break
                    except Exception as e:
                        last = e
            if g is None:
                raise last
            return dict(type=type_, cols=cols, fit=g, lo=Z.min(0), hi=Z.max(0), mu=mu, sd=sd)
        if type_ == "rf":
            m = RandomForestClassifier(n_estimators=500, max_features=max(1, int(math.floor(math.sqrt(len(cols))))),
                                       min_samples_leaf=5, n_jobs=cfg.cores, random_state=cfg.seed)
            m.fit(Xa, y, sample_weight=w)
            return dict(type=type_, cols=cols, fit=m)
        if type_ == "gbm":
            m = HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.05, max_depth=3, min_samples_leaf=10,
                                               early_stopping=True, validation_fraction=0.2, n_iter_no_change=30,
                                               random_state=cfg.seed)
            m.fit(Xa, y, sample_weight=w)
            return dict(type=type_, cols=cols, fit=m)
        if type_ == "xgboost":
            import xgboost as xgb
            kw = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                      n_jobs=cfg.cores, random_state=cfg.seed, eval_metric="logloss")
            if xgb_gpu_ok():
                kw.update(device="cuda", tree_method="hist")
            m = xgb.XGBClassifier(**kw)
            m.fit(Xa, y, sample_weight=w)
            if kw.get("device") == "cuda":
                try:
                    m.get_booster().set_param({"device": "cpu"})   # predict on CPU: our inputs are numpy arrays, avoids the device-mismatch warning
                except Exception:
                    pass
            return dict(type=type_, cols=cols, fit=m)
        if type_ == "svm":
            n = len(y)
            if n > cfg.svm_max_n:
                i1 = np.flatnonzero(y == 1); i0 = np.flatnonzero(y == 0)
                k1 = min(len(i1), cfg.svm_max_n // 2); k0 = min(len(i0), cfg.svm_max_n - k1)
                ii = np.concatenate([rng.choice(i1, k1, replace=False), rng.choice(i0, k0, replace=False)])
            else:
                ii = np.arange(n)
            ys = y[ii]
            cw = {0: 1.0, 1: float((ys == 0).sum() / max((ys == 1).sum(), 1))}
            m = make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True, class_weight=cw, random_state=cfg.seed))
            m.fit(Xa[ii], ys)
            return dict(type=type_, cols=cols, fit=m)
        raise ValueError("unknown model type " + str(type_))
    except Exception as e:
        warn("model '", type_, "' failed: ", str(e)[:200])
        return None

def predict_model_raw(obj, X):
    Xa = X[obj["cols"]].to_numpy(dtype="float64")
    t = obj["type"]; f = obj["fit"]
    if t == "maxent":
        return np.asarray(f.predict(Xa), dtype=float)
    if t == "gam":
        Z = (Xa - obj["mu"]) / obj["sd"]
        return np.asarray(f.predict_mu(np.clip(Z, obj["lo"], obj["hi"])), dtype=float)
    return f.predict_proba(Xa)[:, 1].astype(float)

def predict_model(obj, X):
    """models without their own threading are split across workers"""
    if obj["type"] in ("maxent", "gam", "svm") and cfg.cores > 1 and len(X) >= 20000:
        chunks = np.array_split(np.arange(len(X)), cfg.cores)
        res = Parallel(n_jobs=cfg.cores)(delayed(predict_model_raw)(obj, X.iloc[i]) for i in chunks)
        return np.concatenate(res)
    return predict_model_raw(obj, X)

## 10. Spatial cross-validation (block folds, balanced on presences)

**What this cell does:** makes spatial block folds and the 70/30 split so that test points are not neighbours of training points. Output: fold and split labels.

In [ ]:
def spatial_folds(xy, y, k, block_m):
    y = np.asarray(y)
    blk = pd.Series(list(zip(np.floor(xy[:, 0] / block_m).astype(int), np.floor(xy[:, 1] / block_m).astype(int))))
    codes, ub = pd.factorize(blk)
    nb = len(ub)
    np_ = np.bincount(codes, weights=(y == 1).astype(float), minlength=nb)
    nt = np.bincount(codes, minlength=nb).astype(float)
    order = np.lexsort((-nt, -np_))
    fold_p = np.zeros(k); fold_t = np.zeros(k); assign = np.zeros(nb, int)
    for b in order:
        f = int(np.lexsort((fold_t, fold_p))[0])
        assign[b] = f; fold_p[f] += np_[b]; fold_t[f] += nt[b]
    fold = assign[codes]
    ok = all(len(np.unique(y[fold == f])) == 2 and len(np.unique(y[fold != f])) == 2 for f in range(k))
    if (not ok) or nb < k:
        warn("spatial blocks too few or unbalanced. Falling back to random folds.")
        fold = rng.permutation(np.resize(np.arange(k), len(y)))
    return fold

def cv_models(X, y, w, fold, types):
    oof = pd.DataFrame(np.nan, index=range(len(X)), columns=types)
    for f in sorted(np.unique(fold)):
        tr = fold != f; te = fold == f
        for m in types:
            obj = fit_model(m, X[tr], y[tr], w[tr])
            if obj is not None:
                try:
                    oof.loc[te, m] = predict_model_raw(obj, X[te])
                except Exception:
                    pass
    return oof

def rescale_q(p, q, edge=0.02):
    """Map a raw model output onto 0-1: the 1st percentile of the reference scores goes to `edge` (0.02), the 99th to 1 - edge,
    linear in between. Values beyond the anchors are NOT clamped: they continue with a smooth tail that keeps the order, so two
    different cells never get the same score (no ties at 0 or 1) and the AUC is not lowered by clipping."""
    p = np.asarray(p, dtype=float)
    span = max(float(q[1]) - float(q[0]), 1e-9)
    z = (p - float(q[0])) / span
    k = 1.0 - 2.0 * edge                    # slope inside the anchors
    s = edge / k                            # tail length chosen so the slope is continuous at the anchors
    with np.errstate(invalid="ignore", divide="ignore"):
        out = edge + k * z
        out = np.where(z < 0, edge / (1.0 - z / s), out)
        out = np.where(z > 1, 1.0 - edge / (1.0 + (z - 1.0) / s), out)
    return out

def make_ref(v, edge=0.01):
    """reference distribution of scores (usually the available-environment cells): 101 quantiles, stored so that new scores can be
    turned into percentiles later"""
    v = np.asarray(v, dtype=float); v = v[np.isfinite(v)]
    if len(v) < 20:
        return None
    probs = np.linspace(0.0, 1.0, 101)
    xs = np.quantile(v, probs)
    span = max(float(xs[-1] - xs[0]), 1e-9)
    xs = np.maximum.accumulate(xs) + np.arange(len(xs)) * span * 1e-9      # strictly increasing, so no two scores share a percentile
    return dict(x=xs, p=edge + (1.0 - 2.0 * edge) * probs, edge=edge)

def to_pctl(v, ref):
    """percentile (0-1) of a score within the reference distribution. Order-preserving and NOT clamped: beyond the reference range the
    score keeps rising or falling slowly, so cells with very different values never tie at 0 or 1."""
    v = np.asarray(v, dtype=float)
    if ref is None:
        return np.full(v.shape, np.nan)
    xs, ps, e = ref["x"], ref["p"], ref["edge"]
    with np.errstate(invalid="ignore", divide="ignore"):
        out = np.interp(v, xs, ps)
        full = (ps[-1] - ps[0]) / max(xs[-1] - xs[0], 1e-9)
        sl = (ps[5] - ps[0]) / (xs[5] - xs[0]) if xs[5] - xs[0] > 1e-6 * (xs[-1] - xs[0]) else full
        sh = (ps[-1] - ps[-6]) / (xs[-1] - xs[-6]) if xs[-1] - xs[-6] > 1e-6 * (xs[-1] - xs[0]) else full
        d_lo = np.maximum(xs[0] - v, 0.0); d_hi = np.maximum(v - xs[-1], 0.0)
        out = np.where(v < xs[0], e / (1.0 + d_lo * sl / e), out)
        out = np.where(v > xs[-1], 1.0 - e / (1.0 + d_hi * sh / e), out)
    return out

## 11. Prediction for a table of cells (used for raster windows and for points)

**What this cell does:** turns predictor values into the three scores for any set of cells: **statistical** (weighted ML ensemble), **mechanistic**, and **hybrid** = (1 - w) x statistical percentile + w x mechanistic percentile. Both parts are first turned into percentiles of the available-environment scores, so the weight w is the real influence, and nothing is clamped, so cells do not tie. If the map date falls in a calendar month that the training records do not cover, a pest with a valid mechanistic part is scored by it alone, and the others keep the statistical map; every cell always gets a score and a class, and such maps are marked as indicative. Used for both rasters and points.

In [ ]:
def prep_X(po, df):
    X = design_matrix(df, po["vars"], po["crop_levels"])
    for c in po["X_cols"]:
        if c not in X.columns:
            X[c] = np.nan
    X = X[po["X_cols"]].copy()
    n_imp = 0; by_col = {}
    for c in po["X_cols"]:
        v = X[c].to_numpy(dtype="float64")
        v = np.clip(v, po["ranges"].loc["min", c], po["ranges"].loc["max", c])   # no extrapolation beyond the training range
        na = np.isnan(v)
        if na.any():
            v[na] = po["medians"][c]; n_imp += int(na.sum()); by_col[c] = int(na.sum())
        X[c] = v
    prep_X.last_by_col = by_col
    return X, n_imp

def _breaks_of(po):
    return np.asarray(po.get("risk_breaks", cfg.risk_breaks), dtype=float)

def predict_pest_df(po, df, month=None):
    """Scores for a table of cells. `month` (1-12, one value or one per row) is the calendar month of the map date. If the training
    records do not cover that month (po['covered_months']) the statistical part is an extrapolation: with cfg.uncovered_mode =
    "mechanistic" a pest that has a valid mechanistic part is scored by it alone for that month. Every cell always gets a score and a class."""
    n = len(df)
    X, n_imp = prep_X(po, df)
    mt = list(po["models"])
    P = np.full((n, len(mt)), np.nan)
    for j, m in enumerate(mt):
        P[:, j] = rescale_q(predict_model(po["models"][m], X), po["scale"][m])
    w = np.array([po["ens_w"][m] for m in mt])
    S = P @ w
    SD = np.sqrt(np.maximum(((P - S[:, None]) ** 2) @ w, 0))
    host = df["host"].to_numpy(dtype=float) if "host" in df.columns else np.ones(n)
    host = np.where(np.isnan(host), 0.0, host)
    M = np.full(n, np.nan) if po.get("static") else mech_index(po["pest"], df, emp=po.get("emp_thr"))
    w_m = po["w_mech"]
    Hraw = po["w_stat"] * S + (w_m * M if w_m > 0 else 0.0)
    scaling = po.get("scaling", "stretch")              # models saved by an older version keep the old stretch
    Pm = None
    if scaling == "percentile":
        Ps = to_pctl(S, po["ref_stat"])
        Pm = to_pctl(M, po["ref_mech"]) if (w_m > 0 and po.get("ref_mech") is not None) else None
        H = ((1 - w_m) * Ps + w_m * Pm if Pm is not None else Ps) * host
    else:
        lo, hi = po.get("stretch", (0.0, 1.0))
        H = clamp01((Hraw - lo) / max(hi - lo, 1e-9)) * host
    # ---- calendar months the training records do not cover: pests with a valid mechanistic part use it alone for that month ----
    cov_m = po.get("covered_months")
    if month is not None and cov_m and cfg.uncovered_mode == "mechanistic" and Pm is not None:
        mo = np.broadcast_to(np.asarray(month, dtype=float), (n,))
        unc = ~np.isin(mo, cov_m) & np.isfinite(mo) & np.isfinite(Pm)
        H = np.where(unc, Pm * host, H)
    cls = np.where(host == 0, 0.0, 1.0 + np.searchsorted(_breaks_of(po), H, side="right"))
    cls = np.where(np.isnan(H) & (host != 0), np.nan, cls)
    out = pd.DataFrame(P, columns=["stat_" + m for m in mt])
    out["stat_ensemble"] = S * host
    out["stat_sd"] = SD
    out["mechanistic"] = M
    out["hybrid"] = H
    out["hybrid_raw"] = Hraw * host
    out["risk_class"] = cls
    out["host"] = host
    out.attrs["n_imputed"] = n_imp
    out.attrs["by_col"] = dict(prep_X.last_by_col)
    return out

## 12. Train one pest

**What this cell does:** trains one pest end to end. It splits 70/30, runs spatial CV on the 70 %, weights the models (prior x performance x an over-fit factor, with a stricter floor for the GAM), sets the mechanistic share (fixed, below 0.5, and 0 if the mechanistic part fails its AUC check), builds the percentile scale from the held-out scores, sets the class breaks from the record scores, notes which calendar months the records cover, scores everything on the 30 % hold-out, then refits on all rows. **Output:** accuracy tables, risk-class table and the saved model for the pest.

In [ ]:
def _risk_col(D, p, prev):
    """hybrid risk of an already trained pest at the rows of D, computed (recursively) after the columns it needs exist"""
    col = RISK_COL[p]
    if col in D.columns:
        return col
    for c in prev[p]["vars"]:
        if c in RISK_COL.values() and c not in D.columns:
            src = [k for k, v in RISK_COL.items() if v == c][0]
            if prev.get(src) is not None:
                _risk_col(D, src, prev)
    D[col] = predict_pest_df(prev[p], D)["hybrid"].to_numpy()
    return col

def vector_weights(prev):
    ps = [p for p in cfg.mlnd_vectors if prev.get(p) is not None]
    wd = cfg.mlnd_vector_weights or {}
    w = {p: float(wd.get(p, 1.0)) for p in ps}
    return w

def vector_pressure(df, w):
    num = np.zeros(len(df)); den = np.zeros(len(df))
    for p, wi in w.items():
        v = df[RISK_COL[p]].to_numpy(dtype=float)
        ok = np.isfinite(v)
        num += np.where(ok, wi * v, 0.0); den += np.where(ok, wi, 0.0)
    return np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)

def add_coupling(D, pest, prev):
    if pest == "APHIDS" and cfg.aphids_use_faw_risk and prev.get("FAW") is not None:
        _risk_col(D, "FAW", prev)
    if pest == "MLND" and cfg.mlnd_vectors:
        w = vector_weights(prev)
        if w:
            for p in w:
                _risk_col(D, p, prev)
            D["vector_pressure"] = vector_pressure(D, w)
            if not cfg.mlnd_vector_weights:
                warn("MLND vector weights are EQUAL (placeholder). Set cfg.mlnd_vector_weights from literature. Vectors used: ", ", ".join(w))
            D.attrs["vector_w"] = w
    return D

def random_split_rows(X, y, xy, types, ew):
    """COMPARISON ONLY: the same models scored on a random stratified 70/30 split. Test points sit next to training points, so
    these AUCs are optimistic. They are reported next to the spatial-block AUCs to show how much a random split flatters the result."""
    trr, ter = split_train_test(xy, y, cfg.test_frac, cfg.cv_block_m, "random", cfg.seed)
    Xa, ya = X.iloc[trr].reset_index(drop=True), np.asarray(y)[trr]
    Xb, yb = X.iloc[ter].reset_index(drop=True), np.asarray(y)[ter]
    wa = balance_weights(ya); out, P, W = [], [], []
    for m in types:
        ob = fit_model(m, Xa, ya, wa)
        if ob is None:
            continue
        q = np.nanquantile(predict_model_raw(ob, Xa), [0.01, 0.99])
        pt = rescale_q(predict_model_raw(ob, Xb), q)
        P.append(pt); W.append(float(ew.get(m, 0.0)))
        out.append(dict(component=m + "_random_split", kind="Comparison only (random split)", test_auc=auc_of(yb, pt),
                        n_test_presence=int((yb == 1).sum()), n_test_background=int((yb == 0).sum())))
    if P:
        W = np.array(W) if sum(W) > 0 else np.ones(len(P))
        ens = np.column_stack(P) @ (W / W.sum())
        out.append(dict(component="stat_ensemble_random_split", kind="Comparison only (random split)", test_auc=auc_of(yb, ens),
                        n_test_presence=int((yb == 1).sum()), n_test_background=int((yb == 0).sum())))
        log("random 70/30 split (comparison only) ensemble test AUC %.3f" % out[-1]["test_auc"])
    return out

def train_pest(ctx, pest, prev=None):
    prev = prev or {}
    county = ctx.county
    log("======== TRAIN ", pest, " | ", county, " ========")
    pts = read_incidence(county, pest, ctx)
    if pts is None or len(pts) < cfg.min_presence:
        warn(pest, ": fewer than ", cfg.min_presence, " usable presences. Skipped.")
        return None
    if len(pts) < 30:
        warn(pest, ": only ", len(pts), " presences. Treat the statistical part with caution.")
    bg = sample_background(cfg.n_background, ctx, pts.date.to_numpy())
    allp = pd.concat([pts[["x", "y", "date"]].assign(presence=1), bg[["x", "y", "date"]].assign(presence=0)], ignore_index=True)
    D = build_static_matrix(ctx, allp[["x", "y", "date"]]) if cfg.static_mode else build_training_matrix(ctx, allp[["x", "y", "date"]])
    D["presence"] = allp.presence.to_numpy()
    D = add_coupling(D, pest, prev)
    vec_w = D.attrs.get("vector_w")

    want = pred_set(pest) + [c for c in COUPLING[pest] if c in D.columns]
    if pest == "APHIDS" and not cfg.aphids_use_faw_risk:
        want = [c for c in want if c != "faw_risk"]
    miss = [c for c in want if c not in D.columns]
    if miss:
        warn(pest, ": predictors not available and skipped: ", ", ".join(miss))
    vars_ = [c for c in want if c in D.columns]
    if len(vars_) < 3:
        warn(pest, ": fewer than 3 predictors. Skipped."); return None

    crop_levels = None
    if "crop.type" in vars_:
        tb = D.loc[(D.presence == 1) & D["crop.type"].notna(), "crop.type"].value_counts()
        crop_levels = sorted(float(k) for k, v in tb.items() if v >= 3)     # rare levels dropped
        if not crop_levels:
            vars_ = [c for c in vars_ if c != "crop.type"]
    X = design_matrix(D, vars_, crop_levels)
    y = D.presence.to_numpy().astype(int)
    if len(np.unique(y)) < 2:
        warn(pest, ": training set has only one class (no background cells). Check cfg.host_mode and the crop mask. Skipped.")
        return None

    na_share = X.isna().mean()
    if (na_share > 0.30).any():
        warn(pest, ": dropped for >30% missing: ", ", ".join(X.columns[na_share > 0.30]))
        X = X.loc[:, (na_share <= 0.30).to_numpy()]
    med = X.median()
    X = X.fillna(med)
    sdv = X.std()
    X = X.loc[:, (np.isfinite(sdv) & (sdv > 0)).to_numpy()]
    cont = [c for c in X.columns if not c.startswith("crop.type_")]
    keep_c = prune_collinear(X[cont], cfg.collinear_cut, keep=[c for c in ["faw_risk", "vector_pressure"] if c in cont])
    dropped = [c for c in cont if c not in keep_c]
    if dropped:
        log(pest, ": removed collinear predictors (|r| > ", cfg.collinear_cut, "): ", ", ".join(dropped))
    X = X[keep_c + [c for c in X.columns if c not in cont]]
    log(pest, ": ", int((y == 1).sum()), " presences, ", int((y == 0).sum()), " background, ", X.shape[1], " predictors kept")

    rng_df = pd.DataFrame({"min": X.min(), "max": X.max()}).T
    med = med[list(X.columns)]

    # ---- 70 / 30 split: the 30 % test part is never used for fitting, weighting or thresholds ----
    xy_all = D[["x", "y"]].to_numpy()
    tr, te = split_train_test(xy_all, y, cfg.test_frac, cfg.cv_block_m, cfg.split_method, cfg.seed)
    log(pest, ": 70/30 split (", cfg.split_method, "): train ", len(tr), " rows with ", int(y[tr].sum()), " presences | test ",
        len(te), " rows with ", int(y[te].sum()), " presences")
    Xtr, ytr = X.iloc[tr].reset_index(drop=True), y[tr]
    Xte, yte = X.iloc[te].reset_index(drop=True), y[te]
    wtr = balance_weights(ytr)

    types = list(cfg.models)
    if cfg.res_m < 100 and not cfg.svm_at_fine_res and "svm" in types:
        types.remove("svm")
        log("SVM left out: its prediction is too slow on a ", cfg.res_m, " m grid. Set cfg.svm_at_fine_res = True to keep it.")
    for pk, mod in (("maxent", "elapid"), ("gam", "pygam"), ("xgboost", "xgboost")):
        if pk in types and importlib.util.find_spec(mod) is None:
            types.remove(pk); warn("not run (package missing): ", pk)

    # ---- spatial cross-validation INSIDE the 70 % training part (used for the weights) ----
    fold = spatial_folds(xy_all[tr], ytr, cfg.cv_folds, cfg.cv_block_m)
    log(pest, ": spatial cross-validation on the training part (", cfg.cv_folds, " folds) for ", ", ".join(types))
    oof = cv_models(Xtr, ytr, wtr, fold, types)
    auc_cv = {m: auc_of(ytr, oof[m]) for m in types}

    # ---- fit on the 70 % and score the 30 % ----
    mtr = {}
    for m in types:
        ob = fit_model(m, Xtr, ytr, wtr)
        if ob is not None:
            mtr[m] = ob
    if not mtr:
        warn(pest, ": no statistical model could be fitted. Skipped."); return None
    types = list(mtr)
    qs = {}
    for m in types:
        v = oof[m].to_numpy()
        if np.isnan(v).all():
            v = predict_model_raw(mtr[m], Xtr)
        qs[m] = np.nanquantile(v, [0.01, 0.99])

    # ---- model weights = prior x performance on THIS data (CV AUC above 0.5) x an over-fit factor ----
    # over-fit gap = AUC of the model on the rows it was fitted on minus its spatial CV AUC. Random Forest and boosted trees reach about 1.0 on
    # their own rows, so a large gap means the model memorised the records and will transfer badly to new places or dates.
    fit_auc, gap, ofac = {}, {}, {}
    for m in types:
        try:
            fit_auc[m] = auc_of(ytr, predict_model_raw(mtr[m], Xtr))
        except Exception:
            fit_auc[m] = np.nan
        g = fit_auc[m] - auc_cv[m] if (np.isfinite(fit_auc[m]) and np.isfinite(auc_cv[m])) else 0.0
        gap[m] = float(max(g, 0.0))
        ofac[m] = float(np.exp(-gap[m] / cfg.overfit_scale)) if cfg.overfit_scale else 1.0
    log(pest, ": over-fit gap (own-row AUC minus CV AUC) | ", " | ".join("%s %.2f (x%.2f)" % (m, gap[m], ofac[m]) for m in types))
    prior = {m: float(cfg.model_prior.get(m, 1.0)) for m in types}
    floor = {m: max(cfg.min_auc, float((cfg.min_auc_by_model or {}).get(m, 0.0))) for m in types}
    perf = {m: (0.0 if (np.isnan(auc_cv[m]) or auc_cv[m] < floor[m]) else max(auc_cv[m] - 0.5, 0.0)) for m in types}
    ew = {m: prior[m] * perf[m] * ofac[m] for m in types}
    stat_reliable = True
    if sum(ew.values()) == 0:
        warn(pest, ": no model reached CV AUC ", cfg.min_auc, ". Prior weights only. The statistical part is UNRELIABLE for this pest.")
        ew = dict(prior); stat_reliable = False
    tot = sum(ew.values()); ew = {m: v / tot for m, v in ew.items()}
    wv = np.array([ew[m] for m in types])
    oofs = np.column_stack([np.nan_to_num(rescale_q(oof[m].to_numpy(), qs[m]), nan=0.0) for m in types])
    Pte = np.column_stack([rescale_q(predict_model_raw(mtr[m], Xte), qs[m]) for m in types])
    S_tr = oofs @ wv; S_te = Pte @ wv
    auc_s = auc_of(ytr, S_tr)

    # ---- mechanistic part and hybrid weight ----
    has_mech = has_mech_part(pest) and not cfg.static_mode      # the static (susceptibility) mode has no mechanistic part
    emp_thr = None
    if has_mech and pest not in MECH_PESTS:
        # AAW, Stem borer: temperature limits taken from the records of the 70 % training part (NOT literature values)
        if "tmean_mean" in D.columns:
            emp_thr = empirical_temp_thr(D["tmean_mean"].to_numpy(dtype=float)[tr][ytr == 1])
        if emp_thr is None:
            has_mech = False
            warn(pest, ": too few records to take thermal limits from them. No mechanistic part.")
        else:
            log(pest, ": EMPIRICAL thermal limits from the training records (tlow %.1f, optimum %.1f to %.1f, tup %.1f C). Not literature values, and the AUC "
                "below on the training rows is optimistic because the limits come from the same records. Read the held-out AUC." % emp_thr)
    D_nohost = D.copy(); D_nohost["host"] = 1.0          # judge the thermal rules on climate alone
    M_all = np.asarray(mech_index(pest, D_nohost, emp=emp_thr), dtype=float) if has_mech else np.full(len(D), np.nan)
    M_tr, M_te = M_all[tr], M_all[te]
    auc_m = auc_of(ytr, M_tr) if has_mech else float("nan")
    if not has_mech:
        w_m = 0.0
        log(pest, ": no mechanistic part (static mode or no sourced thresholds). Output = statistical only.")
    elif cfg.mech_min_auc is None or (np.isfinite(auc_m) and auc_m >= cfg.mech_min_auc):
        w_m = float(min(cfg.hybrid_w_mech_fixed, cfg.hybrid_w_mech_max))
        if not (0.0 <= w_m < 0.5):
            raise ValueError("cfg.hybrid_w_mech_fixed must be below 0.5 so the statistical part keeps the higher weight")
    else:
        w_m = 0.0
        warn(pest, ": mechanistic AUC is ", round(auc_m, 3) if np.isfinite(auc_m) else "NA", " (below ", cfg.mech_min_auc,
             "). The mechanistic part is switched OFF (weight 0) because it does not match the observed records. Review its thresholds.")
    Hr_tr = (1 - w_m) * S_tr + (w_m * M_tr if w_m > 0 else 0.0)
    Hr_te = (1 - w_m) * S_te + (w_m * M_te if w_m > 0 else 0.0)

    # ---- put the blended score on a 0-1 scale (no clamp, so no ties) ----
    scaling = cfg.risk_scaling
    lo, hi = 0.0, 1.0
    ref_stat = ref_mech = None
    if scaling == "percentile":
        # Each part becomes a percentile of the available-environment (background) scores. The statistical reference is the HELD-OUT part:
        # a model fitted on all the data scores new places the way it scores the hold-out, not the way cross-validation scored the training rows.
        ref_tr = make_ref(S_tr[ytr == 0])
        ref_stat = make_ref(S_te[yte == 0]) if (yte == 0).sum() >= 300 else None
        if ref_stat is None:
            warn(pest, ": fewer than 300 held-out background points, the statistical reference uses the cross-validation scores instead.")
            ref_stat = ref_tr
        Ps_tr, Ps_te = to_pctl(S_tr, ref_tr), to_pctl(S_te, ref_stat)
        if w_m > 0:
            ref_mech = make_ref(M_all[y == 0])
            H_tr = (1 - w_m) * Ps_tr + w_m * to_pctl(M_tr, ref_mech)
            H_te = (1 - w_m) * Ps_te + w_m * to_pctl(M_te, ref_mech)
        else:
            H_tr, H_te = Ps_tr, Ps_te
        log(pest, ": percentile scale built from ", int((yte == 0).sum()), " held-out and ", int((ytr == 0).sum()), " cross-validated background scores")
    else:
        if scaling == "stretch":
            bg = Hr_tr[ytr == 0]
            lo, hi = [float(v) for v in np.nanquantile(bg, list(cfg.stretch_q))]
            if not np.isfinite(lo + hi) or hi - lo < 1e-6:
                lo, hi = float(np.nanmin(Hr_tr)), float(np.nanmax(Hr_tr))
            if hi - lo < 1e-6:
                lo, hi = 0.0, 1.0
        sc = lambda h: clamp01((np.asarray(h, dtype=float) - lo) / (hi - lo))
        H_tr, H_te = sc(Hr_tr), sc(Hr_te)
        log(pest, ": risk stretch lo=%.3f hi=%.3f (percentiles %s of background scores)" % (lo, hi, tuple(cfg.stretch_q)))

    # ---- class breaks ----
    breaks = tuple(float(v) for v in cfg.risk_breaks)
    if cfg.class_mode == "presence_quantiles":
        pres = np.r_[H_tr[ytr == 1], H_te[yte == 1]]
        pres = pres[np.isfinite(pres)]
        if len(pres) >= 20:
            qb = np.quantile(pres, list(cfg.class_presence_q))
            breaks = tuple(float(v) for v in (np.maximum.accumulate(qb) + np.arange(len(qb)) * 1e-9))
        else:
            warn(pest, ": fewer than 20 scored presences, fixed class breaks used.")
    log(pest, ": class breaks ", tuple(round(v, 3) for v in breaks), " (", cfg.class_mode, ")")

    # ---- calendar months covered by the training records ----
    dts = pd.to_datetime(D["date"], errors="coerce")
    pmon = dts[y == 1].dt.month.dropna().astype(int)
    mshare = pmon.value_counts(normalize=True).reindex(range(1, 13)).fillna(0.0)
    covered = [int(m) for m in range(1, 13) if mshare[m] >= cfg.min_month_share]
    if cfg.static_mode or not covered:
        covered = list(range(1, 13))
    log(pest, ": months covered by the records (at least ", int(100 * cfg.min_month_share), "% of them): ", covered,
        " | share of records per month: ", {int(m): round(float(v), 3) for m, v in mshare.items() if v > 0})

    # ---- accuracy on the 30 % hold-out, thresholds fixed on the 70 %. Presence-background measures only: no absences are assumed ----
    def acc_row(name, kind, p_tr, p_te, pri=np.nan, wt=np.nan, cvauc=np.nan):
        pm = presence_metrics(ytr, p_tr, yte, p_te)
        return dict(component=name, kind=kind, prior_weight=pri, final_weight=wt, cv_auc_train=cvauc,
                    train_auc=auc_of(ytr, p_tr), test_auc=auc_of(yte, p_te), boyce=boyce_index(yte, p_te), **pm,
                    n_test_presence=int((yte == 1).sum()), n_test_background=int((yte == 0).sum()))
    rows = []
    for j, m in enumerate(types):
        rows.append(acc_row(m, "ML model", oofs[:, j], Pte[:, j], prior[m], ew[m], auc_cv[m]))
        rows[-1].update(fit_auc=fit_auc[m], overfit_gap=gap[m], overfit_factor=ofac[m])
    rows.append(acc_row("stat_ensemble", "Statistical ensemble", S_tr, S_te, np.nan, np.nan, auc_s))
    if has_mech:
        rows.append(acc_row("mechanistic", "Mechanistic", M_tr, M_te, np.nan, w_m, auc_m))
    rows.append(acc_row("hybrid", "Hybrid (final output)", H_tr, H_te, np.nan, np.nan, auc_of(ytr, H_tr)))
    if cfg.report_random_split:
        try:
            rows += random_split_rows(X, y, xy_all, types, ew)
        except Exception as e:
            warn(pest, ": random-split comparison failed: ", e)
    metrics = pd.DataFrame(rows)
    metrics.insert(0, "county", county); metrics.insert(0, "pest", pest)
    metrics["stat_reliable"] = stat_reliable; metrics["n_presence_total"] = int((y == 1).sum())
    log(pest, ": TEST AUC | ", " | ".join("%s %.3f" % (c, a) for c, a in zip(metrics.component, metrics.test_auc)))
    log(pest, ": hybrid weights  statistical %.2f  mechanistic %.2f" % (1 - w_m, w_m))

    # risk-class table on the test points: do the presences fall in the high classes?
    cls_te = 1 + np.searchsorted(np.asarray(breaks), H_te, side="right")
    ctab = []
    for c, lab in zip([1, 2, 3, 4, 5], ["Very low", "Low", "Medium", "High", "Very high"]):
        k = cls_te == c
        npres = int(((yte == 1) & k).sum()); nbg = int(((yte == 0) & k).sum())
        ctab.append(dict(pest=pest, county=county, risk_class=c, label=lab, test_presences=npres, test_background=nbg,
                         pct_of_presences=100.0 * npres / max((yte == 1).sum(), 1),
                         pct_of_background=100.0 * nbg / max((yte == 0).sum(), 1)))
    ctab = pd.DataFrame(ctab)
    ctab["presence_to_background_ratio"] = ctab.pct_of_presences / ctab.pct_of_background.replace(0, np.nan)

    # ---- deployed models: refit on 100 % of the points (the 30 % test is only used above) ----
    models = dict(mtr)
    if cfg.refit_on_all:
        wall = balance_weights(y); models = {}
        for m in types:
            ob = fit_model(m, X, y, wall)
            models[m] = ob if ob is not None else mtr[m]
        log(pest, ": models refitted on all ", len(y), " rows for deployment")

    po = dict(pest=pest, county=county, vars=vars_, crop_levels=crop_levels, X_cols=list(X.columns), ranges=rng_df,
              medians={c: float(med[c]) for c in X.columns}, models=models, scale=qs, ens_w=ew, w_stat=1 - w_m,
              w_mech=w_m, stretch=(lo, hi), scaling=scaling, ref_stat=ref_stat, ref_mech=ref_mech,
              emp_thr=emp_thr, covered_months=covered, month_share={int(m): float(v) for m, v in mshare.items()}, class_mode=cfg.class_mode, metrics=metrics, class_table=ctab, thr_p10_hybrid=float(np.quantile(H_tr[ytr == 1], 0.10)),
              stat_reliable=stat_reliable, vector_w=vec_w, collinear_dropped=dropped,
              trained=datetime.now().isoformat(timespec="seconds"), n_presence=int((y == 1).sum()),
              risk_breaks=breaks, test_frac=cfg.test_frac, split_method=cfg.split_method, static=bool(cfg.static_mode))
    od = out_dir(county, "models")
    if emp_thr is not None:
        pd.DataFrame(dict(pest=pest, county=county, par=["tlow", "topt_lo", "topt_hi", "tup"], value_C=list(emp_thr), status="empirical",
                          source="percentiles %s of the temperature at the %d training records (70 %% part)" % (tuple(cfg.mech_empirical_q), int((ytr == 1).sum())))) \
            .to_csv(os.path.join(od, pest + "_empirical_thresholds.csv"), index=False)
    dump(po, os.path.join(od, pest + "_model.joblib"))
    metrics.to_csv(os.path.join(od, pest + "_accuracy_holdout.csv"), index=False)
    ctab.to_csv(os.path.join(od, pest + "_risk_class_table_test.csv"), index=False)
    tt = D[["x", "y", "date", "presence"]].reset_index(drop=True).copy()
    tt["split"] = "train"; tt.loc[te, "split"] = "test"
    tt["hybrid_score"] = np.nan; tt.loc[tr, "hybrid_score"] = H_tr; tt.loc[te, "hybrid_score"] = H_te
    tt.to_csv(os.path.join(od, pest + "_split_points.csv"), index=False)
    pd.concat([tt[["x", "y", "date", "presence", "split"]], X.reset_index(drop=True)], axis=1) \
        .to_csv(os.path.join(od, pest + "_training_table.csv"), index=False)
    if "rf" in models:
        vi = pd.DataFrame(dict(predictor=models["rf"]["cols"], importance=models["rf"]["fit"].feature_importances_))
        vi.sort_values("importance", ascending=False).to_csv(os.path.join(od, pest + "_importance_rf.csv"), index=False)
    return po

## 13. Block-wise raster prediction (works at 30 m, never holds the full stack in memory)

**What this cell does:** predicts the maps block by block and writes one GeoTIFF per product (statistical, mechanistic, hybrid, risk class, model SD). Output: files in `predictions/<date>/`.

In [ ]:
RISK_LABELS = {0: "No host", 1: "Very low", 2: "Low", 3: "Medium", 4: "High", 5: "Very high"}

def predict_pest_raster(po, stk, grid, outdir, tag):
    nm_out = ["stat_" + m for m in po["models"]] + ["stat_ensemble", "stat_sd", "mechanistic", "hybrid", "hybrid_raw", "risk_class", "host"]
    avail = set(stk.names)
    if {"s.nir", "s.red", "s.green"} <= avail:
        avail.add("mtvi2")
    if po.get("vector_w") and all(RISK_COL[p] in avail for p in po["vector_w"]):
        avail.add("vector_pressure")
    avail |= {a for a, b in ALIASES.items() if b in avail}          # derived on the fly in each window
    missing_layers = [c for c in po["vars"] + ([] if po.get("static") else ["tmean_mean", "tmean_sd", "tmean_mean_30d", "tmean_sd_30d"]) if c not in avail]
    if missing_layers:
        warn(po["pest"], ": layers missing for this date (filled with training medians): ", ", ".join(missing_layers))
    if ".inside" not in avail:
        raise RuntimeError("internal error: grid layer .inside missing from the predictor stack")
    allf = os.path.join(outdir, "%s_%s_all.tif" % (po["pest"], tag))
    try:
        month = int(pd.Timestamp(tag).month)            # calendar month of the map date (None for the static map)
    except Exception:
        month = None
    season_ok = not (month is not None and po.get("covered_months") and month not in po["covered_months"])
    basis = "statistical + mechanistic"
    if not season_ok:
        if cfg.uncovered_mode == "mechanistic" and po.get("w_mech", 0) > 0 and po.get("ref_mech") is not None:
            basis = "mechanistic only (month outside the record season)"
        else:
            basis = "statistical, extrapolated (month outside the record season)"
        warn(po["pest"], ": the training records do not cover month ", month, " (covered: ", po["covered_months"], "). Map basis: ", basis,
             ". Treat as indicative.")
    elif po.get("w_mech", 0) <= 0:
        basis = "statistical only"
    rpb = max(1, int(cfg.block_cells // grid.width))
    starts = list(range(0, grid.height, rpb))
    prof = dict(driver="GTiff", height=grid.height, width=grid.width, count=len(nm_out), dtype="float32", crs=grid.crs,
                transform=grid.transform, nodata=np.nan, compress="deflate", BIGTIFF="IF_SAFER")
    n_imp = 0; miss_col = {}; n_cells_in = 0
    with rasterio.open(allf, "w", **prof) as dst:
        for k, s in enumerate(starts, 1):
            n = min(rpb, grid.height - s)
            win = Window(0, s, grid.width, n)
            df = derive_monthly_columns(stk.read_window(win), stk.meta)
            if po.get("vector_w") and "vector_pressure" not in df.columns and all(RISK_COL[p] in df.columns for p in po["vector_w"]):
                df["vector_pressure"] = vector_pressure(df, po["vector_w"])
            res = np.full((len(nm_out), n * grid.width), np.nan, dtype="float32")
            ins = np.flatnonzero(~df[".inside"].isna().to_numpy())
            if len(ins):
                pr = predict_pest_df(po, df.iloc[ins].reset_index(drop=True), month=month)
                res[:, ins] = pr[nm_out].to_numpy(dtype="float32").T
                n_imp += pr.attrs.get("n_imputed", 0); n_cells_in += len(ins)
                for c_, k_ in pr.attrs.get("by_col", {}).items():
                    miss_col[c_] = miss_col.get(c_, 0) + k_
            dst.write(res.reshape(len(nm_out), n, grid.width), window=win)
            if k % 5 == 0 or k == len(starts):
                log(po["pest"], ": block ", k, "/", len(starts))
    if n_imp > 0:
        log(po["pest"], ": ", format(n_imp, ","), " missing predictor values filled with training medians")
        top = sorted(miss_col.items(), key=lambda kv: -kv[1])[:8]
        log(po["pest"], ": missing share of cells per predictor (largest first): ",
            ", ".join("%s %.0f%%" % (c_, 100.0 * k_ / max(n_cells_in, 1)) for c_, k_ in top))
    files = {}
    with rasterio.open(allf) as src:
        for i, nm in enumerate(nm_out, 1):
            f = os.path.join(outdir, "%s_%s_%s.tif" % (po["pest"], tag, nm))
            a = src.read(i)
            if nm in ("risk_class", "host"):
                if nm == "host":
                    a = np.round(a)          # saved as 0/1 (share of cropland rounded)
                u = np.where(np.isnan(a), 255, a).astype("uint8")
                write_tif(f, u, grid, names=[nm], dtype="uint8", nodata=255)
            else:
                write_tif(f, a, grid, names=[nm])
            files[nm] = f
    try:
        os.remove(allf)
    except OSError:
        pass
    # area per risk class
    with rasterio.open(files["risk_class"]) as r:
        cl = r.read(1)
    vals, cnt = np.unique(cl[cl != 255], return_counts=True)
    summ = pd.DataFrame(dict(pest=po["pest"], tag=tag, cls=vals, label=[RISK_LABELS.get(int(v)) for v in vals],
                             cells=cnt, area_ha=cnt * cfg.res_m ** 2 / 10000.0, season_covered=season_ok, map_basis=basis))
    summ.to_csv(os.path.join(outdir, "%s_%s_risk_area_summary.csv" % (po["pest"], tag)), index=False)
    return files

## 14. Predict one date for all pests (any date inside the weather record)

**What this cell does:** builds the weather and index layers for one map date and predicts all pests with the trained models.

In [ ]:
# predictor name -> index name in Notebook C's GeoTIFF file names (<County>_<NAME>_<yyyy>-<mm>.tif). Needs Notebooks A, A2, B and C run with the 18-index dataset.
FC_BAND_NAMES = {"s.red": "S_RED", "s.nir": "S_NIR", "s.red.edge3": "S_RE3", "s.coastal": "S_COASTAL", "s.vapour": "S_WV",
                 "l.coastal": "L_COASTAL", "l.blue": "L_BLUE", "l.lst": "L_LST_K"}

def fc_band_stack(ctx, ym):
    """Sentinel-2 and Landsat band layers of a forecast month, read from Notebook C's GeoTIFFs. No-data inside the county gets the county mean,
    LST goes from Kelvin to deg C. A band without a C file is simply absent here, so the seasonal stand-in (October 2025) still serves it."""
    if not _is_fc_ym(ym):
        return None
    need = needed_names()
    items, got, miss = [], [], []
    for pn, cn in FC_BAND_NAMES.items():
        if pn not in need:
            continue
        p = fc_vi_file(ctx.county, cn, ym)
        if p is None:
            miss.append(pn); continue
        f = os.path.join(out_dir(ctx.county, "aligned"), "%s_%s_fc_%s.tif" % (pn, ym, cfg.vi_fc_scenario))
        try:
            align_to_grid(p, ctx.grid, "bilinear", f)
            _fill_fc_with_mean(f, ctx.grid, cn, ctx.county, ym)
            if pn == "l.lst":
                with rasterio.open(f) as r:
                    a = r.read(1).astype("float32")
                if np.isfinite(a).any() and np.nanmedian(a) > 150:
                    write_tif(f, a - 273.15, ctx.grid, names=[pn])
            items.append((pn, f, 1)); got.append(pn)
        except Exception as e:
            warn(ctx.county, ": could not prepare Notebook C band ", cn, " for ", ym, " (", e, ")"); miss.append(pn)
    log(ctx.county, " ", ym, ": bands from Notebook C: ", ", ".join(got) or "none", " | stand-in (no C file): ", ", ".join(miss) or "none")
    return Stack(items) if items else None

def predict_one_date(ctx, d, objs):
    d = pd.Timestamp(d); tag = d.strftime("%Y-%m-%d")
    log("######## Predicting ", ctx.county, " for ", tag, " ########")
    od = out_dir(ctx.county, "predictions", tag)
    ym = pick_ym(d, ctx.avail)
    mon = get_monthly(ctx.county, ctx.grid, ym)
    wxs = wx_to_grid(ctx.county, ctx.grid, ctx.W, d, ctx.elev, ctx.ec)
    lan = None
    if needs_landsat():
        sc = nearest_scene(ctx.county, d)
        if sc:
            lan = get_landsat_stack(ctx.county, ctx.grid, sc[0])
            log("Landsat scene ", sc[1].date(), " used for ", tag)
        else:
            log("No Landsat monthly stack for ", tag, " in ", ctx.county)
    fcb = fc_band_stack(ctx, ym)                      # first in the merge: a Notebook C band wins over the stand-in layers
    base = Stack.from_file(grid_path(ctx.county), [".inside"]).merged(ctx.host, ctx.static, fcb, mon, lan, wxs)
    return _predict_pests(ctx, base, od, tag, objs)

def _predict_pests(ctx, base, od, tag, objs):
    """all pests on one predictor stack (a date, or the annual stack of the static mode)"""
    made, hyb = {}, {}
    for p in cfg.pests:
        po = objs.get(p)
        if po is None:
            log(p, ": no trained model, skipped"); continue
        stk = base
        if p == "APHIDS" and "faw_risk" in po["vars"]:
            if hyb.get("FAW"):
                stk = stk.merged(Stack.from_file(hyb["FAW"], ["faw_risk"]))
            else:
                warn("APHIDS needs faw_risk but FAW was not mapped. Median fill used.")
        for vp in (po.get("vector_w") or {}):
            if hyb.get(vp):
                stk = stk.merged(Stack.from_file(hyb[vp], [RISK_COL[vp]]))
            else:
                warn(p, " needs ", RISK_COL[vp], " but ", vp, " was not mapped. Its weight is ignored for those cells.")
        made[p] = predict_pest_raster(po, stk, ctx.grid, od, tag)
        hyb[p] = made[p]["hybrid"]
    if hyb:
        def stack_files(key):
            arrs = []
            for p in hyb:
                with rasterio.open(made[p][key]) as r:
                    arrs.append(r.read(1))
            return np.stack(arrs)
        write_tif(os.path.join(od, "ALL_PESTS_%s_hybrid.tif" % tag), stack_files("hybrid"), ctx.grid,
                  names=["%s_hybrid" % p for p in hyb])
        write_tif(os.path.join(od, "ALL_PESTS_%s_risk_class.tif" % tag), stack_files("risk_class"), ctx.grid,
                  names=["%s_class" % p for p in hyb], dtype="uint8", nodata=255)
    return made

def _dynamic_stack_for_date(ctx, d):
    d = pd.Timestamp(d)
    mon = get_monthly(ctx.county, ctx.grid, pick_ym(d, ctx.avail))
    wxs = wx_to_grid(ctx.county, ctx.grid, ctx.W, d, ctx.elev, ctx.ec)
    lan = None
    if needs_landsat():
        sc = nearest_scene(ctx.county, d)
        if sc:
            lan = get_landsat_stack(ctx.county, ctx.grid, sc[0])
    return Stack().merged(mon, lan, wxs)

def build_annual_stack(ctx):
    """STATIC mode: mean over the reference dates of every date-dependent predictor (and the seasonal SD of three weather layers),
    written once as a GeoTIFF window by window. Cached: delete the file to rebuild."""
    refs = static_ref_dates(ctx)
    od = out_dir(ctx.county, "aligned")
    f = os.path.join(od, "annual_static_%s_%s_%s.tif" % (cfg.res_m, refs[0].strftime("%Y%m%d"), refs[-1].strftime("%Y%m%d")))
    grid = ctx.grid
    static_names = set(ctx.static.names) | set(ctx.host.names)
    need = set(v for p in PRED_SETS.values() for v in p) | set(_SEASON_BASE.values()) | {"mtvi2"}
    stacks = []
    for d in refs:
        try:
            stacks.append((d, _dynamic_stack_for_date(ctx, d)))
        except Exception as e:
            warn("static mode: reference date ", d.date(), " skipped: ", e)
    if len(stacks) < 2:
        raise RuntimeError("static mode needs at least 2 usable reference dates")
    probe = Window(0, 0, grid.width, 1)
    names = []
    for d, s in stacks:
        for c in derive_monthly_columns(s.read_window(probe), s.meta).columns:
            if c in need and c not in static_names and c not in names:
                names.append(c)
    out_names = names + [n for n, b in _SEASON_BASE.items() if b in names]
    if os.path.exists(f) and cfg.reuse_cache and grid.same_geometry(f):
        log("annual stack reused: ", f)
        return Stack.from_file(f, out_names)
    log("building the annual stack from ", len(stacks), " reference dates (", len(names), " layers)")
    rpb = max(1, int(cfg.block_cells // grid.width) // 3)
    prof = dict(driver="GTiff", height=grid.height, width=grid.width, count=len(out_names), dtype="float32", crs=grid.crs,
                transform=grid.transform, nodata=np.nan, compress="deflate", BIGTIFF="IF_SAFER")
    with rasterio.open(f, "w", **prof) as dst:
        for s0 in range(0, grid.height, rpb):
            n = min(rpb, grid.height - s0); win = Window(0, s0, grid.width, n)
            S = np.zeros((len(names), n * grid.width)); S2 = np.zeros_like(S); C = np.zeros_like(S)
            for d, s in stacks:
                df = derive_monthly_columns(s.read_window(win), s.meta).reindex(columns=names)
                a = df.to_numpy(dtype=float).T
                ok = np.isfinite(a)
                S += np.where(ok, a, 0.0); S2 += np.where(ok, a * a, 0.0); C += ok
            with np.errstate(invalid="ignore", divide="ignore"):
                mean = np.where(C > 0, S / C, np.nan)
                sd = np.sqrt(np.maximum(np.where(C > 0, S2 / C, np.nan) - mean ** 2, 0.0))
            res = [mean[i] for i in range(len(names))] + [sd[names.index(b)] for nm, b in _SEASON_BASE.items() if b in names]
            dst.write(np.stack(res).reshape(len(out_names), n, grid.width).astype("float32"), window=win)
    log("annual stack saved: ", f)
    return Stack.from_file(f, out_names)

def predict_static(ctx, objs):
    """STATIC susceptibility maps (one per pest, no date) from the annual-mean predictors"""
    od = out_dir(ctx.county, "predictions", "static")
    ann = build_annual_stack(ctx)
    base = Stack.from_file(grid_path(ctx.county), [".inside"]).merged(ctx.host, ctx.static, ann)
    return _predict_pests(ctx, base, od, "static", objs)

## 15. Drivers

**What this cell does:** the drivers that train every pest of a county, save results and call the output steps. `run_pipeline(counties=[...])` is the one function you call.

In [ ]:
def save_run_info(county):
    od = out_dir(county, "models")
    THR.to_csv(os.path.join(od, "mechanistic_thresholds_and_sources.csv"), index=False)
    with open(os.path.join(od, "config_used.json"), "w") as fh:
        json.dump(vars(cfg), fh, indent=2, default=str)
    import platform
    ver = {m: getattr(sys.modules.get(m), "__version__", "") for m in
           ["numpy", "pandas", "scipy", "rasterio", "geopandas", "sklearn", "xgboost", "pygam", "elapid"]}
    with open(os.path.join(od, "session_info.txt"), "w") as fh:
        fh.write("python %s on %s\n" % (sys.version.split()[0], platform.platform()))
        for k, v in ver.items():
            fh.write("%s %s\n" % (k, v))

def load_models(county, models_dir=None):
    od = models_dir or out_dir(county, "models")
    o = {}
    for p in cfg.pests:
        f = os.path.join(od, p + "_model.joblib")
        if os.path.exists(f):
            o[p] = load(f)
    return o

def run_pipeline(counties=None, pests=None, predict_dates=None, train=True):
    counties = counties or cfg.counties
    pests = pests or cfg.pests
    predict_dates = predict_dates if predict_dates is not None else cfg.predict_dates
    cfg.pests = list(pests)
    res = {}
    for cty in counties:
        set_logfile(os.path.join(out_dir(cty), "run_log_%s.txt" % datetime.now().strftime("%Y%m%d_%H%M%S")))
        t0 = time.time()
        ctx = build_context(cty)
        rep0 = data_report(cty)
        log("Incidence files found: ", rep0["incidence"])
        log("Daily weather files: ", rep0["weather_files"])
        save_run_info(cty)
        objs = {} if train else load_models(cty)
        if train:
            prev = {}
            for p in pests:
                try:
                    po = train_pest(ctx, p, prev)
                except Exception as e:
                    import traceback; traceback.print_exc()
                    warn(p, " training failed: ", e); po = None
                if po is not None:
                    objs[p] = po; prev[p] = po
        if train and objs:
            try:
                save_bundle(cty, objs)
            except Exception as e:
                warn("could not save the model bundle: ", e)
        dts = predict_dates
        if dts is None and ctx.W:
            dts = [wx_last_date(ctx.W).strftime("%Y-%m-%d")]
        maps = {}
        if cfg.static_mode:
            dts = []
            try:
                maps["static"] = predict_static(ctx, objs)
            except Exception as e:
                import traceback; traceback.print_exc()
                warn("static prediction failed: ", e)
        for d in (dts or []):
            try:
                maps[str(d)] = predict_one_date(ctx, d, objs)
            except Exception as e:
                import traceback; traceback.print_exc()
                warn("prediction for ", d, " failed: ", e)
        try:
            finalize_county(cty, objs, maps)          # accuracy workbook, GeoTIFF export, PNG maps (section 16)
        except Exception as e:
            import traceback; traceback.print_exc()
            warn("final outputs (accuracy file, exports, plots) failed for ", cty, ": ", e)
        res[cty] = dict(models=objs, maps=maps, ctx=ctx)
        log("Finished ", cty, " in %.1f min" % ((time.time() - t0) / 60))
    return res

import contextlib
@contextlib.contextmanager
def _override(**kw):
    old = {k: getattr(cfg, k) for k in kw if v_is_set(kw[k])}
    try:
        for k, v in kw.items():
            if v_is_set(v):
                setattr(cfg, k, v)
        yield
    finally:
        for k, v in old.items():
            setattr(cfg, k, v)

def v_is_set(v):
    return v is not None

def run_static(counties=None, pests=None, out_root=None):
    """STATIC susceptibility run: annual-mean predictors (terrain, soil, annual means and seasonality of the project's own weather,
    vegetation index and surface temperature layers), host-only background, spatial 70/30 test PLUS a random 70/30 comparison.
    Each county is trained and mapped separately. Everything is written to a SEPARATE folder (default <out_root>_susceptibility),
    nothing in the dynamic run's folder is changed. The caches of the dynamic run are reused."""
    root = out_root or cfg.static_out_root or (cfg.out_root.rstrip("/\\") + "_susceptibility")
    log("STATIC susceptibility run. Outputs go to: ", root)
    with _override(out_root=root, static_mode=True, report_random_split=True, cache_root=cfg.out_root):
        return run_pipeline(counties=counties, pests=pests, predict_dates=[])

def predict_dates_only(county, dates, pests=None, h_root=None, out_root=None, models_dir=None, outputs=True, v_root=None, forecast=None, fc_scenario=None):
    """Reuse saved models, no retraining: map any dates for which the input datasets exist.
    h_root    : drive/folder holding NEW soil / crop / Sentinel-2 datasets in the SAME folder structure (default cfg.h_root)
    forecast  : True = append the SEAS5 forecast days (cfg.fc_root) after the historical weather, so dates up to the end of the forecast can be mapped
    fc_scenario: "mean", "p10" or "p90" (default cfg.fc_scenario)
    v_root    : folder holding NEW weather / vegetation index / DEM / Landsat datasets in the same structure as F:/VI_prediction (default cfg.v_root)
    out_root  : where to write the new maps (default cfg.out_root). Use a new folder with new inputs, so old cached layers are not reused.
    models_dir: folder with the saved models (default <cfg.out_root>/<County>/models of the training run)"""
    md = models_dir or out_dir(county, "models")          # resolved BEFORE out_root is overridden
    with _override(h_root=h_root, v_root=v_root, out_root=out_root, pests=list(pests) if pests else None, use_forecast=forecast, fc_scenario=fc_scenario):
        if out_root:
            os.makedirs(cfg.out_root, exist_ok=True)
        set_logfile(os.path.join(out_dir(county), "run_log_predict_%s.txt" % datetime.now().strftime("%Y%m%d_%H%M%S")))
        ctx = build_context(county)
        objs = load_models(county, md)
        if not objs:
            raise RuntimeError("No saved models in %s. Run run_pipeline() first." % md)
        maps = {str(d): predict_one_date(ctx, d, objs) for d in dates}
        if outputs:
            finalize_county(county, objs, maps, write_accuracy_file=False)
        return maps

def predict_points(county, points, pests=None, h_root=None, models_dir=None, v_root=None, forecast=None, fc_scenario=None):
    """Score a table of points with the saved models. `points` is a DataFrame with columns lon, lat, date (yyyy-mm-dd).
    Returns the table with <PEST>_risk (0-1 hybrid) and <PEST>_class (1 Very low ... 5 Very high, 0 = no host) for each pest."""
    md = models_dir or out_dir(county, "models")
    with _override(h_root=h_root, v_root=v_root, use_forecast=forecast, fc_scenario=fc_scenario):
        ctx = build_context(county)
        objs = load_models(county, md)
        pts = points.copy()
        tr_ = Transformer.from_crs("EPSG:4326", ctx.grid.crs, always_xy=True)
        xx, yy = tr_.transform(pts["lon"].to_numpy(), pts["lat"].to_numpy())
        D = build_training_matrix(ctx, pd.DataFrame(dict(x=xx, y=yy, date=pd.to_datetime(pts["date"]))))
        for p in (pests or cfg.pests):
            po = objs.get(p)
            if po is None:
                continue
            if po.get("vector_w") and all(RISK_COL[v] in D.columns for v in po["vector_w"]):
                D["vector_pressure"] = vector_pressure(D, po["vector_w"])
            mo = pd.to_datetime(pts["date"]).dt.month.to_numpy() if "date" in pts.columns else None
            out = predict_pest_df(po, D, month=mo)
            D[RISK_COL[p]] = out["hybrid"].to_numpy()
            pts[p + "_risk"] = out["hybrid"].to_numpy()
            pts[p + "_class"] = out["risk_class"].to_numpy()
        return pts

## 16. Saved model bundle, accuracy workbook, GeoTIFF export and maps
Everything below runs automatically at the end of `run_pipeline`, and can be called again on its own:
- `save_bundle(county, objs)`: all trained pests of a county in ONE file (`PestRisk_bundle_<County>.joblib`) plus a JSON model card per pest.
- `write_accuracy(county, objs)`: the 70/30 hold-out accuracy workbook `Accuracy_Assessment_<County>.xlsx` (and CSV copies).
- `export_geotiffs(county, tag)`: every product as an EPSG:4326 (decimal degrees) GeoTIFF.
- `plot_maps(county, tag)`: one PNG per pest and product (Spectral_r, decimal-degree axes with E and N, interior ticks, outline, north arrow, scale bar).

**What this cell does:** saves the trained models, writes the accuracy tables, exports GeoTIFFs in decimal degrees and draws the maps for the **statistical, mechanistic and hybrid** outputs plus the risk class (grey = no training evidence for that month). Called automatically at the end of a run.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
from matplotlib.colors import BoundaryNorm, ListedColormap
from rasterio.warp import calculate_default_transform

CLASS_LABELS = ["Very low", "Low", "Medium", "High", "Very high"]
PEST_NAME = dict(FAW="Fall armyworm", APHIDS="Aphids", THRIPS="Thrips", STEMBORER="Stem borer",
                 AAW="African armyworm", MLND="Maize Lethal Necrosis Disease")
PRODUCT_TITLE = dict(hybrid="Hybrid risk (statistical + mechanistic)", hybrid_raw="Hybrid risk (raw, before percentile scaling)", risk_class="Risk class",
                     stat_ensemble="Statistical suitability (similarity to record conditions)", mechanistic="Mechanistic risk", stat_sd="Model disagreement (SD)")

def _json_safe(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, (pd.DataFrame,)):
        return o.to_dict("records")
    if isinstance(o, (np.ndarray, tuple, set)):
        return list(o)
    return str(o)

def save_bundle(county, objs):
    """One file holding every trained pest of the county, plus a readable model card per pest."""
    od = out_dir(county, "models")
    bundle = dict(county=county, created=datetime.now().isoformat(timespec="seconds"), pests=list(objs), models=objs,
                  cfg=json.loads(json.dumps(vars(cfg), default=_json_safe)),
                  how_to_use="from the notebook: predict_dates_only(county, dates, h_root=NEW_INPUT_FOLDER, out_root=NEW_OUTPUT_FOLDER) "
                             "or predict_points(county, points_df). The bundle can also be opened with joblib.load.")
    f = os.path.join(od, "PestRisk_bundle_%s.joblib" % county)
    dump(bundle, f)
    for p, po in objs.items():
        card = dict(pest=p, county=county, trained=po.get("trained"), n_presence=po.get("n_presence"),
                    predictors=po["X_cols"], crop_levels=po.get("crop_levels"), ensemble_weights=po["ens_w"],
                    hybrid_weights=dict(statistical=po["w_stat"], mechanistic=po["w_mech"]), risk_scaling=po.get("scaling", "stretch"), risk_stretch=list(po.get("stretch", (0, 1))), covered_months=po.get("covered_months"), month_share=po.get("month_share"),
                    risk_breaks=list(po.get("risk_breaks", cfg.risk_breaks)), vector_weights=po.get("vector_w"),
                    statistical_part_reliable=bool(po.get("stat_reliable", True)), split=dict(test_frac=po.get("test_frac"), method=po.get("split_method")),
                    holdout_accuracy=po["metrics"].round(4).to_dict("records"))
        with open(os.path.join(od, "model_card_%s.json" % p), "w") as fh:
            json.dump(card, fh, indent=2, default=_json_safe)
    log("Model bundle saved: ", f)
    return f

def _latest_tag(county):
    d = os.path.join(cfg.out_root, county, "predictions")
    if not os.path.isdir(d):
        return None
    t = sorted(x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x)))
    return t

def map_check(county, objs, tags=None):
    """does the FINAL class map put the held-out presences in the high classes? (spatial check: the map date is one date,
    the records are from many dates, so this is a check of the spatial pattern)"""
    rows = []
    tags = tags or (_latest_tag(county) or [])
    for tag in tags:
        for p in objs:
            fc = os.path.join(cfg.out_root, county, "predictions", tag, "%s_%s_risk_class.tif" % (p, tag))
            fs = os.path.join(out_dir(county, "models"), p + "_split_points.csv")
            fa = os.path.join(cfg.out_root, county, "predictions", tag, "%s_%s_risk_area_summary.csv" % (p, tag))
            if not (os.path.exists(fc) and os.path.exists(fs)):
                continue
            sp = pd.read_csv(fs); sp = sp[(sp.split == "test") & (sp.presence == 1)]
            with rasterio.open(fc) as r:
                vals = np.array([v[0] for v in r.sample(list(zip(sp.x, sp.y)))])
            area = pd.read_csv(fa) if os.path.exists(fa) else None
            tot_area = float(area.loc[area.cls >= 1, "area_ha"].sum()) if area is not None else np.nan
            for c, lab in zip([1, 2, 3, 4, 5], CLASS_LABELS):
                n = int((vals == c).sum())
                a = float(area.loc[area.cls == c, "area_ha"].sum()) if area is not None else np.nan
                rows.append(dict(pest=p, county=county, map_date=tag, risk_class=c, label=lab, test_presences=n,
                                 pct_of_test_presences=100.0 * n / max(len(vals), 1), area_ha=a,
                                 pct_of_host_area=100.0 * a / tot_area if tot_area else np.nan))
    df = pd.DataFrame(rows)
    if len(df):
        df["presence_share_to_area_share"] = df.pct_of_test_presences / df.pct_of_host_area.replace(0, np.nan)
    return df

def write_accuracy(county, objs, tags=None):
    """Accuracy workbook: ML models, final outputs, risk classes, weights, map check."""
    od = out_dir(county, "accuracy")
    met = pd.concat([po["metrics"] for po in objs.values()], ignore_index=True)
    ml = met[met.kind == "ML model"].copy()
    fin = met[met.kind != "ML model"].copy()
    wts = ml[["pest", "county", "component", "prior_weight", "cv_auc_train", "final_weight"]].rename(
        columns=dict(component="model", cv_auc_train="cv_auc_on_70pct", final_weight="ensemble_weight"))
    hyb = pd.DataFrame([dict(pest=p, county=county, statistical_weight=po["w_stat"], mechanistic_weight=po["w_mech"],
                             risk_scaling=po.get("scaling", "stretch"), stretch_low=po["stretch"][0], stretch_high=po["stretch"][1],
                             break_1=po["risk_breaks"][0], break_2=po["risk_breaks"][1], break_3=po["risk_breaks"][2], break_4=po["risk_breaks"][3],
                             months_covered=",".join(str(m) for m in po.get("covered_months", [])),
                             statistical_part_reliable=bool(po.get("stat_reliable", True)), n_presence_total=po["n_presence"])
                        for p, po in objs.items()])
    ctab = pd.concat([po["class_table"] for po in objs.values()], ignore_index=True)
    mc = map_check(county, objs, tags)
    readme = pd.DataFrame(dict(note=[
        "70/30 hold-out: %d%% of the points (whole spatial blocks when split_method = block) were kept aside and used ONLY for these numbers." % round(100 * cfg.test_frac),
        "Ensemble weights and the mechanistic weight were fixed on the 70% part. The test part never influenced them. The final map score is a PERCENTILE: the statistical and the mechanistic score are each turned into a percentile of the available-environment (background) scores, then blended with the fixed weights. No clamp is used, so cells do not tie at 0 or 1.",
        "The statistical percentile reference is the HELD-OUT 30% part, because a model fitted on all the data scores new places the way it scores the hold-out, not the way cross-validation scored the training rows. The class breaks are quantiles of the scores of the training (cross-validated) and held-out records together (Very low = below the score that 90% of records reach, Low up to the 30th percentile of the records, Medium up to the 60th, High up to the 85th, Very high above; see cfg.class_presence_q). These are set from the same records, so the Risk_class_table is DESCRIPTIVE, not an independent test. AUC, Boyce and the sensitivity numbers do not depend on the class breaks.",
        "Model weights = prior x (CV AUC - 0.5) x an over-fit factor. Over-fit gap = AUC on the model's own training rows minus its spatial CV AUC; the factor is exp(-gap / cfg.overfit_scale). See the overfit_gap and overfit_factor columns in Model_accuracy.",
        "Months: a calendar month is COVERED if it holds at least cfg.min_month_share of the pest's records (see months_covered in Hybrid_settings). For a map date in a month that is not covered, the statistical part is an extrapolation. Pests with a valid mechanistic part then show the mechanistic score only. The others keep the statistical map. These maps carry a note in the title and season_covered = False and map_basis in the area-summary csv: read them as indicative.",
        "BACKGROUND POINTS ARE NOT ABSENCES. They sample the environment that is available in the host area. The models learn how the climate, soil, terrain and vegetation at the records differ from it, so a score is a relative environmental SUITABILITY (similarity to the conditions at the records), not the probability that the pest is present.",
        "For that reason no absence-based measure is reported: no specificity, overall accuracy, kappa, TSS or Brier score.",
        "AUC: 0.5 = no better than chance. It reads as how well held-out records are ranked above available-environment cells. With background it cannot reach 1, so compare AUCs between models and pests, not against 1.",
        "Boyce index: +1 = records sit in the high scores, 0 = random, negative = wrong way round. It needs no absences.",
        "threshold_p10: the 10th percentile of the scores at the TRAINING records (threshold_min: the lowest one). sensitivity_p10 = share of the held-out records at or above it (about 0.90 expected if the model transfers to new places).",
        "pct_landscape_above_p10 = share of the available-environment cells at or above that threshold: smaller = a more selective map. sens_to_area_ratio_p10 = sensitivity divided by that share (above 1 = better than a random map of the same size).",
        "Sheet Model_accuracy: every ML model. Sheet Final_outputs: statistical ensemble, mechanistic, final hybrid (the percentile map score).",
        "Sheet Risk_class_table: where the test records fall in Very low, Low, Medium, High, Very high. presence_to_background_ratio = share of records in the class divided by the share of the available environment in it (predicted / expected). Above 1 = the class holds more records than its area would give by chance.",
        "Sheet Map_check: the same check on the saved class MAP of one date, using the area of each class. The map date differs from the record dates.",
        "statistical_part_reliable = FALSE means no model reached the minimum CV AUC (cfg.min_auc). Treat that pest's map as indicative only.",
        "Rows named ..._random_split (static susceptibility run only) are COMPARISON ONLY: a random split puts test points next to training points, so those AUCs are optimistic. Report the spatial-block AUC.",
        "Small samples (a few dozen presences) give wide uncertainty on every number here. Report them with the number of test presences."]))
    sheets = [("README", readme), ("Model_accuracy", ml), ("Final_outputs", fin), ("Ensemble_weights", wts),
              ("Hybrid_settings", hyb), ("Risk_class_table", ctab), ("Map_check", mc)]
    xl = os.path.join(od, "Accuracy_Assessment_%s.xlsx" % county)
    try:
        with pd.ExcelWriter(xl, engine="openpyxl") as w:
            for nm, df in sheets:
                df = df.copy()
                for c in df.columns:
                    if pd.api.types.is_float_dtype(df[c]):
                        df[c] = df[c].round(4)
                df.to_excel(w, sheet_name=nm, index=False)
                ws = w.sheets[nm]
                for i, c in enumerate(df.columns, 1):
                    width = 110 if nm == "README" else min(max(len(str(c)), 12) + 2, 32)
                    ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = width
        log("Accuracy workbook saved: ", xl)
    except Exception as e:
        warn("Excel writer not available (", e, "). CSV files only.")
        xl = None
    for nm, df in sheets:
        df.to_csv(os.path.join(od, "%s_%s.csv" % (nm, county)), index=False)
    return xl

def _to_wgs84(src_file, dst_file, categorical):
    with rasterio.open(src_file) as src:
        arr = src.read(1).astype("float32")
        nd = src.nodata
        if nd is not None and not np.isnan(nd):
            arr[arr == nd] = np.nan
        tr, w, h = calculate_default_transform(src.crs, "EPSG:4326", src.width, src.height, *src.bounds)
        dst = np.full((h, w), np.nan, dtype="float32")
        reproject(arr, dst, src_transform=src.transform, src_crs=src.crs, dst_transform=tr, dst_crs="EPSG:4326",
                  resampling=Resampling.nearest if categorical else Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    os.makedirs(os.path.dirname(dst_file), exist_ok=True)
    if categorical:
        u = np.where(np.isnan(dst), 255, dst).astype("uint8")
        prof = dict(driver="GTiff", height=h, width=w, count=1, dtype="uint8", crs="EPSG:4326", transform=tr, nodata=255, compress="deflate")
        with rasterio.open(dst_file, "w", **prof) as o:
            o.write(u, 1)
    else:
        prof = dict(driver="GTiff", height=h, width=w, count=1, dtype="float32", crs="EPSG:4326", transform=tr, nodata=np.nan, compress="deflate")
        with rasterio.open(dst_file, "w", **prof) as o:
            o.write(dst, 1)
    l, t = tr * (0, 0); r, b = tr * (w, h)
    return dst, (l, r, b, t)

def export_geotiffs(county, tag, pests=None):
    """every product in cfg.export_products as an EPSG:4326 GeoTIFF: <County>/exports_wgs84/<date>/"""
    n = 0
    for p in (pests or cfg.pests):
        for prod in cfg.export_products:
            f = os.path.join(cfg.out_root, county, "predictions", tag, "%s_%s_%s.tif" % (p, tag, prod))
            if prod == "mechanistic" and (not has_mech_part(p) or cfg.static_mode):
                continue                                           # no sourced thresholds for this pest, nothing to export
            if os.path.exists(f):
                _to_wgs84(f, os.path.join(out_dir(county, "exports_wgs84", tag), "%s_%s_%s_wgs84.tif" % (p, tag, prod)), prod == "risk_class")
                n += 1
    log("Exported ", n, " GeoTIFFs (EPSG:4326) for ", county, " ", tag)
    return n

def _outline(county):
    try:
        return read_county_boundary(county).to_crs(4326)
    except Exception:
        return None

def _fx(dec):
    return FuncFormatter(lambda v, _: "%.*f\u00b0%s" % (dec, abs(v), "E" if v >= 0 else "W"))
def _fy(dec):
    return FuncFormatter(lambda v, _: "%.*f\u00b0%s" % (dec, abs(v), "N" if v >= 0 else "S"))

def _save_png(fig, png):
    """save a figure. If Windows refuses the target path (OSError 22), save to the temp folder and copy it over.
    A map that still cannot be saved gives a warning with the exact path (repr shows hidden characters) and the run goes on."""
    import tempfile, shutil
    png = os.path.abspath(png)
    os.makedirs(os.path.dirname(png), exist_ok=True)
    try:
        fig.savefig(png, dpi=cfg.plot_dpi, bbox_inches="tight", facecolor="white")
        return True
    except OSError as e1:
        try:
            tmp = os.path.join(tempfile.gettempdir(), "pestrisk_map_tmp.png")
            fig.savefig(tmp, dpi=cfg.plot_dpi, bbox_inches="tight", facecolor="white")
            shutil.copyfile(tmp, png)
            warn("map saved through the temp folder because the direct save failed (", e1, ")")
            return True
        except Exception as e2:
            warn("MAP NOT SAVED: ", repr(png), " | direct: ", e1, " | via temp folder: ", e2)
            return False

def _plot_one(arr, extent, outline, title, png, prod):
    l, r, b, t = extent
    fig, ax = plt.subplots(figsize=(7.5, 7.0))
    if prod == "risk_class":
        cm = plt.get_cmap(cfg.cmap)
        cols = [cm(x) for x in (0.08, 0.28, 0.50, 0.74, 0.95)]
        a = np.where((arr >= 1) & (arr <= 5), arr, np.nan)
        im = ax.imshow(a, extent=extent, origin="upper", cmap=ListedColormap(cols), norm=BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], 5), interpolation="nearest")
        cb = fig.colorbar(im, ax=ax, ticks=[1, 2, 3, 4, 5], shrink=0.8, pad=0.03)
        cb.ax.set_yticklabels(CLASS_LABELS); cb.set_label("Risk class")
    else:
        vmax = 1.0 if prod != "stat_sd" else (float(np.nanmax(arr)) if np.isfinite(arr).any() else 1.0)
        im = ax.imshow(arr, extent=extent, origin="upper", cmap=cfg.cmap, vmin=0, vmax=vmax, interpolation="nearest")
        cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.03); cb.set_label("Risk (0 to 1)" if prod != "stat_sd" else "SD")
    if outline is not None:
        outline.boundary.plot(ax=ax, color="black", linewidth=0.8)
    ax.set_xlim(l, r); ax.set_ylim(b, t)
    span = max(r - l, t - b); dec = 1 if span > 3 else 2 if span > 0.4 else 3
    ax.xaxis.set_major_locator(MaxNLocator(5)); ax.yaxis.set_major_locator(MaxNLocator(5))
    ax.xaxis.set_major_formatter(_fx(dec)); ax.yaxis.set_major_formatter(_fy(dec))
    ax.tick_params(axis="both", which="major", direction="in", length=6, width=1, top=True, right=True, labelsize=9)
    for lab in ax.get_yticklabels():
        lab.set_rotation(90); lab.set_va("center")
    ax.set_xlabel("Longitude (decimal degrees)"); ax.set_ylabel("Latitude (decimal degrees)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_aspect(1.0 / np.cos(np.radians(0.5 * (b + t))))
    ax.annotate("N", xy=(0.93, 0.94), xytext=(0.93, 0.84), xycoords="axes fraction", ha="center", va="center", fontsize=12,
                fontweight="bold", arrowprops=dict(facecolor="black", width=3, headwidth=10, headlength=10))
    km = 111.32 * np.cos(np.radians(0.5 * (b + t)))
    L = max([n for n in (1, 2, 5, 10, 20, 25, 50, 100) if n <= (r - l) * km / 4] or [1]); x0 = l + 0.05 * (r - l); y0 = b + 0.05 * (t - b)
    ax.plot([x0, x0 + L / km], [y0, y0], color="black", lw=3, solid_capstyle="butt", zorder=5)
    ax.text(x0 + L / km / 2, y0 + 0.012 * (t - b), "%d km" % L, ha="center", va="bottom", fontsize=8, zorder=5)
    fig.tight_layout()
    try:
        _save_png(fig, png)
        if getattr(cfg, "show_plots", True):
            plt.show()
    finally:
        plt.close(fig)

def plot_maps(county, tag, pests=None):
    """one figure per pest and product in cfg.plot_products, saved to <County>/maps_png/<date>/"""
    outline = _outline(county); n = 0
    for p in (pests or cfg.pests):
        for prod in cfg.plot_products:
            f = os.path.join(cfg.out_root, county, "predictions", tag, "%s_%s_%s.tif" % (p, tag, prod))
            if not os.path.exists(f) or (prod == "mechanistic" and (not has_mech_part(p) or cfg.static_mode)):
                continue
            tmp = os.path.join(out_dir(county, "exports_wgs84", tag), "%s_%s_%s_wgs84.tif" % (p, tag, prod))
            arr, ext = _to_wgs84(f, tmp, prod == "risk_class")
            if prod == "risk_class":
                arr = np.where(arr == 255, np.nan, arr)
            title = "%s - %s%s%s County, %s" % (PEST_NAME.get(p, p), PRODUCT_TITLE.get(prod, prod), chr(10), county, tag)
            try:
                fsum = os.path.join(cfg.out_root, county, "predictions", tag, "%s_%s_risk_area_summary.csv" % (p, tag))
                sm = pd.read_csv(fsum) if os.path.exists(fsum) else None
                if sm is not None and "season_covered" in sm.columns and not bool(sm["season_covered"].iloc[0]):
                    title += chr(10) + "(month outside the record season, indicative)"
            except Exception:
                pass
            try:
                _plot_one(arr, ext, outline, title, os.path.join(out_dir(county, "maps_png", tag), "%s_%s_%s.png" % (p, tag, prod)), prod)
                n += 1
            except Exception as e:
                warn("map ", p, " ", prod, " ", tag, " failed: ", type(e).__name__, ": ", e)
    log("Saved ", n, " PNG maps for ", county, " ", tag, " in ", out_dir(county, "maps_png", tag))

def finalize_county(county, objs, maps, write_accuracy_file=True):
    """accuracy workbook, GeoTIFF export in decimal degrees and PNG maps for every predicted date"""
    tags = [str(d) if str(d) == "static" else pd.Timestamp(d).strftime("%Y-%m-%d") for d in maps]
    steps = []
    if write_accuracy_file and objs:
        steps.append(("accuracy workbook", lambda: write_accuracy(county, objs, tags)))
    for tag in tags:
        steps.append(("GeoTIFF export " + tag, lambda t=tag: export_geotiffs(county, t, list(objs))))
        steps.append(("maps " + tag, lambda t=tag: plot_maps(county, t, list(objs))))
    for name, fn in steps:                         # one failing step never stops the others
        try:
            fn()
        except Exception as e:
            warn(name, " failed: ", type(e).__name__, ": ", e)

## 17. Run, one county at a time
Each county has its own cell. **Kakamega is active.** The Nyeri and Kilifi cells are commented out: remove the leading `# `
(select the cell, Ctrl+/ in Jupyter) once their datasets are complete. Each run writes under `<out_root>/<County>/`:
- `models/` saved models (one `.joblib` per pest, one `PestRisk_bundle_<County>.joblib` for all pests, `model_card_<PEST>.json`), 70/30 accuracy tables, training tables
- `accuracy/Accuracy_Assessment_<County>.xlsx` the separate accuracy file (ML models, final outputs, risk classes, weights, map check)
- `predictions/<date>/` all rasters in the model CRS: `*_stat_ensemble.tif` (statistical), `*_mechanistic.tif`, `*_hybrid.tif`, `*_risk_class.tif`
- `exports_wgs84/<date>/` the same rasters as EPSG:4326 (decimal degrees) GeoTIFFs
- `maps_png/<date>/` one PNG map per pest and product: statistical, mechanistic (FAW, Aphids, MLND only), hybrid and risk class

**What the cells below do:** run the pipeline per county. Read the explainer above each cell before running it.

In [ ]:
rep = data_report("Kakamega")
print("Incidence files:", rep["incidence"])
print("Daily weather files:", rep["weather_files"])
print("Layers not found:", ", ".join(rep["layers"].loc[~rep["layers"].found, "layer"]) or "none")

### KAKAMEGA (active)
All six pests have records. Landsat scenes exist. Choose the map dates in `predict_dates` (None = last date in the weather record).

In [ ]:
results_kakamega = run_pipeline(counties=["Kakamega"], predict_dates=cfg.predict_dates)

### NYERI (commented out: run when all Nyeri datasets are in place)
All six pests have records. Landsat scenes exist. Choose the map dates in `predict_dates` (None = last date in the weather record).

In [ ]:
rep = data_report("Nyeri"); print(rep["incidence"], rep["weather_files"])
results_nyeri = run_pipeline(counties=["Nyeri"], predict_dates=cfg.predict_dates)

### KILIFI 
Check first: Kilifi has monthly Landsat stacks (2020-2024). Thrips (25), African armyworm (28) and MLND (62) have few records, so read the accuracy
workbook carefully (look at `statistical_part_reliable` and the number of test presences).

In [ ]:
rep = data_report("Kilifi"); print(rep["incidence"], rep["weather_files"])
results_kilifi = run_pipeline(counties=["Kilifi"], predict_dates=cfg.predict_dates)

### OCTOBER 2026 NOWCAST: 
It uses the saved models, observed inputs and SEAS5 rain only 

In [ ]:

COUNTIES_OCT = ["Kakamega", "Nyeri", "Kilifi"]
FC_SCEN = "mean"        # SEAS5 rain scenario: "mean", "p10" or "p90"
OCT_DATE = None         # None = latest day covered by ALL observed variables in all three counties (max 2026-10-31); or set e.g. "2026-10-20"

cfg.pests = ["FAW", "APHIDS", "THRIPS", "STEMBORER", "MLND"]   # the five trained pests, MLND last (add "AAW" before "MLND" once it is trained)
cfg.vi_fc_months = []            # observed vegetation indices, no Notebook C
cfg.fc_only_vars = ["precip"]    # SEAS5 fills rain only; temperature, RH and wind stay observed
cfg.fc_scenario = FC_SCEN
cfg.uncovered_mode = "hybrid"    # always the 80/20 blend, never the mechanistic-only override
_MONTHLY_CACHE.clear(); _WARNED_YM.clear()

# 1) map date: the latest day that temperature, RH and wind have observed data
cfg.use_forecast = False
last = {}
for c in COUNTIES_OCT:
    ends = {}
    for v in ("tmax", "tmin", "rh", "wind"):
        x = read_wx_var(c, v)
        ends[v] = x["dates"].max() if x is not None else pd.NaT
    print(c, "last observed day:", {k: (str(t.date()) if pd.notna(t) else "none") for k, t in ends.items()})
    last[c] = min(t for t in ends.values() if pd.notna(t))
OCT = pd.Timestamp(OCT_DATE) if OCT_DATE else min(min(last.values()), pd.Timestamp("2026-10-31"))
OCT_DATES = [OCT.strftime("%Y-%m-%d")]
print("MAP DATE:", OCT_DATES[0])

# 2) SEAS5 rain must cover 1 October up to the map date (CHIRPS ends 30 September)
cfg.use_forecast = True
for c in COUNTIES_OCT:
    ff = _fc_files(c, "rainfall", "rain")
    n = len([d for d in ff if pd.Timestamp("2026-10-01") <= d <= OCT])
    print(c, "SEAS5 rain days 1 Oct to", OCT.date(), ":", n, "of", OCT.day, "" if n == OCT.day else "  <-- WARNING: gap, rain windows will be partly empty")
cfg.use_forecast = False

# 3) remove cached weather layers of this date, so nothing from an earlier run is reused
for c in COUNTIES_OCT:
    for f in glob.glob(os.path.join(out_dir(c, "wx"), "wx_%s*.tif" % OCT_DATES[0])):
        os.remove(f)

# 4) map titles: drop the "(month outside the record season, indicative)" line
if not getattr(_plot_one, "_no_note", False):
    _orig_plot_one = _plot_one
    def _plot_one(arr, extent, outline, title, png, prod):
        title = chr(10).join(l for l in title.split(chr(10)) if not l.startswith("(month outside"))
        return _orig_plot_one(arr, extent, outline, title, png, prod)
    _plot_one._no_note = True

# 5) map with the saved models
oct_maps = {}
for c in COUNTIES_OCT:
    try:
        oct_maps[c] = predict_dates_only(c, OCT_DATES, forecast=True, fc_scenario=FC_SCEN)
    except Exception as e:
        import traceback; traceback.print_exc()
        warn("October prediction failed for ", c, ": ", e)

## Forecast for 2026-10-14

In [ ]:
import gc, glob, os

# --- memory-friendly weather layer writer (unchanged from yours) ---
def wx_to_grid(county, grid, W, d, elev, ecoarse):
    d = pd.Timestamp(d)
    f = os.path.join(out_dir(county, "wx"), "wx_%s%s.tif" % (d.strftime("%Y-%m-%d"), ("_fc_" + cfg.fc_scenario) if cfg.use_forecast else ""))
    if cfg.reuse_cache and os.path.exists(f) and grid.same_geometry(f):
        with rasterio.open(f) as r:
            return Stack.from_file(f, list(r.descriptions))
    nat = wx_native(W, d)
    names = [nm for _, (L, _tm) in nat.items() for nm in L]
    off = None
    if cfg.lapse_correct and elev is not None and ecoarse is not None:
        off = cfg.lapse_rate * (elev - ecoarse)
    tl = set(temp_layers())
    prof = dict(driver="GTiff", height=grid.height, width=grid.width, count=len(names), dtype="float32",
                crs=grid.crs, transform=grid.transform, nodata=np.nan, compress="deflate", BIGTIFF="IF_SAFER")
    if grid.height >= 256 and grid.width >= 256:
        prof.update(tiled=True, blockxsize=256, blockysize=256)
    tmp = f + ".part"
    i = 0
    with rasterio.open(tmp, "w", **prof) as dst:
        for g, (L, tm) in nat.items():
            for nm, arr in L.items():
                p, tr = pad_native(arr, tm)
                out = np.full((grid.height, grid.width), np.nan, dtype="float32")
                reproject(p.astype("float32"), out, src_transform=tr, src_crs=tm["crs"], dst_transform=grid.transform,
                          dst_crs=grid.crs, resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
                if g == "temp" and off is not None and nm in tl:
                    out = out + off
                i += 1
                dst.write(out, i); dst.set_band_description(i, nm)
                del out
    os.replace(tmp, f)
    return Stack.from_file(f, names)

# --- settings: October forecast driven by Notebook C and SEAS5 (all four weather variables) ---
FC_DIRS = {
    "Kakamega": "F:/CORRECTION/geotiff",          # new rerun
    "Nyeri":    "F:/PREDICTED_INDICES/geotiff",   # earlier C run
    "Kilifi":   "F:/PREDICTED_INDICES/geotiff",   # earlier C run
}
cfg.vi_fc_scenario = "mean"
cfg.vi_fc_months = ["2026-10"]
cfg.fc_only_vars = None            # SEAS5 fills temperature, rain, RH and wind, as in C
cfg.fc_scenario = "mean"
cfg.weather_end = "2026-09-30"     # observed weather ends here, October comes from SEAS5
cfg.use_forecast = False           # predict_dates_only(forecast=True) switches it on
cfg.plot_products = ["stat_ensemble", "mechanistic", "hybrid_raw"]
_MONTHLY_CACHE.clear(); _WARNED_YM.clear()

OUT_FC = "F:/pest_unified_model_Cinputs"
OCT_DATES = ["2026-10-14"]
COUNTIES_RUN = ["Kakamega", "Nyeri", "Kilifi"]

# --- remove cached layers of this date in the output folder, so nothing old is reused ---
for c in COUNTIES_RUN:
    base = os.path.join(OUT_FC, c)
    for pat in (os.path.join(base, "aligned", "*_fc_mean.tif"),
                os.path.join(base, "wx", "wx_%s*.tif" % OCT_DATES[0])):
        for f in glob.glob(pat):
            os.remove(f)

# --- run, one county at a time, each with its own C folder ---
for c in COUNTIES_RUN:
    cfg.vi_fc_dir = FC_DIRS[c]
    print(c, "reads C forecasts from", cfg.vi_fc_dir)
    try:
        predict_dates_only(c, OCT_DATES, forecast=True, fc_scenario="mean", out_root=OUT_FC)
    except Exception as e:
        import traceback; traceback.print_exc()
        warn("forecast prediction failed for ", c, ": ", e)
    gc.collect()

# --- summary table (share_raw must be defined in another cell) ---
allrows = []
for c in COUNTIES_RUN:
    try:
        allrows += share_raw(OUT_FC, c, OCT_DATES[0])
    except Exception as e:
        warn("share_raw skipped for ", c, ": ", e)
print(pd.DataFrame(allrows).to_string(index=False))

## Validation of the model using validation data for both forecast and nowcast

In [ ]:
import os, glob, gc
import numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt

COUNTIES = ["Kakamega", "Nyeri", "Kilifi"]
VAL_DATE = "2026-10-14"                      # must exist in BOTH runs
NOW_ROOT = cfg.out_root                      # nowcast predictions
FC_ROOT = "F:/pest_unified_model_Cinputs"    # forecast predictions
VAL_OUT = os.path.join(FC_ROOT, "validation_" + VAL_DATE)
os.makedirs(VAL_OUT, exist_ok=True)

# --- 0. check both runs have the date ---
for c in COUNTIES:
    for nm, root in (("nowcast", NOW_ROOT), ("forecast", FC_ROOT)):
        d = os.path.join(root, c, "predictions")
        have = sorted(os.listdir(d)) if os.path.isdir(d) else []
        print("%-9s %-9s %s %s" % (c, nm, VAL_DATE, "OK" if VAL_DATE in have else "MISSING; available: %s" % have[-8:]))

# --- helpers ---
def read_preds(root, county, date):
    out = {}
    for f in sorted(glob.glob(os.path.join(root, county, "predictions", date, "*_hybrid_raw.tif"))):
        pest = os.path.basename(f).split("_")[0]
        with rasterio.open(f) as s:
            a = s.read(1).astype("float32"); nod = s.nodata; tr = tuple(s.transform)
        valid = np.isfinite(a)
        if nod is not None and not np.isnan(nod): valid &= (a != nod)
        a[~valid] = np.nan
        out[pest] = (a, tr)
    return out

def to_class(x):                              # same rule as share_raw: 0 = no risk, 1..5 = very_low..very_high
    x = np.round(np.clip(x, 0, 1), 3)
    return np.where(x <= 0, 0, 1 + np.searchsorted(BRK_RAW, x, side="right"))

def qw_kappa(a, b, k=6):                      # quadratic-weighted kappa for ordinal classes
    O = np.zeros((k, k)); np.add.at(O, (a, b), 1)
    w = np.subtract.outer(np.arange(k), np.arange(k)) ** 2 / (k - 1) ** 2
    E = np.outer(O.sum(1), O.sum(0)) / O.sum()
    with np.errstate(divide="ignore", invalid="ignore"):
        return 1 - (w * O).sum() / (w * E).sum()

def plot_triplet(n, f, title, path, max_px=1500):
    s = max(1, int(np.ceil(max(n.shape) / max_px)))      # downsample for plotting only
    n, f = n[::s, ::s], f[::s, ::s]
    d = f - n
    lim = max(0.05, float(np.nanpercentile(np.abs(d), 98))) if np.isfinite(d).any() else 0.05
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    for a, arr, t, kw in ((ax[0], n, "Nowcast", dict(cmap="YlOrRd", vmin=0, vmax=1)),
                          (ax[1], f, "Forecast", dict(cmap="YlOrRd", vmin=0, vmax=1)),
                          (ax[2], d, "Forecast - nowcast", dict(cmap="RdBu_r", vmin=-lim, vmax=lim))):
        im = a.imshow(arr, interpolation="nearest", **kw); a.set_title(t); a.axis("off"); fig.colorbar(im, ax=a, shrink=0.7)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=120); plt.close(fig)

# --- 1. compare pixel by pixel ---
rows = []
for c in COUNTIES:
    now, fc = read_preds(NOW_ROOT, c, VAL_DATE), read_preds(FC_ROOT, c, VAL_DATE)
    for p in sorted(set(now) | set(fc)):
        if p not in now or p not in fc:
            print("skipped %s %s: only in %s" % (c, p, "nowcast" if p in now else "forecast")); continue
        n, tn = now[p]; f, tf = fc[p]
        if n.shape != f.shape or not np.allclose(tn, tf):
            print("skipped %s %s: grids differ %s vs %s" % (c, p, n.shape, f.shape)); continue
        m = np.isfinite(n) & np.isfinite(f)
        if m.sum() < 10: print("skipped %s %s: no overlap" % (c, p)); continue
        nm_, fm_ = n[m].astype("float64"), f[m].astype("float64")
        cn, cf = to_class(nm_), to_class(fm_)
        rows.append(dict(county=c, pest=p, pixels=int(m.sum()),
                         mean_now=nm_.mean(), mean_fc=fm_.mean(),
                         bias=(fm_ - nm_).mean(), mae=np.abs(fm_ - nm_).mean(),
                         rmse=np.sqrt(((fm_ - nm_) ** 2).mean()),
                         pearson=np.corrcoef(nm_, fm_)[0, 1],
                         spearman=pd.Series(nm_).corr(pd.Series(fm_), method="spearman"),
                         same_class_pct=100 * (cn == cf).mean(),
                         within1_pct=100 * (np.abs(cn - cf) <= 1).mean(),
                         wkappa=qw_kappa(cn, cf)))
        pd.crosstab(pd.Series(cn, name="nowcast"), pd.Series(cf, name="forecast")).to_csv(
            os.path.join(VAL_OUT, "transition_%s_%s.csv" % (c, p)))
        try:
            plot_triplet(n, f, "%s, %s, %s" % (p, c, VAL_DATE), os.path.join(VAL_OUT, "diff_%s_%s.png" % (c, p)))
        except Exception as e:
            plt.close("all"); print("plot skipped %s %s: %s" % (c, p, e))
        del n, f, m, nm_, fm_, cn, cf; gc.collect()
    del now, fc; gc.collect()

val = pd.DataFrame(rows)
val.to_csv(os.path.join(VAL_OUT, "validation_summary.csv"), index=False)
print(val.round(3).to_string(index=False))
print("\nBy pest:\n", val.groupby("pest")[["bias", "mae", "spearman", "same_class_pct", "within1_pct", "wkappa"]].mean().round(3))
print("\nBy county:\n", val.groupby("county")[["bias", "mae", "spearman", "same_class_pct", "within1_pct", "wkappa"]].mean().round(3))

### STATIC SUSCEPTIBILITY RUN (separate folder, counties kept separate)
**What the next cell does:** trains and maps each county again, but with *static* predictors. Every date-dependent layer (weather windows, monthly
vegetation indices, Landsat surface temperature) is replaced by its **mean over the last 12 months** of the weather record (the 15th of each month),
plus the seasonal spread of temperature, rainfall and humidity. Terrain, soil and crop type are unchanged. No bioclim layers are used.
The result is one susceptibility map per pest with no date. There is **no mechanistic part** in this mode, so the output is the statistical map.
Background points are host (crop) cells only. The test is the same spatial 70/30 split as the main run, and a **random 70/30 split is also
scored for comparison only** (rows ending `_random_split` in the accuracy file). **Everything is written to a separate folder**
(`<out_root>_susceptibility`, change it with `cfg.static_out_root`), so the main run is untouched. The aligned-layer caches of the main run are reused.
Run the main run first (or at least once), so the caches exist.

In [ ]:
results_static_kakamega = run_static(counties=["Kakamega"])

### STATIC RUN, NYERI (commented out: run when all Nyeri datasets are in place)
**What the next cell does:** the same static run for Nyeri only, in the same separate folder.

In [ ]:
results_static_nyeri = run_static(counties=["Nyeri"])

### STATIC RUN, KILIFI (commented out: run when all Kilifi datasets are in place)
**What the next cell does:** the same static run for Kilifi only, in the same separate folder.

In [ ]:
results_static_nyeri = run_static(counties=["Nyeri"])